In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10,5)
sns.set_style("whitegrid")

# Load train and store dataset 
train = pd.read_csv('''C:/Users/shrut/OneDrive/Desktop/Master thesis/Dataset/rossmann-store-sales/train.csv''')
store = pd.read_csv('''C:/Users/shrut/OneDrive/Desktop/Master thesis/Dataset/rossmann-store-sales/store.csv''')

# Convert Date
train["Date"] = pd.to_datetime(train["Date"])

# Merge train and store csv files

df = pd.merge(train, store, on="Store", how="left") ## left join

## Basic Info 
df.info()
df.head()


In [ ]:
df[df["CompetitionDistance"].isna()]["Store"].nunique()


In [ ]:
df.describe().T ## Descriptive statistics


In [ ]:
# Number of unique stores
print("Number of stores:", df["Store"].nunique()) 


In [ ]:
## Date range
print("Date range:", df["Date"].min(), "→", df["Date"].max())

In [ ]:
df.isna().sum().sort_values(ascending=False) ## checking for the missing values int the merged dataset


In [ ]:
df['Open'].value_counts()

In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.dayofweek + 1
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

In [ ]:
df.head(5)


# EDA

In [ ]:
!pip install plotly==5.24.1

In [ ]:
import plotly.express as px
import plotly.graph_objects as go


# total sales vs time


In [ ]:
import plotly.express as px

daily_sales = df.groupby("Date", as_index=False)["Sales"].sum()

fig = px.line(
    daily_sales,
    x="Date",
    y="Sales",
    title="Total Sales Over Time"
)

fig.update_layout(xaxis_title="", hovermode="x unified")
fig.show()


# AVG MONTHLY SALE BY YEAR 13,14,15

In [ ]:
avg_day = df.groupby("DayOfWeek", as_index=False)["Sales"].mean().round(2)  ## this is for the avd day sales for all data for 2.5 years
avg_month = df.groupby("Month", as_index=False)["Sales"].mean().round(2)   ## avg month sales for all the data
print("Average Sales by Day of Week: \n", avg_day)

print("\nAverage Sales by Month: \n", avg_month)


# Average Monthly Sales by Year

In [ ]:
tmp = df.groupby(["Year", "Month"], as_index=False)["Sales"].mean()

fig = px.line(tmp, x="Month", y="Sales", color="Year",
              markers=True,
              title="Average Monthly Sales by Year",
              labels=dict(Sales="Average Sales (€)", Month="Month", Year="Year"),
              category_orders={"Month": list(range(1,13))})
fig.show()


# Average Sales by Day of Week

In [ ]:
fig_dow = px.line(
    avg_day,
    x="DayOfWeek",
    y="Sales",
    markers=True,
    title="Average Sales by Day of Week",
    color_discrete_sequence=["#7E57C2"]
)
fig_dow.update_traces(line=dict(width=3))
fig_dow.update_layout(
    xaxis_title="Day of Week (1=Mon … 7=Sun)",
    yaxis_title="Average Sales (€)",
    font=dict(size=14, color="#333"),
    plot_bgcolor="white",
    
    paper_bgcolor="white",
    title=dict(x=0.5, font=dict(size=22))
)
fig_dow.show()


In [ ]:
df["DayOfWeek"].value_counts().sort_index() ## numbner of rows in the dataset


# so, the reason why sunday has more sales than saturday is beacuse may be its near the train station or airport stations.

In [ ]:
avg_dow = df.groupby("DayOfWeek")["Sales"].mean().round(2)
print(avg_dow)  

In [ ]:
sunday_stores = df[(df["DayOfWeek"] == 7) & (df["Open"] == 1)]["Store"].unique()
print("Stores open on Sunday:", len(sunday_stores))
print("Example Store IDs:", sunday_stores[:10])

In [ ]:
store_sunday = store[store["Store"].isin(sunday_stores)].copy()
store_sunday.head(10)

In [ ]:
# counts by attributes
print("Count of Sunday-open stores:", store_sunday["Store"].nunique())
print(store_sunday[["StoreType","Assortment","Promo2"]].value_counts())

# averages to compare with all stores
summ = (
    pd.concat([
        store.assign(Group="All stores"),
        store_sunday.assign(Group="Sunday-open")
    ], ignore_index=True)
    .groupby("Group")[["Promo2"]]
    .mean()
)

In [ ]:
# --- Build store-level metadata once ---
store_meta = (
    df.sort_values('Date')
      .groupby('Store', as_index=False)
      .agg({
          'StoreType': 'first',
          'Assortment': 'first',
          'Promo2': 'max'   # Promo2 is 0/1; max handles NaNs as well
      })
)

# --- Identify Sunday-open stores (any Sunday with Open=1) ---
sunday_stores = df[(df['DayOfWeek'] == 6) & (df['Open'] == 1)]['Store'].unique()
store_sunday_meta = store_meta[store_meta['Store'].isin(sunday_stores)].copy()

# --- Counts by attributes (per store, not per row) ---
print("Count of Sunday-open stores:", store_sunday_meta['Store'].nunique())
print(store_sunday_meta[['StoreType','Assortment','Promo2']].value_counts())

# --- Compare Promo2 share (per store) between groups ---
summ = (
    pd.concat([
        store_meta.assign(Group="All stores"),
        store_sunday_meta.assign(Group="Sunday-open")
    ], ignore_index=True)
    .groupby("Group")[['Promo2']]
    .mean()  # share of stores with Promo2=1
    .rename(columns={'Promo2': 'Share_Promo2'})
    .round(3)
)

print("\nPromo2 share by group:\n", summ)


## Checking for the one store's sales

In [ ]:
import plotly.express as px

store_sales = (
    df[df["Store"] == 601]
    .groupby("Date", as_index=False)["Sales"]
    .sum()
)

fig = px.line(
    
    store_sales,
    x="Date",
    y="Sales",
    title="Store 601 — Sales Over Time"
)

fig.update_layout(xaxis_title="")
fig.show()

# Average sales by store type (a,b,c,d)

In [ ]:
tmp = (df[df["Open"]==1]
       .groupby("StoreType", as_index=False)["Sales"].mean()
       .sort_values("Sales", ascending=False))
fig = px.bar(
    tmp, x="StoreType", y="Sales",
    color="Sales", color_continuous_scale="Plasma",
    title="Average Sales by Store Type"
)
fig.update_traces(width=0.7, texttemplate="€ %{y:,.0f}", textposition="outside")
fig.update_layout(xaxis_title="Store Type", yaxis_title="Average Sales (€)",
                  font=dict(size=14), plot_bgcolor="white", paper_bgcolor="white", margin=dict(t=100),width=600, height=400)
fig.update_yaxes(range=[0, 12000])
fig.update_xaxes(tickfont=dict(size=17, family="Arial", color="black"))
fig.update_yaxes(tickfont=dict(size=17, family="Arial", color="black"))
fig.update_layout(coloraxis_colorbar=dict(title="Sales (€)", thickness=30, len=0.6, x=1, y=0.5))

fig.show()


In [ ]:
print(df.columns)


In [ ]:
tmp = df.groupby("SchoolHoliday", as_index=False)["Sales"].mean()
tmp["SchoolHoliday"] = tmp["SchoolHoliday"].map({0:"No School Holiday", 1:"School Holiday"})
fig = px.bar(
    tmp, x="SchoolHoliday", y="Sales",
    color="Sales", color_continuous_scale="Plasma",
    title="Average Sales: School Holiday vs Non-Holiday"
)
fig.update_traces(width=0.5, texttemplate="€ %{y:,.0f}", textposition="outside")
fig.update_layout(xaxis_title="School holidays", yaxis_title="Average Sales (€)",
                  font=dict(size=14), plot_bgcolor="white", paper_bgcolor="white", margin=dict(t=100),width=600, height=400)
fig.update_yaxes(range=[0, 8000])
fig.update_xaxes(tickfont=dict(size=17, family="Arial", color="black"))
fig.update_yaxes(tickfont=dict(size=17, family="Arial", color="black"))
fig.update_layout(coloraxis_colorbar=dict(title="Sales (€)", thickness=30, len=0.6, x=1, y=0.5))

fig.show() 


In [ ]:
# Keep only open-store days
df_open = df[df["Open"] == 1].copy()

In [ ]:
fig = px.scatter(
    df_open,
    x="Customers",
    y="Sales",
    color="StoreType",
    title="Sales vs Customers by Store Type",
    opacity=0.5,
    labels={"Sales": "Sales (€)", "Customers": "Customers"},
    color_discrete_map={
        "a": "#1f77b4",   # blue
        "b": "#e41a1c",   # red
        "c": "#4daf4a",   # green
        "d": "#ff7f00",   # orange (new)
    }
)

fig.update_layout(width=800, height=500, plot_bgcolor="white")
fig.show()


In [ ]:
fig = px.box(
    df,
    x="StoreType",
    y="CompetitionDistance",
    color="StoreType",
    title="Competition Distance by Store Type (log scale)",
    labels={"StoreType": "Store Type", "CompetitionDistance": "Competition Distance (m)"}
)
fig.update_layout(
    yaxis_type="log",
    plot_bgcolor="white",
    paper_bgcolor="white",
    width=700, height=450
)
fig.show()


In [ ]:
total_missing = df["CompetitionDistance"].isna().sum()
total_rows    = len(df)

print("Missing CompetitionDistance values:", total_missing)


In [ ]:
df[df["CompetitionDistance"].isna()].groupby("StoreType").size()


In [ ]:
print(df.columns)

In [ ]:
df = pd.get_dummies(
    df,
    columns=["Assortment", "StoreType"],
    drop_first=True
)
df.head(5)

In [ ]:
df[df["CompetitionDistance"].isna()]["Store"].nunique()

In [ ]:
df['CompetitionDistance'].fillna(df['CompetitionDistance'].max(), inplace=True)
### imputed max value in place of the NOT MISSING DATA....becacuse there are no competetior...

In [ ]:
df.head(5)

In [ ]:
total_missing = df["CompetitionDistance"].isna().sum()
total_rows    = len(df)

print("Missing CompetitionDistance values:", total_missing)



In [ ]:
####CompetitionOpenSinceMonth & Year
# Missing means competition DOES NOT EXIST
df['CompetitionOpenSinceMonth'].fillna(0, inplace=True)
df['CompetitionOpenSinceYear'].fillna(0, inplace=True)

In [ ]:
# Missing promo info means: store is NOT participating in Promo2
df['Promo2'].fillna(0, inplace=True)

In [ ]:
# 4. Promo2SinceWeek & Promo2SinceYear
# Only stores in Promo2 have these — else missing = not in program
df['Promo2SinceWeek'].fillna(0, inplace=True)
df['Promo2SinceYear'].fillna(0, inplace=True)

In [ ]:
# 5. PromoInterval
# Missing = None (store not in Promo2)
df['PromoInterval'].fillna("None", inplace=True)

In [ ]:
# Convert PromoInterval to list of months
df['PromoInterval'] = df['PromoInterval'].fillna("None")
df['PromoIntervalList'] = df['PromoInterval'].apply(lambda x: x.split(','))

# Map month number to month name (first 3 letters)
month_map = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun",
             7:"Jul", 8:"Aug", 9:"Sept", 10:"Oct", 11:"Nov", 12:"Dec"}

df['MonthName'] = df['Month'].map(month_map)

# Create IsPromoMonth
df['IsPromoMonth'] = df.apply(
    lambda row: 1 if row['MonthName'] in row['PromoIntervalList'] else 0,
    axis=1
)

In [ ]:
df.head(5)

In [ ]:
 #Select Store 1
store1 = df[df['Store'] == 1].copy()

# Convert Date to datetime if not done yet
store1['Date'] = pd.to_datetime(store1['Date'])
     
# Set Date as index (required for STL)
store1 = store1.set_index('Date').sort_index()


In [ ]:
from statsmodels.tsa.seasonal import STL ####Seasonality and trend decomposition using LOESS (STL)

stl = STL(store1['Sales'], period=7, robust=True)
res = stl.fit()


In [ ]:
import matplotlib.pyplot as plt

fig = res.plot()
fig.set_size_inches(12, 8)
plt.show()


In [ ]:
df.head(5)

In [ ]:
import pandas as pd
import numpy as np

STORE_ID = 1

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

s = (df[df["Store"] == STORE_ID]
     .sort_values("Date")
     .set_index("Date"))

full_idx = pd.date_range(s.index.min(), s.index.max(), freq="D")
s = s.reindex(full_idx)

s["Sales"] = s["Sales"].fillna(0)
y = s["Sales"].astype(float)

print("Store:", STORE_ID)
print("Date range:", y.index.min().date(), "->", y.index.max().date())
print("Total days:", len(y))


In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(y, model="additive", period=7)
decomp.plot()
plt.suptitle(f"Time Series Decomposition (Store {STORE_ID})", y=1.02)
plt.show()


## Adfuller test for stationarity for store no 3

In [ ]:
y_diff1 = y.diff(1)

import matplotlib.pyplot as plt
plt.figure(figsize=(12, 4))
plt.plot(y_diff1)
plt.title("1st Order Differencing (d = 1)")
plt.show()

p1 = adf_test(y_diff1, "1st Order Differenced Sales Series")


## daily sales plot for store 3

In [ ]:
y_d = y.dropna()   # d = 0 → use original series


## ACF and PACF plots for store id 262
### ACF shows correlation of today’s sales with past sales, weekly seasonality 
### Only the first lag shows significant partial autocorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(y_d, lags=40, ax=ax[0])
ax[0].set_title("ACF (d = 0)")

plot_pacf(y_d, lags=40, ax=ax[1], method="ywm")
ax[1].set_title("PACF (d = 0)")

plt.tight_layout()
plt.show()


In [ ]:
df.head(5)

In [ ]:
!pip install pmdarima

# optimized auto arima code. (run this)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pmdarima as pm
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# =========================
# SETTINGS (80/20 split)
# =========================
SPLIT_RATIO = 0.8
M = 7            # weekly seasonality
H = 7            # 7-day horizon
N_SPLITS = 5     # CV folds
LAGS = 40        # residual ACF/PACF lags

TOTAL_DAYS_ROLL = 60   # rolling/weekday analysis span (TRAIN only)
BACKTEST_DAYS = 60     # backtest span (TRAIN only)

H_FUTURE = 7           # future forecast days
N_HISTORY_PLOT = 120   # history shown in final forecast plot

# =========================
# PREP (make sure y is correct)
# =========================
assert isinstance(y, pd.Series), "y must be a pandas Series"
assert isinstance(y.index, pd.DatetimeIndex), "y must have a DatetimeIndex"
y = y.sort_index()

print("y starts:", y.index.min())
print("y ends  :", y.index.max())
print("len(y)  :", len(y))


In [ ]:
# =========================
# 1) 80/20 chronological split
# =========================
split_point = int(len(y) * SPLIT_RATIO)
y_train_full = y.iloc[:split_point]
y_test_final = y.iloc[split_point:]

print("TRAIN size:", len(y_train_full))
print("TEST  size:", len(y_test_final))
print("TRAIN ends:", y_train_full.index[-1])
print("TEST starts:", y_test_final.index[0])


# STEP 1: Model selection ONCE (TRAIN only)

print("STEP 1: auto_arima model selection")


model1 = pm.auto_arima(
    y_train_full,
    seasonal=True,
    m=M,
    stepwise=True,
    trace=True,
    suppress_warnings=True,
    error_action="ignore",
    test="adf",
    information_criterion="aic",
    d=0,                 # force d=0
    max_p=10, max_q=10,
    n_jobs=-1
)

print(model1.summary())


fixed_order = model1.order
fixed_seasonal_order = model1.seasonal_order

print("Chosen order:", fixed_order)
print("Chosen seasonal_order:", fixed_seasonal_order)
print(f"AIC: {model1.aic():.2f} | BIC: {model1.bic():.2f}")


In [ ]:
# =========================
# Chunk 2: Cross-Validation (TRAIN only, fixed SARIMA)
# =========================
print("\n" + "=" * 70)
print("STEP 2: TimeSeries CV (TRAIN only, fixed model)")
print("=" * 70)

tscv = TimeSeriesSplit(n_splits=N_SPLITS, test_size=H)

rmses = []
maes  = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(y_train_full), 1):
    y_train_fold = y_train_full.iloc[train_idx]
    y_test_fold  = y_train_full.iloc[test_idx]

    fold_model = pm.ARIMA(
        order=fixed_order,                  # (2,0,2)
        seasonal_order=fixed_seasonal_order, # (1,0,1,7)
        suppress_warnings=True
    ).fit(y_train_fold)

    preds = fold_model.predict(n_periods=len(y_test_fold))

    rmse = np.sqrt(mean_squared_error(y_test_fold, preds))
    mae  = mean_absolute_error(y_test_fold, preds)

    rmses.append(rmse)
    maes.append(mae)

    print(
        f"Fold {fold}: "
        f"Train={len(y_train_fold)}, "
        f"Test={len(y_test_fold)} | "
        f"RMSE={rmse:.2f}, MAE={mae:.2f}"
    )

print("\nCV SUMMARY (TRAIN only)")
print(f"RMSE mean: {np.mean(rmses):.3f}")
print(f"RMSE std : {np.std(rmses):.3f}")
print(f"MAE  mean: {np.mean(maes):.3f}")
print(f"MAE  std : {np.std(maes):.3f}")


In [ ]:
# =========================
# Chunk 3: Final evaluation on TEST (30% future)
# =========================
print("\n" + "=" * 70)
print("STEP 3: Final evaluation on TEST (fit TRAIN → predict TEST)")
print("=" * 70)

# Fit on full TRAIN
final_model = pm.ARIMA(
    order=fixed_order,                  # (2,0,2)
    seasonal_order=fixed_seasonal_order, # (1,0,1,7)
    suppress_warnings=True
).fit(y_train_full)

# Predict entire TEST horizon
test_preds = final_model.predict(n_periods=len(y_test_final))

# Metrics
test_rmse = np.sqrt(mean_squared_error(y_test_final, test_preds))
test_mae  = mean_absolute_error(y_test_final, test_preds)

print(f"TEST size: {len(y_test_final)}")
print(f"TEST RMSE : {test_rmse:.2f}")
print(f"TEST MAE  : {test_mae:.2f}")


In [ ]:
# =========================
# Chunk 4: Residual diagnostics (TRAIN model)
# =========================
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

# residuals from TRAIN-fitted model
resid = final_model.resid()

FIGSIZE = (8, 3)
LAGS = 40

# --- Residual time series ---
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(resid)
ax.set_title("Residuals (TRAIN model)")
plt.tight_layout()
plt.show()

# --- ACF of residuals ---
fig, ax = plt.subplots(figsize=FIGSIZE)
plot_acf(resid, lags=LAGS, ax=ax)
ax.set_title("ACF of Residuals")
plt.tight_layout()
plt.show()

# --- PACF of residuals ---
fig, ax = plt.subplots(figsize=FIGSIZE)
plot_pacf(resid, lags=LAGS, method="ywm", ax=ax)
ax.set_title("PACF of Residuals")
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# Chunk 5: Backtest plot (last 60 days of TRAIN only)
# =========================
print("\n" + "=" * 70)
print("STEP 5: Backtest plot (last 60 days of TRAIN)")
print("=" * 70)

BACKTEST_DAYS = 60

# split TRAIN into earlier train + backtest window
y_train_bt = y_train_full.iloc[:-BACKTEST_DAYS]
y_test_bt  = y_train_full.iloc[-BACKTEST_DAYS:]

print("Backtest window:")
print("From:", y_test_bt.index.min())
print("To  :", y_test_bt.index.max())

# fit model on earlier TRAIN
backtest_model = pm.ARIMA(
    order=fixed_order,                   # (2,0,2)
    seasonal_order=fixed_seasonal_order, # (1,0,1,7)
    suppress_warnings=True
).fit(y_train_bt)

# forecast backtest window
bt_preds = backtest_model.predict(n_periods=len(y_test_bt))

# plot
plt.figure(figsize=(12, 4))

# show some history before backtest for context
plt.plot(
    y_train_bt.index[-100:], 
    y_train_bt.values[-100:], 
    color="gray", alpha=0.5, label="Train history"
)

plt.plot(
    y_test_bt.index, 
    y_test_bt.values, 
    marker="o", label="Actual (last 60 days)"
)

plt.plot(
    y_test_bt.index, 
    bt_preds, 
    marker="o", linestyle="--", label="Forecast"
)

plt.title("Backtest: SARIMA(2,0,2)(1,0,1)[7] — Last 60 Days (TRAIN)")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# Chunk 6A: Rolling weekly forecasts (TRAIN only)
# =========================
print("\n" + "=" * 70)
print("STEP 6A: Rolling weekly forecasts (TRAIN only)")
print("=" * 70)

H = 7
TOTAL_DAYS_ROLL = 60   # last ~2 months inside TRAIN

start = len(y_train_full) - TOTAL_DAYS_ROLL
start = max(start, H)  # safety

predictions = []
actuals = []
dates = []

for t in range(start, len(y_train_full), H):
    y_train_roll = y_train_full.iloc[:t]
    y_test_roll  = y_train_full.iloc[t:t+H]

    roll_model = pm.ARIMA(
        order=fixed_order,                   # (2,0,2)
        seasonal_order=fixed_seasonal_order, # (1,0,1,7)
        suppress_warnings=True
    ).fit(y_train_roll)

    fc = roll_model.predict(n_periods=len(y_test_roll))

    predictions.extend(fc)
    actuals.extend(y_test_roll.values)
    dates.extend(y_test_roll.index)

df_roll = pd.DataFrame(
    {"Actual": actuals, "Predicted": predictions},
    index=dates
)

print("Rolling forecast period:")
print("From:", df_roll.index.min())
print("To  :", df_roll.index.max())
print("Total predicted days:", len(df_roll))


In [ ]:
# =========================
# Chunk 6B: Rolling forecast plot
# =========================
plt.figure(figsize=(12, 4))
plt.plot(df_roll.index, df_roll["Actual"], marker="o", label="Actual")
plt.plot(df_roll.index, df_roll["Predicted"], marker="o", linestyle="--", label="Rolling 7-day Forecast")
plt.title("Rolling Weekly Forecast (TRAIN only)")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# Chunk 6C: Weekday-wise analysis
# =========================
df_roll["Weekday"] = df_roll.index.day_name()

weekdays = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

for day in weekdays:
    tmp = df_roll[df_roll["Weekday"] == day]

    plt.figure(figsize=(6, 3))
    plt.plot(tmp.index, tmp["Actual"], marker="o", label="Actual")
    plt.plot(tmp.index, tmp["Predicted"], marker="o", linestyle="--", label="Predicted")
    plt.title(f"{day}: Actual vs Predicted (Rolling)")
    plt.xlabel("Date")
    plt.ylabel("Sales")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# =========================
# Chunk 6D: Error per weekday
# =========================
df_roll["AbsError"] = np.abs(df_roll["Actual"] - df_roll["Predicted"])

weekday_error = (
    df_roll
    .groupby("Weekday")["AbsError"]
    .mean()
    .reindex(weekdays)
)

print("Mean Absolute Error by Weekday:")
print(weekday_error)


In [ ]:
df.head(5)

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])


## sarimax with no. of exogenous variables

# single store automation

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

import pmdarima as pm
import statsmodels.api as sm
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =============================================================================
# IMPROVED UTILITY FUNCTIONS
# =============================================================================

def enforce_non_negative(predictions, open_status=None, index=None):
    """
    Ensure predictions are non-negative.
    If store is closed (Open=0), sales should be 0.
    Returns a pandas Series if index is provided, otherwise numpy array.
    """
    predictions = np.array(predictions)
    predictions = np.maximum(predictions, 0)

    if open_status is not None:
        open_status = np.array(open_status)
        predictions[open_status == 0] = 0

    if index is not None:
        return pd.Series(predictions, index=index)
    return predictions


def calculate_mape_safe(actual, predicted):
    """
    Safe MAPE calculation that handles zero values.
    Returns NaN if all actual values are zero.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)

    mask = actual != 0
    if mask.sum() == 0:
        return np.nan

    predicted = enforce_non_negative(predicted)
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


def calculate_smape(actual, predicted):
    """
    Symmetric MAPE - handles zero values better than MAPE.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)

    predicted = enforce_non_negative(predicted)

    denominator = (np.abs(actual) + np.abs(predicted))
    mask = denominator != 0
    if mask.sum() == 0:
        return np.nan

    return 100 / len(actual) * np.sum(2 * np.abs(predicted[mask] - actual[mask]) / denominator[mask])


def calculate_all_metrics(y_true, y_pred, X_test=None):
    """
    Calculate all performance metrics safely.
    """
    metrics = {}

    if X_test is not None and 'Open' in X_test.columns:
        y_pred_clean = enforce_non_negative(
            y_pred, X_test['Open'],
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )
    else:
        y_pred_clean = enforce_non_negative(
            y_pred,
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )

    metrics['rmse'] = float(np.sqrt(mean_squared_error(y_true, y_pred_clean)))
    metrics['mae'] = float(mean_absolute_error(y_true, y_pred_clean))

    metrics['mape'] = calculate_mape_safe(y_true, y_pred_clean)
    metrics['smape'] = calculate_smape(y_true, y_pred_clean)

    metrics['bias'] = float(np.mean(y_pred_clean - y_true))
    metrics['std_error'] = float(np.std(y_pred_clean - y_true))

    ss_res = np.sum((y_true - y_pred_clean) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    metrics['r2_adj'] = float(1 - (ss_res / ss_tot)) if ss_tot > 0 else np.nan

    return metrics, y_pred_clean


# =============================================================================
# STEP 1 (ONE STORE) — FIXED TEST PERIOD PROTOCOL SELECTION WITH FORECASTING
# =============================================================================

def prepare_store_data(
    df: pd.DataFrame,
    store_id: int,
    exogenous_columns=("Open", "Promo", "SchoolHoliday", "StateHoliday", "IsPromoMonth", "DayOfWeek"),
):
    s = df[df["Store"] == store_id].copy()
    s["Date"] = pd.to_datetime(s["Date"], dayfirst=True)
    s = s.sort_values("Date").set_index("Date")

    y = s["Sales"].astype(float).sort_index()
    X = s[list(exogenous_columns)].copy().loc[y.index]

    X["Open"] = X["Open"].astype(int)
    X["Promo"] = X["Promo"].astype(int)
    X["SchoolHoliday"] = X["SchoolHoliday"].astype(int)
    X["IsPromoMonth"] = X["IsPromoMonth"].astype(int)
    X["StateHoliday"] = (X["StateHoliday"].astype(str) != "0").astype(int)

    dow = pd.get_dummies(X["DayOfWeek"], prefix="DOW", drop_first=True).astype(float)

    X_num = X.drop(columns=["DayOfWeek"]).astype(float)
    Xcand = pd.concat([X_num, dow], axis=1).astype(float)

    return y, Xcand


def select_orders_with_auto_arima(y_tr, X_tr, seasonal_on=True, m=7, trace=False):
    if seasonal_on:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=True,
            m=m,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            D=None,
            max_p=3, max_q=3,
            max_P=2, max_Q=2,
            max_d=0,
            max_D=1,
            start_p=1,
            start_q=1,
            start_P=0,
            start_Q=0,
            test='adf',
            seasonal_test='ocsb',
            n_fits=20,
            with_intercept=True
        )
        return sel.order, sel.seasonal_order, float(sel.aic()), float(sel.bic())
    else:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=False,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            max_p=3, max_q=3,
            max_d=0,
            start_p=1,
            start_q=1,
            test='adf',
            n_fits=15,
            with_intercept=True
        )
        return sel.order, (0, 0, 0, 0), float(sel.aic()), float(sel.bic())


def backward_elimination_exog(y_tr, X_tr, order, seasonal_order, p_thresh=0.05, max_iter=50):
    cols = list(X_tr.columns)

    if len(cols) == 0:
        mod = sm.tsa.SARIMAX(
            y_tr, exog=None, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        return [], res

    for _ in range(max_iter):
        X_use = X_tr[cols] if cols else None

        mod = sm.tsa.SARIMAX(
            y_tr, exog=X_use, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        if not cols:
            return [], res

        pvals = res.pvalues
        p_exog = pvals[[c for c in pvals.index if c in cols]]

        if len(p_exog) == 0:
            return cols, res

        worst_col = p_exog.idxmax()
        worst_p = float(p_exog.max())

        if worst_p <= p_thresh:
            return cols, res

        cols.remove(worst_col)

        if not cols:
            mod2 = sm.tsa.SARIMAX(
                y_tr, exog=None, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False
            )
            res2 = mod2.fit(disp=False)
            return [], res2

    return cols, res


def choose_min_rmse_window(results_df, x_col="window_weeks", y_col="test_rmse", min_weeks=8):
    """
    SIMPLE: Select window with minimum RMSE (with optional min weeks filter).
    """
    if results_df.empty:
        return None

    valid = results_df[results_df[x_col] >= min_weeks].copy()
    if valid.empty:
        valid = results_df.copy()

    min_idx = valid[y_col].idxmin()
    min_weeks_sel = int(valid.loc[min_idx, x_col])
    min_rmse = valid.loc[min_idx, y_col]

    print(f"  ✓ Selected window: {min_weeks_sel} weeks (RMSE: {min_rmse:.2f})")
    return min_weeks_sel


def step1_protocol_selection_weeks(
    df: pd.DataFrame,
    store_id: int,
    test_days: int = 14,
    min_window_weeks: int = 4,
    max_window_weeks: int = 104,
    seasonal_on: bool = True,
    m: int = 7,
    p_thresh: float = 0.05,
    auto_arima_trace: bool = False
):
    """
    Protocol selection with weekly windows.
    Uses SIMPLE MINIMUM RMSE selection.
    """
    y_full, X_full = prepare_store_data(df, store_id)

    end_date = y_full.index[-1]
    test_start = end_date - pd.Timedelta(days=test_days - 1)
    test_end = end_date

    print(f"\n{'='*80}")
    print(f"PROTOCOL SELECTION - FIXED TEST PERIOD (WEEKLY WINDOWS)")
    print(f"Store ID: {store_id}")
    print(f"Test period (FIXED): {test_start.date()} to {test_end.date()}")
    print(f"Testing training windows: {min_window_weeks} to {max_window_weeks} weeks (weekly step)")
    print(f"Selection method: MINIMUM RMSE (simple)")
    print('='*80)

    results = []

    for window_weeks in range(min_window_weeks, max_window_weeks + 1):
        train_end = test_start - pd.Timedelta(days=1)
        train_start = train_end - pd.Timedelta(weeks=window_weeks) + pd.Timedelta(days=1)

        if train_start < y_full.index[0]:
            print(f"Skipping {window_weeks} weeks: insufficient data")
            continue

        y_train = y_full.loc[train_start:train_end]
        y_test = y_full.loc[test_start:test_end]
        X_train = X_full.loc[y_train.index]
        X_test = X_full.loc[y_test.index]

        try:
            order, seas, aic, bic = select_orders_with_auto_arima(
                y_train, X_train, seasonal_on=seasonal_on, m=m, trace=auto_arima_trace
            )
            selected_cols, _ = backward_elimination_exog(
                y_train, X_train, order, seas, p_thresh=p_thresh
            )

            X_tr = X_train[selected_cols] if selected_cols else None
            X_te = X_test[selected_cols] if selected_cols else None

            mod = sm.tsa.SARIMAX(
                y_train, exog=X_tr, order=order, seasonal_order=seas,
                enforce_stationarity=False, enforce_invertibility=False
            )
            res = mod.fit(disp=False)

            pred_raw = res.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_te)
            pred = enforce_non_negative(
                pred_raw,
                X_test['Open'] if 'Open' in X_test.columns else None,
                index=pred_raw.index
            )

            test_rmse = float(np.sqrt(mean_squared_error(y_test, pred)))

            results.append({
                'window_weeks': window_weeks,
                'test_rmse': test_rmse,
                'order': order,
                'seasonal_order': seas,
                'selected_cols': selected_cols,
                'aic': aic
            })

            print(f"Window {window_weeks:3d} weeks: TEST RMSE = {test_rmse:7.2f}")

        except Exception as e:
            print(f"Window {window_weeks:3d} weeks: ERROR - {str(e)[:50]}...")
            continue

    results_df = pd.DataFrame(results)
    if results_df.empty:
        return results_df, None

    optimal_window = choose_min_rmse_window(results_df, min_weeks=8)
    return results_df, optimal_window


# =============================================================================
# FORECAST GENERATION (UPDATED Open+Promo + CI fix when closed)
# =============================================================================

def generate_forecasts_for_optimal_model_weeks(
    df: pd.DataFrame,
    store_id: int,
    optimal_window_weeks: int,
    test_days: int = 14,
    forecast_days: int = 7,
    seasonal_on: bool = True,
    m: int = 7,
    p_thresh: float = 0.05,
    auto_arima_trace: bool = False
):
    """
    Generate forecasts using the optimal window size (in weeks).

    Changes made (only in forecast exog logic + CI handling):
    - Open: deterministic using historical Sunday open rate threshold rule
    - Promo: B1 deterministic weekday rule based on historical weekday promo frequency
    - CI handling: if Open=0 in future_exog, force mean=0 and CI=[0,0]
    """
    y_full, X_full = prepare_store_data(df, store_id)

    end_date = y_full.index[-1]
    test_start = end_date - pd.Timedelta(days=test_days - 1)
    test_end = end_date

    train_end = test_start - pd.Timedelta(days=1)
    train_start = train_end - pd.Timedelta(weeks=optimal_window_weeks) + pd.Timedelta(days=1)

    test_start_date = test_start.strftime('%Y-%m-%d')
    test_end_date = test_end.strftime('%Y-%m-%d')
    last_7_start = (test_end - pd.Timedelta(days=6)).strftime('%Y-%m-%d')
    last_7_end = test_end.strftime('%Y-%m-%d')

    future_start = test_end + pd.Timedelta(days=1)
    future_end = future_start + pd.Timedelta(days=forecast_days - 1)
    future_start_date = future_start.strftime('%Y-%m-%d')
    future_end_date = future_end.strftime('%Y-%m-%d')

    print(f"\n{'='*80}")
    print(f"FORECAST GENERATION - OPTIMAL MODEL")
    print(f"Store ID: {store_id}")
    print(f"Optimal window: {optimal_window_weeks} weeks")
    print(f"Training: {train_start.date()} to {train_end.date()}")
    print(f"Test period: {test_start_date} to {test_end_date}")
    print('='*80)

    y_train = y_full.loc[train_start:train_end]
    y_test = y_full.loc[test_start:test_end]
    X_train = X_full.loc[y_train.index]
    X_test = X_full.loc[y_test.index]

    print(f"\nSelecting model for optimal window...")
    order, seas, aic, bic = select_orders_with_auto_arima(
        y_train, X_train, seasonal_on=seasonal_on, m=m, trace=auto_arima_trace
    )

    selected_cols, _ = backward_elimination_exog(
        y_train, X_train, order, seas, p_thresh=p_thresh
    )

    print(f"Selected model: {order}{seas}")
    print(f"Selected exog: {selected_cols if selected_cols else 'None'}")

    X_tr = X_train[selected_cols] if selected_cols else None
    X_te = X_test[selected_cols] if selected_cols else None

    mod = sm.tsa.SARIMAX(
        y_train, exog=X_tr, order=order, seasonal_order=seas,
        enforce_stationarity=False, enforce_invertibility=False
    )
    res = mod.fit(disp=False)

    print(f"\nPredicting for full test period ({test_days} days)...")
    pred_full_test_raw = res.predict(
        start=y_test.index[0],
        end=y_test.index[-1],
        exog=X_te
    )

    pred_full_test = enforce_non_negative(
        pred_full_test_raw,
        X_test['Open'] if 'Open' in X_test.columns else None,
        index=pred_full_test_raw.index
    )

    negative_count = (pred_full_test_raw < 0).sum()
    if negative_count > 0:
        print(f"  ✓ Fixed {negative_count} negative predictions (min: {pred_full_test_raw.min():.0f} -> 0)")

    closed_count = 0
    if 'Open' in X_test.columns:
        closed_mask = (X_test['Open'] == 0) & (pred_full_test_raw != 0)
        closed_count = int(closed_mask.sum())
        if closed_count > 0:
            print(f"  ✓ Fixed {closed_count} predictions when store closed")

    test_metrics, _ = calculate_all_metrics(y_test, pred_full_test, X_test)

    last_7_days_start = test_end - pd.Timedelta(days=6)
    y_test_last_7 = y_test.loc[last_7_days_start:test_end]
    pred_last_7_raw = pred_full_test_raw.loc[last_7_days_start:test_end]
    pred_last_7 = pred_full_test.loc[last_7_days_start:test_end]

    last_7_metrics, _ = calculate_all_metrics(
        y_test_last_7,
        pred_last_7,
        X_test.loc[last_7_days_start:test_end]
    )

    # -------------------------
    # FUTURE EXOG (UPDATED)
    # -------------------------
    future_dates = pd.date_range(start=future_start, end=future_end, freq='D')
    print(f"\nPreparing future exogenous variables with deterministic Open & Promo (weekday-based)...")

    future_exog = pd.DataFrame(index=future_dates)

    if selected_cols:
        store_data = df[df['Store'] == store_id].copy()
        store_data["Date"] = pd.to_datetime(store_data["Date"], dayfirst=True)

        # Historical Sunday open rate (empirical)
        sunday_data = store_data[store_data['DayOfWeek'] == 7]
        sunday_open_rate = float((sunday_data['Open'] == 1).mean()) if len(sunday_data) > 0 else 0.0
        print(f"  • Historical Sunday open rate: {sunday_open_rate*100:.1f}%")

        # Promo weekday frequency (empirical)
        promo_by_dow = store_data.groupby("DayOfWeek")["Promo"].mean()  # 1..7
        print(f"  • Using weekday promo frequency (historical mean Promo by DOW)")

        for col in selected_cols:
            if col.startswith('DOW_'):
                dow_num = int(col.split('_')[1])
                future_exog[col] = [(1 if (d.dayofweek + 1) == dow_num else 0) for d in future_dates]

            elif col == 'Open':
                # Deterministic rule:
                # - Sundays: Open=1 if sunday_open_rate >= 0.5 else 0
                # - Other days: Open=1
                vals = []
                for d in future_dates:
                    if d.weekday() == 6:  # Sunday
                        vals.append(1 if sunday_open_rate >= 0.5 else 0)
                    else:
                        vals.append(1)
                future_exog[col] = vals

                sunday_count = sum(1 for d in future_dates if d.weekday() == 6)
                predicted_open_sundays = sum(v for i, v in enumerate(vals) if future_dates[i].weekday() == 6)
                print(f"  • Predicted Sunday openings (deterministic): {predicted_open_sundays}/{sunday_count}")

            elif col == 'Promo':
                # B1 deterministic weekday rule:
                # Promo=1 on weekday if historical mean Promo for that weekday >= 0.5
                threshold = 0.5
                vals = []
                for d in future_dates:
                    dow = d.dayofweek + 1  # 1..7
                    hist_rate = float(promo_by_dow.get(dow, 0))
                    vals.append(1 if hist_rate >= threshold else 0)
                future_exog[col] = vals

                print(f"  • Predicted promo days (weekday rule): {sum(vals)}/{len(vals)}")

            elif col == 'SchoolHoliday':
                last_year_dates = [d - pd.DateOffset(years=1) for d in future_dates]
                vals = []
                for d0 in last_year_dates:
                    match = store_data[store_data['Date'] == d0]
                    vals.append(int(match['SchoolHoliday'].iloc[0]) if len(match) > 0 else 0)
                future_exog[col] = vals

            elif col == 'IsPromoMonth':
                future_exog[col] = [1 if d.month in [3, 6, 9, 12] else 0 for d in future_dates]

            elif col == 'StateHoliday':
                future_exog[col] = [0] * len(future_dates)

            else:
                if col in X_test.columns:
                    future_exog[col] = [float(X_test[col].iloc[-1])] * len(future_dates)
                else:
                    future_exog[col] = [0.0] * len(future_dates)

    future_exog_sel = future_exog[selected_cols] if selected_cols else None

    print(f"\nGenerating forecast for next {forecast_days} days...")
    future_forecast = res.get_forecast(steps=forecast_days, exog=future_exog_sel)

    pred_future_raw = future_forecast.predicted_mean.copy()
    ci_future = future_forecast.conf_int().copy()

    # -------------------------
    # CI FIX when store closed:
    # If Open=0 -> mean=0 and CI=[0,0]
    # -------------------------
    if "Open" in future_exog.columns:
        closed_mask = (future_exog["Open"].values == 0)
        if closed_mask.any():
            idx = pred_future_raw.index
            pred_future_raw.loc[idx[closed_mask]] = 0.0
            # conf_int columns are usually ["lower Sales", "upper Sales"] or similar
            ci_future.iloc[closed_mask, 0] = 0.0
            ci_future.iloc[closed_mask, 1] = 0.0
            print(f"  ✓ Forced mean & CI to 0 for {int(closed_mask.sum())} closed day(s)")

    # Still keep non-negative enforcement (safe)
    pred_future = enforce_non_negative(
        pred_future_raw,
        future_exog['Open'] if 'Open' in future_exog.columns else None,
        index=pred_future_raw.index
    )

    future_negative = (pred_future_raw < 0).sum()
    if future_negative > 0:
        print(f"  ✓ Fixed {future_negative} negative future forecasts")

    print(f"\nForecast generation completed!")
    print(f"  • Test period: {test_start_date} to {test_end_date}")
    print(f"  • Last 7 days: {last_7_start} to {last_7_end}")
    print(f"  • Forecast period: {future_start_date} to {future_end_date}")

    return {
        'store_id': store_id,
        'optimal_window_weeks': optimal_window_weeks,
        'order': order,
        'seasonal_order': seas,
        'selected_cols': selected_cols,
        'aic': aic,
        'bic': bic,

        'test_start_date': test_start_date,
        'test_end_date': test_end_date,
        'last_7_start': last_7_start,
        'last_7_end': last_7_end,
        'future_start_date': future_start_date,
        'future_end_date': future_end_date,

        'y_train': y_train,
        'X_train': X_train,

        'y_test': y_test,
        'test_pred_raw': pred_full_test_raw,
        'test_pred': pred_full_test,
        'test_metrics': test_metrics,

        'last_7_dates': y_test_last_7.index,
        'last_7_actual': y_test_last_7,
        'last_7_pred_raw': pred_last_7_raw,
        'last_7_pred': pred_last_7,
        'last_7_metrics': last_7_metrics,

        'future_dates': future_dates,
        'future_pred_raw': pred_future_raw,
        'future_pred': pred_future,
        'future_ci': ci_future,
        'future_exog': future_exog,

        'X_test': X_test
    }


# =============================================================================
# PLOTTING + TABLES + SAVE + CV (UNCHANGED)
# =============================================================================

def plot_forecasting_results(forecast_results, store_id):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    axes[0, 0].plot(forecast_results['y_test'].index,
                    forecast_results['y_test'].values,
                    'b-', linewidth=2, label='Actual', marker='o')
    axes[0, 0].plot(forecast_results['y_test'].index,
                    forecast_results['test_pred'].values,
                    'r--', linewidth=2, label='Predicted (fixed)', marker='s')

    if not np.allclose(forecast_results['test_pred'].values,
                       forecast_results['test_pred_raw'].values, atol=1):
        axes[0, 0].plot(forecast_results['y_test'].index,
                        forecast_results['test_pred_raw'].values,
                        'g:', alpha=0.5, linewidth=1, label='Predicted (raw)')

    axes[0, 0].set_xlabel('Date')
    axes[0, 0].set_ylabel('Sales')
    axes[0, 0].set_title(f'Store {store_id}: Full Test Period\n'
                         f'{forecast_results["test_start_date"]} to {forecast_results["test_end_date"]}\n'
                         f'RMSE: {forecast_results["test_metrics"]["rmse"]:.2f}, '
                         f'MAE: {forecast_results["test_metrics"]["mae"]:.2f}')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].tick_params(axis='x', rotation=45)

    axes[0, 1].plot(forecast_results['last_7_dates'],
                    forecast_results['last_7_actual'].values,
                    'b-', linewidth=3, label='Actual', marker='o', markersize=8)
    axes[0, 1].plot(forecast_results['last_7_dates'],
                    forecast_results['last_7_pred'].values,
                    'r--', linewidth=3, label='Predicted (fixed)', marker='s', markersize=8)

    max_val = max(forecast_results['last_7_actual'].max(),
                  forecast_results['last_7_pred'].max())
    offset = max_val * 0.02 if max_val > 0 else 50

    for date, actual, pred in zip(
        forecast_results['last_7_dates'],
        forecast_results['last_7_actual'].values,
        forecast_results['last_7_pred'].values
    ):
        axes[0, 1].text(date, actual + offset, f'{actual:.0f}', ha='center', fontsize=9, fontweight='bold')
        axes[0, 1].text(date, pred - offset, f'{pred:.0f}', ha='center', fontsize=9, color='red', fontweight='bold')

    axes[0, 1].set_xlabel('Date')
    axes[0, 1].set_ylabel('Sales')
    axes[0, 1].set_title(f'Store {store_id}: Last 7 Days\n'
                         f'{forecast_results["last_7_start"]} to {forecast_results["last_7_end"]}\n'
                         f'RMSE: {forecast_results["last_7_metrics"]["rmse"]:.2f}, '
                         f'MAE: {forecast_results["last_7_metrics"]["mae"]:.2f}')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].tick_params(axis='x', rotation=45)

    axes[1, 0].plot(forecast_results['future_dates'],
                    forecast_results['future_pred'].values,
                    'g-', linewidth=3, label='Forecast (fixed)', marker='o', markersize=8)
    axes[1, 0].fill_between(forecast_results['future_dates'],
                            forecast_results['future_ci'].iloc[:, 0],
                            forecast_results['future_ci'].iloc[:, 1],
                            alpha=0.2, color='green', label='95% CI')

    if not np.allclose(forecast_results['future_pred'].values,
                       forecast_results['future_pred_raw'].values, atol=1):
        axes[1, 0].plot(forecast_results['future_dates'],
                        forecast_results['future_pred_raw'].values,
                        'b:', alpha=0.5, linewidth=1, label='Forecast (raw)')

    future_max = forecast_results['future_pred'].max()
    future_offset = future_max * 0.02 if future_max > 0 else 50

    for i, (date, pred) in enumerate(zip(
        forecast_results['future_dates'],
        forecast_results['future_pred'].values
    )):
        axes[1, 0].text(date, pred + future_offset, f'{pred:.0f}', ha='center', fontsize=9, color='green', fontweight='bold')
        ci_lower = forecast_results['future_ci'].iloc[i, 0]
        ci_upper = forecast_results['future_ci'].iloc[i, 1]
        axes[1, 0].text(date, ci_lower - future_offset, f'[{ci_lower:.0f}-{ci_upper:.0f}]',
                        ha='center', fontsize=8, color='gray', alpha=0.7)

    axes[1, 0].set_xlabel('Date')
    axes[1, 0].set_ylabel('Sales')
    axes[1, 0].set_title(f'Store {store_id}: Next {len(forecast_results["future_dates"])} Days Forecast\n'
                         f'{forecast_results["future_start_date"]} to {forecast_results["future_end_date"]}')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].tick_params(axis='x', rotation=45)

    history_days = 30
    hist_end = forecast_results['y_train'].index[-1]
    hist_start = hist_end - pd.Timedelta(days=history_days - 1)
    y_history = forecast_results['y_train'].loc[hist_start:hist_end]

    axes[1, 1].plot(y_history.index, y_history.values,
                    'b-', alpha=0.7, label='Historical (30 days)', linewidth=1.5)
    axes[1, 1].plot(forecast_results['y_test'].index, forecast_results['y_test'].values,
                    'b-', linewidth=2.5, label='Test Actual', marker='o')
    axes[1, 1].plot(forecast_results['y_test'].index, forecast_results['test_pred'].values,
                    'r--', linewidth=2, label='Test Predicted')
    axes[1, 1].plot(forecast_results['future_dates'], forecast_results['future_pred'].values,
                    'g-', linewidth=2.5, label='Future Forecast', marker='s')
    axes[1, 1].fill_between(forecast_results['future_dates'],
                            forecast_results['future_ci'].iloc[:, 0],
                            forecast_results['future_ci'].iloc[:, 1],
                            alpha=0.2, color='green', label='95% CI')

    forecast_start = forecast_results['future_dates'][0]
    axes[1, 1].axvline(x=forecast_start, color='gray', linestyle='--',
                       linewidth=1.5, alpha=0.7, label='Forecast Start')

    axes[1, 1].set_xlabel('Date')
    axes[1, 1].set_ylabel('Sales')
    axes[1, 1].set_title(f'Store {store_id}: Combined View\n'
                         f'Model: {forecast_results["order"]}{forecast_results["seasonal_order"]}')
    axes[1, 1].legend(loc='upper left', fontsize=9)
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].tick_params(axis='x', rotation=45)

    plt.suptitle(
        f'Store {store_id} - Forecasting Results\n'
        f'Optimal Training Window: {forecast_results["optimal_window_weeks"]} weeks | '
        f'AIC: {forecast_results["aic"]:.1f} | '
        f'Exog Variables: {len(forecast_results["selected_cols"]) if forecast_results["selected_cols"] else 0}',
        fontsize=14, y=1.02
    )

    plt.tight_layout()
    plt.show()


def print_forecast_tables(forecast_results):
    print(f"\n{'='*80}")
    print(f"DETAILED FORECAST TABLES - STORE {forecast_results['store_id']}")
    print('='*80)

    print(f"\n1. LAST 7 DAYS - ACTUAL vs PREDICTED:")
    print(f"   Period: {forecast_results['last_7_start']} to {forecast_results['last_7_end']}")
    print("-" * 120)
    print(f"{'Date':<12} {'Day':<6} {'Actual':<8} {'Predicted':<10} {'Raw Pred':<10} {'Error':<8} {'% Error':<8} {'Fixed':<8}")
    print("-" * 120)

    total_actual = 0
    total_pred = 0
    total_raw_pred = 0
    fixed_count = 0

    for date, actual, pred, pred_raw in zip(
        forecast_results['last_7_dates'],
        forecast_results['last_7_actual'].values,
        forecast_results['last_7_pred'].values,
        forecast_results['last_7_pred_raw'].values
    ):
        error = actual - pred
        pct_error = (error / actual * 100) if actual != 0 else 0
        day_name = date.strftime('%a')

        if abs(pred - pred_raw) > 0.01:
            fixed = "YES"
            fixed_count += 1
        else:
            fixed = "NO"

        print(f"{date.strftime('%Y-%m-%d'):<12} {day_name:<6} "
              f"{actual:<8.0f} {pred:<10.0f} {pred_raw:<10.0f} {error:<8.0f} {pct_error:<8.1f}% {fixed:<8}")

        total_actual += actual
        total_pred += pred
        total_raw_pred += pred_raw

    print("-" * 120)
    total_error = total_actual - total_pred
    total_pct_error = (total_error / total_actual * 100) if total_actual != 0 else 0
    print(f"{'TOTAL':<12} {'':<6} {total_actual:<8.0f} {total_pred:<10.0f} {total_raw_pred:<10.0f} "
          f"{total_error:<8.0f} {total_pct_error:<8.1f}%")
    print(f"  Fixed predictions: {fixed_count}/7")

    print(f"\n   PERFORMANCE METRICS (USING FIXED PREDICTIONS):")
    metrics = forecast_results['last_7_metrics']
    print(f"   • RMSE:  {metrics['rmse']:.2f}")
    print(f"   • MAE:   {metrics['mae']:.2f}")
    print(f"   • MAPE:  {metrics['mape']:.1f}%" if not np.isnan(metrics['mape']) else "   • MAPE:  N/A (all zeros)")
    print(f"   • SMAPE: {metrics['smape']:.1f}%" if not np.isnan(metrics['smape']) else "   • SMAPE: N/A")
    print(f"   • Bias:  {metrics['bias']:.1f}")
    print(f"   • R²:    {metrics['r2_adj']:.3f}" if not np.isnan(metrics['r2_adj']) else "   • R²:    N/A")

    print(f"\n\n2. NEXT {len(forecast_results['future_dates'])} DAYS - FORECAST:")
    print(f"   Period: {forecast_results['future_start_date']} to {forecast_results['future_end_date']}")
    print("-" * 120)
    print(f"{'Date':<12} {'Day':<6} {'Forecast':<10} {'Raw Forecast':<12} {'95% CI':<20} {'Fixed':<8}")
    print("-" * 120)

    total_forecast = 0
    total_raw_forecast = 0
    total_ci_width = 0
    future_fixed_count = 0

    for i, date in enumerate(forecast_results['future_dates']):
        pred = forecast_results['future_pred'].iloc[i]
        pred_raw = forecast_results['future_pred_raw'].iloc[i]
        ci_lower = forecast_results['future_ci'].iloc[i, 0]
        ci_upper = forecast_results['future_ci'].iloc[i, 1]
        ci_width = ci_upper - ci_lower
        day_name = date.strftime('%a')

        if abs(pred - pred_raw) > 0.01:
            fixed = "YES"
            future_fixed_count += 1
        else:
            fixed = "NO"

        print(f"{date.strftime('%Y-%m-%d'):<12} {day_name:<6} "
              f"{pred:<10.0f} {pred_raw:<12.0f} [{ci_lower:.0f}-{ci_upper:.0f}]   {fixed:<8}")

        total_forecast += pred
        total_raw_forecast += pred_raw
        total_ci_width += ci_width

    print("-" * 120)
    print(f"{'TOTAL':<12} {'':<6} {total_forecast:<10.0f} {total_raw_forecast:<12.0f}")
    print(f"{'AVG CI':<12} {'':<6} {'':<10} {'':<12} ±{total_ci_width/7:.0f}")
    print(f"  Fixed forecasts: {future_fixed_count}/7")

    print(f"\n   FORECAST SUMMARY (USING FIXED PREDICTIONS):")
    print(f"   • Average daily: {total_forecast/7:.0f}")
    print(f"   • Minimum:       {forecast_results['future_pred'].min():.0f}")
    print(f"   • Maximum:       {forecast_results['future_pred'].max():.0f}")
    if forecast_results['future_pred'].mean() > 0:
        range_pct = ((forecast_results['future_pred'].max() - forecast_results['future_pred'].min()) /
                     forecast_results['future_pred'].mean() * 100)
        print(f"   • Range:         ±{range_pct:.1f}%")
    else:
        print(f"   • Range:         N/A (zero mean)")

    print(f"\n\n3. PERFORMANCE SUMMARY (USING FIXED PREDICTIONS):")
    print("-" * 90)
    print(f"{'Metric':<20} {'Full Test (14d)':<15} {'Last 7 Days':<15}")
    print("-" * 90)

    test_metrics = forecast_results['test_metrics']
    last7_metrics = forecast_results['last_7_metrics']

    print(f"{'RMSE':<20} {test_metrics['rmse']:<15.2f} {last7_metrics['rmse']:<15.2f}")
    print(f"{'MAE':<20} {test_metrics['mae']:<15.2f} {last7_metrics['mae']:<15.2f}")

    if not np.isnan(test_metrics['mape']):
        print(f"{'MAPE (%)':<20} {test_metrics['mape']:<15.1f} ", end="")
        print(f"{last7_metrics['mape']:<15.1f}" if not np.isnan(last7_metrics['mape']) else f"{'N/A':<15}")
    else:
        print(f"{'MAPE (%)':<20} {'N/A':<15} ", end="")
        print(f"{last7_metrics['mape']:<15.1f}" if not np.isnan(last7_metrics['mape']) else f"{'N/A':<15}")

    if not np.isnan(test_metrics['smape']):
        print(f"{'SMAPE (%)':<20} {test_metrics['smape']:<15.1f} ", end="")
        print(f"{last7_metrics['smape']:<15.1f}" if not np.isnan(last7_metrics['smape']) else f"{'N/A':<15}")
    else:
        print(f"{'SMAPE (%)':<20} {'N/A':<15} ", end="")
        print(f"{last7_metrics['smape']:<15.1f}" if not np.isnan(last7_metrics['smape']) else f"{'N/A':<15}")

    print(f"{'Bias':<20} {test_metrics['bias']:<15.1f} {last7_metrics['bias']:<15.1f}")

    print(f"{'R²':<20} ", end="")
    print(f"{test_metrics['r2_adj']:<15.3f} " if not np.isnan(test_metrics['r2_adj']) else f"{'N/A':<15} ", end="")
    print(f"{last7_metrics['r2_adj']:<15.3f}" if not np.isnan(last7_metrics['r2_adj']) else f"{'N/A':<15}")

    print("-" * 90)

    print(f"\n4. MODEL INFORMATION:")
    print(f"   • Optimal training window: {forecast_results['optimal_window_weeks']} weeks")
    print(f"   • SARIMAX order: {forecast_results['order']}{forecast_results['seasonal_order']}")
    print(f"   • AIC: {forecast_results['aic']:.1f}")
    print(f"   • BIC: {forecast_results['bic']:.1f}")
    print(f"   • Exogenous variables: {forecast_results['selected_cols'] if forecast_results['selected_cols'] else 'None'}")

    print('\n' + '=' * 80)


def plot_protocol_selection(results_df, store_id, optimal_window_weeks):
    if len(results_df) < 2:
        print("Not enough data for visualization")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(results_df['window_weeks'], results_df['test_rmse'], 'o-', linewidth=2, markersize=4)
    axes[0].set_xlabel('Training Window (weeks)')
    axes[0].set_ylabel('TEST RMSE')
    axes[0].set_title(f'Store {store_id}: TEST RMSE vs Training Window')
    axes[0].grid(True, alpha=0.3)

    min_idx = results_df['test_rmse'].idxmin()
    min_weeks = results_df.loc[min_idx, 'window_weeks']
    min_rmse = results_df.loc[min_idx, 'test_rmse']
    axes[0].scatter([min_weeks], [min_rmse], color='green', s=150, zorder=5, marker='*',
                    label=f'Min RMSE: {min_weeks} weeks')
    axes[0].legend()

    axes[1].hist(results_df['window_weeks'], bins=20, alpha=0.7, color='blue', edgecolor='black')
    axes[1].set_xlabel('Training Window (weeks)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of Windows Tested')
    axes[1].grid(True, alpha=0.3)

    if optimal_window_weeks is not None:
        axes[1].axvline(x=optimal_window_weeks, color='red', linestyle='--', linewidth=2,
                        label=f'Selected: {optimal_window_weeks} weeks')
        axes[1].legend()

    plt.suptitle(f'Protocol Selection Results - Store {store_id}\n'
                 f'Optimal Training Window: {optimal_window_weeks} weeks',
                 fontsize=14, y=1.05)
    plt.tight_layout()
    plt.show()


def save_all_results(results_df, forecast_results, store_id, optimal_window_weeks):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    protocol_file = f"step1_protocol_results_store_{store_id}_{timestamp}.csv"
    results_df.to_csv(protocol_file, index=False)

    forecast_data = []

    for date, actual, pred, pred_raw in zip(
        forecast_results['last_7_dates'],
        forecast_results['last_7_actual'].values,
        forecast_results['last_7_pred'].values,
        forecast_results['last_7_pred_raw'].values
    ):
        forecast_data.append({
            'date': date,
            'period': 'last_7_days',
            'type': 'actual_vs_predicted',
            'actual': actual,
            'predicted': pred,
            'predicted_raw': pred_raw,
            'fixed': 'YES' if abs(pred - pred_raw) > 0.01 else 'NO',
            'error': actual - pred,
            'pct_error': ((actual - pred) / actual * 100) if actual != 0 else 0
        })

    for i, date in enumerate(forecast_results['future_dates']):
        pred = forecast_results['future_pred'].iloc[i]
        pred_raw = forecast_results['future_pred_raw'].iloc[i]
        forecast_data.append({
            'date': date,
            'period': 'next_7_days',
            'type': 'forecast',
            'predicted': pred,
            'predicted_raw': pred_raw,
            'fixed': 'YES' if abs(pred - pred_raw) > 0.01 else 'NO',
            'ci_lower': forecast_results['future_ci'].iloc[i, 0],
            'ci_upper': forecast_results['future_ci'].iloc[i, 1]
        })

    forecast_df = pd.DataFrame(forecast_data)
    forecast_file = f"step1_forecast_details_store_{store_id}_{timestamp}.csv"
    forecast_df.to_csv(forecast_file, index=False)

    summary_file = f"step1_summary_store_{store_id}_{timestamp}.txt"
    with open(summary_file, 'w') as f:
        f.write(f"STEP 1 COMPLETE RESULTS - STORE {store_id}\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Optimal training window: {optimal_window_weeks} weeks\n\n")

        f.write("MODEL INFORMATION:\n")
        f.write(f"  • SARIMAX order: {forecast_results['order']}{forecast_results['seasonal_order']}\n")
        f.write(f"  • AIC: {forecast_results['aic']:.1f}\n")
        f.write(f"  • BIC: {forecast_results['bic']:.1f}\n")
        f.write(f"  • Exogenous variables: {forecast_results['selected_cols'] if forecast_results['selected_cols'] else 'None'}\n\n")

        f.write("PERFORMANCE METRICS (USING FIXED PREDICTIONS):\n")
        test_metrics = forecast_results['test_metrics']
        last7_metrics = forecast_results['last_7_metrics']

        f.write(f"  • Full test (14 days): {forecast_results['test_start_date']} to {forecast_results['test_end_date']}\n")
        f.write(f"    - RMSE:  {test_metrics['rmse']:.2f}\n")
        f.write(f"    - MAE:   {test_metrics['mae']:.2f}\n")
        if not np.isnan(test_metrics['mape']):
            f.write(f"    - MAPE:  {test_metrics['mape']:.1f}%\n")
        if not np.isnan(test_metrics['smape']):
            f.write(f"    - SMAPE: {test_metrics['smape']:.1f}%\n")
        f.write(f"    - Bias:  {test_metrics['bias']:.1f}\n")
        if not np.isnan(test_metrics['r2_adj']):
            f.write(f"    - R²:    {test_metrics['r2_adj']:.3f}\n")

        f.write(f"\n  • Last 7 days: {forecast_results['last_7_start']} to {forecast_results['last_7_end']}\n")
        f.write(f"    - RMSE:  {last7_metrics['rmse']:.2f}\n")
        f.write(f"    - MAE:   {last7_metrics['mae']:.2f}\n")
        if not np.isnan(last7_metrics['mape']):
            f.write(f"    - MAPE:  {last7_metrics['mape']:.1f}%\n")
        if not np.isnan(last7_metrics['smape']):
            f.write(f"    - SMAPE: {last7_metrics['smape']:.1f}%\n")
        f.write(f"    - Bias:  {last7_metrics['bias']:.1f}\n")
        if not np.isnan(last7_metrics['r2_adj']):
            f.write(f"    - R²:    {last7_metrics['r2_adj']:.3f}\n\n")

        f.write(f"FORECAST SUMMARY (Next {len(forecast_results['future_dates'])} days): "
                f"{forecast_results['future_start_date']} to {forecast_results['future_end_date']}\n")
        f.write(f"  • Total forecast sales: {forecast_results['future_pred'].sum():.0f}\n")
        f.write(f"  • Average daily forecast: {forecast_results['future_pred'].mean():.0f}\n")
        f.write(f"  • Range: {forecast_results['future_pred'].min():.0f} - {forecast_results['future_pred'].max():.0f}\n\n")

        f.write("PROTOCOL SELECTION RESULTS:\n")
        f.write("-" * 120 + "\n")
        for _, row in results_df.iterrows():
            f.write(f"Window {row['window_weeks']:3d} weeks | TEST RMSE: {row['test_rmse']:7.2f}\n")

    print(f"\nResults saved:")
    print(f"  • Protocol selection: {protocol_file}")
    print(f"  • Forecast details: {forecast_file}")
    print(f"  • Summary: {summary_file}")

    return protocol_file, forecast_file, summary_file


def time_series_cv_rmse(y, X, order, seas, selected_cols, n_splits=7, min_test_size=7):
    n = len(y)
    max_splits = max(2, min(n_splits, (n // min_test_size) - 1))
    if max_splits < 2:
        return np.nan, 1

    tscv = TimeSeriesSplit(n_splits=max_splits)
    rmses = []
    cols = selected_cols if selected_cols else []

    for train_idx, test_idx in tscv.split(y):
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        X_tr = X.iloc[train_idx][cols] if cols else None
        X_te = X.iloc[test_idx][cols] if cols else None

        mod = sm.tsa.SARIMAX(
            y_tr, exog=X_tr, order=order, seasonal_order=seas,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        pred = res.predict(start=y_te.index[0], end=y_te.index[-1], exog=X_te)
        pred = enforce_non_negative(pred)

        rmses.append(np.sqrt(mean_squared_error(y_te, pred)))

    return float(np.mean(rmses)), max_splits


def run_step1_complete_weeks(
    df: pd.DataFrame,
    store_id: int = 1,
    test_days: int = 14,
    min_window_weeks: int = 4,
    max_window_weeks: int = 104,
    seasonal_on: bool = True,
    m: int = 7,
    p_thresh: float = 0.05,
    auto_arima_trace: bool = True,
    n_splits: int = 7
):
    print("\n" + "=" * 80)
    print("MASTER THESIS - STEP 1: PROTOCOL SELECTION (WEEKLY WINDOWS) + FORECASTING")
    print(f"Store ID: {store_id}")
    print("=" * 80)

    results_df, optimal_window_weeks = step1_protocol_selection_weeks(
        df=df,
        store_id=store_id,
        test_days=test_days,
        min_window_weeks=min_window_weeks,
        max_window_weeks=max_window_weeks,
        seasonal_on=seasonal_on,
        m=m,
        p_thresh=p_thresh,
        auto_arima_trace=auto_arima_trace
    )

    if results_df.empty or optimal_window_weeks is None:
        print("\n❌ Protocol selection failed (no valid windows).")
        return None, None, None

    plot_protocol_selection(results_df, store_id, optimal_window_weeks)

    forecast_results = generate_forecasts_for_optimal_model_weeks(
        df=df,
        store_id=store_id,
        optimal_window_weeks=optimal_window_weeks,
        test_days=test_days,
        forecast_days=7,
        seasonal_on=seasonal_on,
        m=m,
        p_thresh=p_thresh,
        auto_arima_trace=auto_arima_trace
    )

    cv_rmse, used_splits = time_series_cv_rmse(
        y=forecast_results["y_train"],
        X=forecast_results["X_train"],
        order=forecast_results["order"],
        seas=forecast_results["seasonal_order"],
        selected_cols=forecast_results["selected_cols"],
        n_splits=n_splits,
        min_test_size=7
    )
    print(f"\nCV (train-only) average RMSE: {cv_rmse if np.isfinite(cv_rmse) else 'NA'}  | folds used: {used_splits}")
    print(f"(n_splits={n_splits} is OK if there's enough data; code auto-reduces if not.)")

    print_forecast_tables(forecast_results)
    plot_forecasting_results(forecast_results, store_id)

    save_all_results(results_df, forecast_results, store_id, optimal_window_weeks)

    print("\n✅ STEP 1 COMPLETED SUCCESSFULLY!")
    print(f"   Selected window: {optimal_window_weeks} weeks")
    print(f"   For Step 2, start with window_weeks = {optimal_window_weeks} for all stores.")

    return results_df, forecast_results, optimal_window_weeks

# =============================================================================
# MAIN EXECUTION - USING YOUR REAL DATA
# =============================================================================
if __name__ == "__main__":
    # Your real df is already loaded from the earlier code
    # Just verify it and run the analysis
    
    print("\n" + "="*80)
    print("VERIFYING REAL DATA BEFORE ANALYSIS")
    print("="*80)
    
    print(f"Data shape: {df.shape}")
    print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"Number of unique stores: {df['Store'].nunique()}")
    print(f"\nColumns in dataframe:")
    print(df.columns.tolist())
    
    # Check if required columns exist
    required_cols = ['Store', 'Date', 'Sales', 'Open', 'Promo', 'SchoolHoliday', 
                     'StateHoliday', 'DayOfWeek', 'IsPromoMonth']
    
    print("\nChecking required columns:")
    all_good = True
    for col in required_cols:
        if col in df.columns:
            print(f"  ✓ {col}")
        else:
            print(f"  ✗ {col} - MISSING!")
            all_good = False
    
    if not all_good:
        print("\n❌ Missing required columns. Please check your data loading code.")
        exit()
    
    # Check PromoInterval (optional, but nice to have)
    if 'PromoInterval' in df.columns:
        print(f"\nPromoInterval sample:")
        print(df['PromoInterval'].value_counts().head())
    
    # Choose a store to analyze (change this to any store ID you want)
    STORE_ID = 1
    
    print(f"\n{'-'*50}")
    print(f"Analyzing Store {STORE_ID}")
    print('-'*50)
    
    # Check if store exists
    if STORE_ID not in df['Store'].unique():
        print(f"❌ Store {STORE_ID} not found in data!")
        print(f"Available stores: {sorted(df['Store'].unique())[:10]}...")
        exit()
    
    # Check Sunday statistics for the selected store
    sunday_data = df[(df['Store'] == STORE_ID) & (df['DayOfWeek'] == 7)]
    print(f"\nStore {STORE_ID} Sunday statistics:")
    print(f"  Total Sundays: {len(sunday_data)}")
    print(f"  Sundays open: {sunday_data['Open'].sum()}")
    print(f"  Sundays closed: {len(sunday_data) - sunday_data['Open'].sum()}")
    if len(sunday_data[sunday_data['Open'] == 1]) > 0:
        print(f"  Average sales on open Sundays: {sunday_data[sunday_data['Open'] == 1]['Sales'].mean():.0f}")
    
    # Check data availability for this store
    store_dates = df[df['Store'] == STORE_ID]['Date']
    print(f"\nStore {STORE_ID} data range: {store_dates.min()} to {store_dates.max()}")
    print(f"Total records for store {STORE_ID}: {len(df[df['Store'] == STORE_ID])}")
    
    print("\n" + "="*80)
    print(f"STARTING ANALYSIS FOR STORE {STORE_ID}")
    print("="*80)
    
    results_df, forecast_results, optimal_window_weeks = run_step1_complete_weeks(
        df=df,  # Using your REAL dataframe
        store_id=STORE_ID,
        test_days=14,
        min_window_weeks=4,
        max_window_weeks=104,
        seasonal_on=True,
        m=7,
        p_thresh=0.05,
        auto_arima_trace=True,
        n_splits=7
    )


In [ ]:
#real data 2-25 stores

# final 2-25 sstores

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import traceback
from scipy import stats

import pmdarima as pm
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def enforce_non_negative(predictions, open_status=None, index=None):
    """Ensure predictions are non-negative and zero when store closed."""
    predictions = np.array(predictions)
    predictions = np.maximum(predictions, 0)

    if open_status is not None:
        open_status = np.array(open_status)
        predictions[open_status == 0] = 0

    if index is not None:
        return pd.Series(predictions, index=index)
    return predictions


def calculate_mape_safe(actual, predicted):
    """Safe MAPE calculation that handles zero values."""
    actual = np.array(actual)
    predicted = np.array(predicted)

    mask = actual != 0
    if mask.sum() == 0:
        return np.nan

    predicted = enforce_non_negative(predicted)
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


def calculate_smape(actual, predicted):
    """Symmetric MAPE - handles zero values better."""
    actual = np.array(actual)
    predicted = np.array(predicted)

    predicted = enforce_non_negative(predicted)

    denominator = (np.abs(actual) + np.abs(predicted))
    mask = denominator != 0
    if mask.sum() == 0:
        return np.nan

    return 100 / len(actual) * np.sum(2 * np.abs(predicted[mask] - actual[mask]) / denominator[mask])


def calculate_all_metrics(y_true, y_pred, X_test=None):
    """Calculate all performance metrics safely."""
    metrics = {}

    if X_test is not None and 'Open' in X_test.columns:
        y_pred_clean = enforce_non_negative(
            y_pred, X_test['Open'],
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )
    else:
        y_pred_clean = enforce_non_negative(
            y_pred,
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )

    metrics['rmse'] = float(np.sqrt(mean_squared_error(y_true, y_pred_clean)))
    metrics['mae'] = float(mean_absolute_error(y_true, y_pred_clean))
    metrics['mape'] = calculate_mape_safe(y_true, y_pred_clean)
    metrics['smape'] = calculate_smape(y_true, y_pred_clean)
    metrics['bias'] = float(np.mean(y_pred_clean - y_true))

    ss_res = np.sum((y_true - y_pred_clean) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    metrics['r2_adj'] = float(1 - (ss_res / ss_tot)) if ss_tot > 0 else np.nan

    return metrics, y_pred_clean


# =============================================================================
# DATA PREP PER STORE
# =============================================================================

def prepare_store_data(df, store_id):
    """Prepare data for a specific store. Assumes df is your REAL cleaned dataset."""
    s = df[df["Store"] == store_id].copy()
    s["Date"] = pd.to_datetime(s["Date"], errors="coerce")  # robust
    s = s.sort_values("Date").set_index("Date")

    y = s["Sales"].astype(float).sort_index()

    # Required exog columns
    X = s[["Open", "Promo", "SchoolHoliday", "StateHoliday", "IsPromoMonth", "DayOfWeek"]].copy().loc[y.index]

    # types
    X["Open"] = X["Open"].astype(int)
    X["Promo"] = X["Promo"].astype(int)
    X["SchoolHoliday"] = X["SchoolHoliday"].astype(int)
    X["IsPromoMonth"] = X["IsPromoMonth"].astype(int)

    # StateHoliday may be '0','a','b','c' or numeric; convert to 0/1
    X["StateHoliday"] = (X["StateHoliday"].astype(str) != "0").astype(int)

    # Day-of-week dummies (1..7)
    dow = pd.get_dummies(X["DayOfWeek"], prefix="DOW", drop_first=True).astype(float)

    X_num = X.drop(columns=["DayOfWeek"]).astype(float)
    Xcand = pd.concat([X_num, dow], axis=1).astype(float)

    # return store_data with Date column for future-exog logic
    return y, Xcand, s.reset_index()


# =============================================================================
# MODEL SELECTION + EXOG SELECTION
# =============================================================================

def select_orders_with_auto_arima(y_tr, X_tr, seasonal_on=True, m=7, trace=False):
    """Select SARIMAX orders using auto_arima (AIC)."""
    if seasonal_on:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=True,
            m=m,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            D=None,
            max_p=3, max_q=3,
            max_P=2, max_Q=2,
            max_d=0,
            max_D=1,
            start_p=1, start_q=1,
            start_P=0, start_Q=0,
            test='adf',
            seasonal_test='ocsb',
            n_fits=20,
            with_intercept=True
        )
        return sel.order, sel.seasonal_order, float(sel.aic()), float(sel.bic())
    else:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=False,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            max_p=3, max_q=3,
            max_d=0,
            start_p=1, start_q=1,
            test='adf',
            n_fits=15,
            with_intercept=True
        )
        return sel.order, (0, 0, 0, 0), float(sel.aic()), float(sel.bic())


def backward_elimination_exog(y_tr, X_tr, order, seasonal_order, p_thresh=0.05, max_iter=50):
    """Select significant exogenous variables using p-values (backward elimination)."""
    cols = list(X_tr.columns)

    if len(cols) == 0:
        mod = sm.tsa.SARIMAX(
            y_tr, exog=None, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        return [], res

    for _ in range(max_iter):
        X_use = X_tr[cols] if cols else None

        mod = sm.tsa.SARIMAX(
            y_tr, exog=X_use, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        if not cols:
            return [], res

        pvals = res.pvalues
        p_exog = pvals[[c for c in pvals.index if c in cols]]

        if len(p_exog) == 0:
            return cols, res

        worst_col = p_exog.idxmax()
        worst_p = float(p_exog.max())

        if worst_p <= p_thresh:
            return cols, res

        cols.remove(worst_col)

        if not cols:
            mod2 = sm.tsa.SARIMAX(
                y_tr, exog=None, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False
            )
            res2 = mod2.fit(disp=False)
            return [], res2

    return cols, res


# =============================================================================
# FUTURE EXOG LOGIC (B1 - deterministic)
# =============================================================================

def build_future_exog_B1(store_data, selected_cols, future_dates):
    """
    Deterministic future exog:
    - Open: Sundays open if historical Sunday open-rate >= 0.5, else closed on Sundays; non-Sunday open=1
    - Promo: promo on a weekday if historical promo-rate for that weekday >= 0.5
    - SchoolHoliday: use last-year same date if available, else 0
    - StateHoliday: 0 (unknown future)
    - IsPromoMonth: months in [3,6,9,12]
    - DOW_x: generated from future dates
    """
    future_exog = pd.DataFrame(index=future_dates)

    sd = store_data.copy()
    if "Date" not in sd.columns:
        raise ValueError("store_data must include a 'Date' column.")
    sd["Date"] = pd.to_datetime(sd["Date"], errors="coerce")
    sd = sd.sort_values("Date")

    sunday_data = sd[sd['DayOfWeek'] == 7]
    sunday_open_rate = float((sunday_data['Open'] == 1).mean()) if len(sunday_data) > 0 else 0.0

    promo_by_dow = sd.groupby("DayOfWeek")["Promo"].mean()  # 1..7

    for col in selected_cols:
        if col.startswith('DOW_'):
            dow_num = int(col.split('_')[1])
            future_exog[col] = [(1 if (d.dayofweek + 1) == dow_num else 0) for d in future_dates]

        elif col == 'Open':
            vals = []
            for d in future_dates:
                if d.weekday() == 6:  # Sunday
                    vals.append(1 if sunday_open_rate >= 0.5 else 0)
                else:
                    vals.append(1)
            future_exog[col] = vals

        elif col == 'Promo':
            vals = []
            for d in future_dates:
                dow = d.dayofweek + 1
                hist_rate = float(promo_by_dow.get(dow, 0))
                vals.append(1 if hist_rate >= 0.5 else 0)
            future_exog[col] = vals

        elif col == 'SchoolHoliday':
            sd_idx = sd.set_index("Date")
            last_year_dates = [d - pd.DateOffset(years=1) for d in future_dates]
            vals = []
            for d0 in last_year_dates:
                d0n = pd.Timestamp(d0).normalize()
                if d0n in sd_idx.index:
                    vals.append(int(sd_idx.loc[d0n, 'SchoolHoliday']))
                else:
                    vals.append(0)
            future_exog[col] = vals

        elif col == 'IsPromoMonth':
            future_exog[col] = [1 if d.month in [3, 6, 9, 12] else 0 for d in future_dates]

        elif col == 'StateHoliday':
            future_exog[col] = [0] * len(future_dates)

        else:
            future_exog[col] = [0.0] * len(future_dates)

    return future_exog, sunday_open_rate


def force_ci_zero_on_closed(pred_mean, ci_df, open_series):
    """Force predictions and CI to 0 when Open==0."""
    open_arr = np.array(open_series).astype(int)
    closed = open_arr == 0
    if closed.any():
        idx = pred_mean.index
        pred_mean.loc[idx[closed]] = 0.0
        ci_df.iloc[closed, 0] = 0.0
        ci_df.iloc[closed, 1] = 0.0
    return pred_mean, ci_df


def sunday_category(rate):
    if rate == 0:
        return "Always Closed (0%)"
    if rate == 1:
        return "Always Open (100%)"
    return "Sometimes Open (1-99%)"


# =============================================================================
# FORECAST GENERATION FOR A SINGLE STORE (FIXED WINDOW)
# =============================================================================

def generate_forecast_for_store(
    df: pd.DataFrame,
    store_id: int,
    window_weeks: int = 9,
    test_days: int = 14,
    forecast_days: int = 7,
    seasonal_on: bool = True,
    m: int = 7,
    p_thresh: float = 0.05,
    auto_arima_trace: bool = False
):
    y_full, X_full, store_data_raw = prepare_store_data(df, store_id)

    end_date = y_full.index[-1]
    test_start = end_date - pd.Timedelta(days=test_days - 1)
    test_end = end_date

    train_end = test_start - pd.Timedelta(days=1)
    train_start = train_end - pd.Timedelta(weeks=window_weeks) + pd.Timedelta(days=1)

    if train_start < y_full.index[0]:
        raise ValueError(f"Insufficient data for store {store_id}: need {window_weeks} weeks before test start.")

    y_train = y_full.loc[train_start:train_end]
    y_test = y_full.loc[test_start:test_end]
    X_train = X_full.loc[y_train.index]
    X_test = X_full.loc[y_test.index]

    order, seas, aic, bic = select_orders_with_auto_arima(
        y_train, X_train, seasonal_on=seasonal_on, m=m, trace=auto_arima_trace
    )

    selected_cols, _ = backward_elimination_exog(
        y_train, X_train, order, seas, p_thresh=p_thresh
    )

    X_tr = X_train[selected_cols] if selected_cols else None
    X_te = X_test[selected_cols] if selected_cols else None

    mod = sm.tsa.SARIMAX(
        y_train, exog=X_tr, order=order, seasonal_order=seas,
        enforce_stationarity=False, enforce_invertibility=False
    )
    res = mod.fit(disp=False)

    start_i = len(y_train)
    end_i = len(y_train) + len(y_test) - 1

    pred_test_raw = res.predict(start=start_i, end=end_i, exog=X_te)
    pred_test_raw.index = y_test.index

    pred_test = enforce_non_negative(
        pred_test_raw,
        X_test['Open'] if 'Open' in X_test.columns else None,
        index=pred_test_raw.index
    )

    test_metrics, _ = calculate_all_metrics(y_test, pred_test, X_test)

    last_7_start = test_end - pd.Timedelta(days=6)
    y_test_last7 = y_test.loc[last_7_start:test_end]
    pred_last7_raw = pred_test_raw.loc[last_7_start:test_end]
    pred_last7 = pred_test.loc[last_7_start:test_end]

    last7_metrics, _ = calculate_all_metrics(
        y_test_last7, pred_last7,
        X_test.loc[last_7_start:test_end] if X_test is not None else None
    )

    future_dates = pd.date_range(
        start=test_end + pd.Timedelta(days=1),
        periods=forecast_days,
        freq='D'
    )

    if selected_cols:
        future_exog, _ = build_future_exog_B1(store_data_raw, selected_cols, future_dates)
        future_exog_sel = future_exog[selected_cols]
    else:
        future_exog = pd.DataFrame(index=future_dates)
        future_exog_sel = None

    future_forecast = res.get_forecast(steps=forecast_days, exog=future_exog_sel)
    pred_future_raw = future_forecast.predicted_mean.copy()
    ci_future = future_forecast.conf_int().copy()

    if "Open" in future_exog.columns:
        pred_future_raw, ci_future = force_ci_zero_on_closed(pred_future_raw, ci_future, future_exog["Open"])

    pred_future = enforce_non_negative(
        pred_future_raw,
        future_exog['Open'] if 'Open' in future_exog.columns else None,
        index=pred_future_raw.index
    )

    sunday_rate_full = float(
        (store_data_raw[store_data_raw['DayOfWeek'] == 7]['Open'] == 1).mean()
    ) if len(store_data_raw[store_data_raw['DayOfWeek'] == 7]) > 0 else 0.0

    avg_sales = float(y_train.mean())
    promo_rate = float((store_data_raw['Promo'] == 1).mean())

    return {
        'store_id': store_id,
        'success': True,
        'forecast_results': {
            'store_id': store_id,
            'optimal_window_weeks': window_weeks,
            'order': order,
            'seasonal_order': seas,
            'selected_cols': selected_cols,
            'aic': aic,
            'bic': bic,

            'train_start_date': train_start.strftime('%Y-%m-%d'),
            'train_end_date': train_end.strftime('%Y-%m-%d'),
            'test_start_date': test_start.strftime('%Y-%m-%d'),
            'test_end_date': test_end.strftime('%Y-%m-%d'),

            'last_7_start': last_7_start.strftime('%Y-%m-%d'),
            'last_7_end': test_end.strftime('%Y-%m-%d'),

            'future_start_date': future_dates[0].strftime('%Y-%m-%d'),
            'future_end_date': future_dates[-1].strftime('%Y-%m-%d'),

            'y_train': y_train,
            'y_test': y_test,

            'test_pred_raw': pred_test_raw,
            'test_pred': pred_test,
            'test_metrics': test_metrics,

            'last_7_dates': y_test_last7.index,
            'last_7_actual': y_test_last7,
            'last_7_pred_raw': pred_last7_raw,
            'last_7_pred': pred_last7,
            'last_7_metrics': last7_metrics,

            'future_dates': future_dates,
            'future_pred_raw': pred_future_raw,
            'future_pred': pred_future,
            'future_ci': ci_future,
            'future_exog': future_exog,

            'X_test': X_test
        },
        'store_characteristics': {
            'store_id': store_id,
            'sunday_open_rate': sunday_rate_full,
            'sunday_category': sunday_category(sunday_rate_full),
            'avg_sales': avg_sales,
            'promo_rate': promo_rate,
            'model_order': str(order),
            'seasonal_order': str(seas),
            'selected_cols': selected_cols,
            'rmse': test_metrics['rmse'],
            'mae': test_metrics['mae'],
            'mape': test_metrics['mape'],
            'smape': test_metrics['smape'],
            'bias': test_metrics['bias'],
            'r2': test_metrics['r2_adj'],
            'aic': aic,
            'bic': bic
        }
    }


# =============================================================================
# PLOTTING + TABLES (UPDATED: bottom-right is CONNECTED)
# =============================================================================

def plot_forecasting_results(forecast_results, store_id):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # --- (0,0) Full test ---
    axes[0, 0].plot(forecast_results['y_test'].index,
                    forecast_results['y_test'].values,
                    linewidth=2, label='Actual', marker='o')
    axes[0, 0].plot(forecast_results['y_test'].index,
                    forecast_results['test_pred'].values,
                    linestyle='--', linewidth=2, label='Predicted (fixed)', marker='s')
    axes[0, 0].set_title(f"Store {store_id} - Full Test ({forecast_results['test_start_date']} to {forecast_results['test_end_date']})")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].tick_params(axis='x', rotation=45)

    # --- (0,1) Last 7 days ---
    axes[0, 1].plot(forecast_results['last_7_dates'],
                    forecast_results['last_7_actual'].values,
                    linewidth=2, label='Actual', marker='o')
    axes[0, 1].plot(forecast_results['last_7_dates'],
                    forecast_results['last_7_pred'].values,
                    linestyle='--', linewidth=2, label='Predicted', marker='s')
    axes[0, 1].set_title(f"Store {store_id} - Last 7 Days")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].tick_params(axis='x', rotation=45)

    # --- (1,0) Future forecast ---
    axes[1, 0].plot(forecast_results['future_dates'],
                    forecast_results['future_pred'].values,
                    linewidth=2, label='Forecast', marker='o')
    axes[1, 0].fill_between(forecast_results['future_dates'],
                            forecast_results['future_ci'].iloc[:, 0],
                            forecast_results['future_ci'].iloc[:, 1],
                            alpha=0.2, label='95% CI')
    axes[1, 0].set_title(f"Store {store_id} - Next {len(forecast_results['future_dates'])} Days Forecast")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].tick_params(axis='x', rotation=45)

    # --- (1,1) Combined plot (CONNECTED LINE) ---
    # Actual test
    axes[1, 1].plot(forecast_results['y_test'].index,
                    forecast_results['y_test'].values,
                    label="Test Actual", marker='o')

    # Build ONE connected prediction series (test + future)
    test_pred_series = pd.Series(
        np.asarray(forecast_results['test_pred']),
        index=forecast_results['y_test'].index
    )
    future_pred_series = pd.Series(
        np.asarray(forecast_results['future_pred']),
        index=forecast_results['future_dates']
    )
    pred_all = pd.concat([test_pred_series, future_pred_series])

    axes[1, 1].plot(pred_all.index,
                    pred_all.values,
                    label="Pred (Test+Future)", linestyle='--', marker='s')

    # CI only for future
    axes[1, 1].fill_between(forecast_results['future_dates'],
                            forecast_results['future_ci'].iloc[:, 0],
                            forecast_results['future_ci'].iloc[:, 1],
                            alpha=0.2, label='95% CI')

    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()


def print_forecast_tables(forecast_results):
    print("\n" + "=" * 80)
    print(f"STORE {forecast_results['store_id']} - SUMMARY")
    print("=" * 80)
    print(f"Train: {forecast_results['train_start_date']} to {forecast_results['train_end_date']}")
    print(f"Test : {forecast_results['test_start_date']} to {forecast_results['test_end_date']}")
    print(f"Model: {forecast_results['order']}{forecast_results['seasonal_order']}")
    print(f"Exog : {forecast_results['selected_cols']}")
    print(f"RMSE : {forecast_results['test_metrics']['rmse']:.2f}")
    print("\nNext 7 days forecast:")
    out = pd.DataFrame({
        "Date": forecast_results["future_dates"],
        "Forecast": forecast_results["future_pred"].values,
        "CI_low": forecast_results["future_ci"].iloc[:, 0].values,
        "CI_high": forecast_results["future_ci"].iloc[:, 1].values
    })
    print(out.to_string(index=False))
    print("=" * 80)


# =============================================================================
# SUMMARY ANALYSIS FUNCTIONS (unchanged)
# =============================================================================

def analyze_sunday_patterns(results_df):
    print(f"\n{'='*60}")
    print("3A. SUNDAY PATTERN ANALYSIS")
    print('='*60)

    sunday_stats = results_df.groupby('sunday_category').agg({
        'store_id': 'count',
        'rmse': ['mean', 'std', 'min', 'max'],
        'mape': 'mean'
    }).round(2)

    sunday_stats.columns = ['Count', 'Avg_RMSE', 'Std_RMSE', 'Min_RMSE', 'Max_RMSE', 'Avg_MAPE']
    sunday_stats = sunday_stats.reset_index()
    print(sunday_stats.to_string(index=False))
    return sunday_stats


def analyze_volume_impact(results_df):
    print(f"\n{'='*60}")
    print("3B. STORE VOLUME ANALYSIS")
    print('='*60)

    results_df = results_df.copy()
    results_df['volume_quartile'] = pd.qcut(
        results_df['avg_sales'],
        q=4,
        labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'],
        duplicates='drop'
    )

    volume_stats = results_df.groupby('volume_quartile').agg({
        'store_id': 'count',
        'avg_sales': 'mean',
        'rmse': ['mean', 'std'],
        'mape': 'mean'
    }).round(2)

    volume_stats.columns = ['Count', 'Avg_Sales', 'Avg_RMSE', 'Std_RMSE', 'Avg_MAPE']
    volume_stats = volume_stats.reset_index()
    print(volume_stats.to_string(index=False))
    return volume_stats


def analyze_model_architectures(results_df):
    print(f"\n{'='*60}")
    print("3C. MODEL ARCHITECTURE ANALYSIS")
    print('='*60)

    results_df = results_df.copy()
    results_df['model_type'] = results_df.apply(
        lambda row: f"{row['model_order']}{row['seasonal_order']}", axis=1
    )

    model_stats = results_df.groupby('model_type').agg({
        'store_id': 'count',
        'rmse': 'mean'
    }).round(2)

    model_stats.columns = ['Count', 'Avg_RMSE']
    model_stats = model_stats.reset_index()
    model_stats['Pct'] = (model_stats['Count'] / len(results_df) * 100).round(1)
    model_stats = model_stats.sort_values('Count', ascending=False)

    print(model_stats.head(15).to_string(index=False))
    return model_stats


def analyze_exogenous_variables(results_df):
    print(f"\n{'='*60}")
    print("3D. EXOGENOUS VARIABLE IMPORTANCE")
    print('='*60)

    all_exog = []
    for exog_list in results_df['selected_cols'].dropna():
        if isinstance(exog_list, list):
            all_exog.extend(exog_list)

    if not all_exog:
        print("No exogenous variable data available")
        return None

    exog_counts = pd.Series(all_exog).value_counts()
    exog_stats = pd.DataFrame({
        'Variable': exog_counts.index,
        'Count': exog_counts.values,
        'Pct_Stores': (exog_counts.values / len(results_df) * 100).round(1)
    })
    print(exog_stats.head(20).to_string(index=False))
    return exog_stats


# =============================================================================
# STEP 2 RUNNER (USES YOUR REAL DF)
# =============================================================================

def run_step2_stores_2_to_25(df, window_weeks=9):
    """
    Run Step 2 for stores 2..25 using FIXED window.
    (No synthetic data, no saving)
    """
    print("\n" + "=" * 80)
    print(f"STEP 2: PROCESSING STORES 2-25 WITH FIXED {window_weeks}-WEEK WINDOW (REAL DF)")
    print("=" * 80)

    store_ids = list(range(2, 26))  # EXACTLY stores 2..25
    print(f"Stores: {store_ids}")
    print("=" * 80)

    all_results = []
    failures = []

    for i, store_id in enumerate(store_ids, 1):
        print(f"\n{'='*80}")
        print(f"STORE {i} OF {len(store_ids)} (Store {store_id})")
        print('='*80)

        try:
            result = generate_forecast_for_store(
                df=df,
                store_id=store_id,
                window_weeks=window_weeks,
                test_days=14,
                forecast_days=7,
                seasonal_on=True,
                m=7,
                p_thresh=0.05,
                auto_arima_trace=False
            )

            if result['success']:
                print_forecast_tables(result['forecast_results'])
                plot_forecasting_results(result['forecast_results'], store_id)
                all_results.append(result['store_characteristics'])

        except Exception as e:
            print(f"\n❌ Store {store_id} failed: {str(e)}")
            failures.append({'store_id': store_id, 'error': str(e)})
            traceback.print_exc()

    if all_results:
        results_df = pd.DataFrame(all_results)

        print(f"\n{'='*80}")
        print("COMPREHENSIVE ANALYSIS OF STORES 2-25")
        print('='*80)

        analyze_sunday_patterns(results_df)
        analyze_volume_impact(results_df)
        analyze_model_architectures(results_df)
        analyze_exogenous_variables(results_df)

        print(f"\n{'='*60}")
        print("SUMMARY STATISTICS (Stores 2-25)")
        print('='*60)
        print(f"Successful: {len(all_results)}/24")
        print(f"Failed: {len(failures)}/24")

        return results_df, failures

    print("\n❌ No stores processed successfully.")
    return None, failures


# =============================================================================
# MAIN EXECUTION (REAL DATA ONLY)
# =============================================================================

if __name__ == "__main__":
    """
    IMPORTANT:
    - This script expects you ALREADY have your cleaned/processed dataframe named `df`
      in memory (the same df you used for Store 1).
    - If you load it here, do it above this line and set df = ...
    """

    # -------------------------
    # 2) QUICK VALIDATION
    # -------------------------
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    required = ["Store","Date","Sales","Open","Promo","SchoolHoliday","StateHoliday","IsPromoMonth","DayOfWeek"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in df: {missing}")

    # -------------------------
    # 3) RUN STEP 2 (stores 2..25)
    # -------------------------
    results, failures = run_step2_stores_2_to_25(df, window_weeks=81)


# this is for 2-1115 stores 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  # (kept, but NOT used)
from datetime import datetime
import traceback

import pmdarima as pm
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def enforce_non_negative(predictions, open_status=None, index=None):
    """Ensure predictions are non-negative and zero when store closed."""
    predictions = np.array(predictions)
    predictions = np.maximum(predictions, 0)

    if open_status is not None:
        open_status = np.array(open_status)
        predictions[open_status == 0] = 0

    if index is not None:
        return pd.Series(predictions, index=index)
    return predictions


def calculate_mape_safe(actual, predicted):
    """Safe MAPE calculation that handles zero values."""
    actual = np.array(actual)
    predicted = np.array(predicted)

    mask = actual != 0
    if mask.sum() == 0:
        return np.nan

    predicted = enforce_non_negative(predicted)
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


def calculate_smape(actual, predicted):
    """Symmetric MAPE - handles zero values better."""
    actual = np.array(actual)
    predicted = np.array(predicted)

    predicted = enforce_non_negative(predicted)

    denominator = (np.abs(actual) + np.abs(predicted))
    mask = denominator != 0
    if mask.sum() == 0:
        return np.nan

    return 100 / len(actual) * np.sum(
        2 * np.abs(predicted[mask] - actual[mask]) / denominator[mask]
    )


def calculate_all_metrics(y_true, y_pred, X_test=None):
    """Calculate all performance metrics safely."""
    metrics = {}

    if X_test is not None and 'Open' in X_test.columns:
        y_pred_clean = enforce_non_negative(
            y_pred, X_test['Open'],
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )
    else:
        y_pred_clean = enforce_non_negative(
            y_pred,
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )

    metrics['rmse'] = float(np.sqrt(mean_squared_error(y_true, y_pred_clean)))
    metrics['mae'] = float(mean_absolute_error(y_true, y_pred_clean))
    metrics['mape'] = calculate_mape_safe(y_true, y_pred_clean)
    metrics['smape'] = calculate_smape(y_true, y_pred_clean)
    metrics['bias'] = float(np.mean(y_pred_clean - y_true))

    ss_res = np.sum((y_true - y_pred_clean) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    metrics['r2_adj'] = float(1 - (ss_res / ss_tot)) if ss_tot > 0 else np.nan

    return metrics, y_pred_clean


# =============================================================================
# DATA PREP PER STORE
# =============================================================================

def prepare_store_data(df, store_id):
    """Prepare data for a specific store. Assumes df is your REAL cleaned dataset."""
    s = df[df["Store"] == store_id].copy()
    s["Date"] = pd.to_datetime(s["Date"], errors="coerce")  # robust
    s = s.sort_values("Date").set_index("Date")

    y = s["Sales"].astype(float).sort_index()

    # Required exog columns
    X = s[["Open", "Promo", "SchoolHoliday", "StateHoliday", "IsPromoMonth", "DayOfWeek"]].copy().loc[y.index]

    # types
    X["Open"] = X["Open"].astype(int)
    X["Promo"] = X["Promo"].astype(int)
    X["SchoolHoliday"] = X["SchoolHoliday"].astype(int)
    X["IsPromoMonth"] = X["IsPromoMonth"].astype(int)

    # StateHoliday may be '0','a','b','c' or numeric; convert to 0/1
    X["StateHoliday"] = (X["StateHoliday"].astype(str) != "0").astype(int)

    # Day-of-week dummies (1..7)
    dow = pd.get_dummies(X["DayOfWeek"], prefix="DOW", drop_first=True).astype(float)

    X_num = X.drop(columns=["DayOfWeek"]).astype(float)
    Xcand = pd.concat([X_num, dow], axis=1).astype(float)

    # return store_data with Date column for future-exog logic
    return y, Xcand, s.reset_index()


# =============================================================================
# MODEL SELECTION + EXOG SELECTION
# =============================================================================

def select_orders_with_auto_arima(y_tr, X_tr, seasonal_on=True, m=7, trace=False):
    """Select SARIMAX orders using auto_arima (AIC)."""
    if seasonal_on:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=True,
            m=m,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            D=None,
            max_p=3, max_q=3,
            max_P=2, max_Q=2,
            max_d=0,
            max_D=1,
            start_p=1, start_q=1,
            start_P=0, start_Q=0,
            test='adf',
            seasonal_test='ocsb',
            n_fits=20,
            with_intercept=True
        )
        return sel.order, sel.seasonal_order, float(sel.aic()), float(sel.bic())
    else:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=False,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            max_p=3, max_q=3,
            max_d=0,
            start_p=1, start_q=1,
            test='adf',
            n_fits=15,
            with_intercept=True
        )
        return sel.order, (0, 0, 0, 0), float(sel.aic()), float(sel.bic())


def backward_elimination_exog(y_tr, X_tr, order, seasonal_order, p_thresh=0.05, max_iter=50):
    """Select significant exogenous variables using p-values (backward elimination)."""
    cols = list(X_tr.columns)

    if len(cols) == 0:
        mod = sm.tsa.SARIMAX(
            y_tr, exog=None, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        return [], res

    for _ in range(max_iter):
        X_use = X_tr[cols] if cols else None

        mod = sm.tsa.SARIMAX(
            y_tr, exog=X_use, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        if not cols:
            return [], res

        pvals = res.pvalues
        p_exog = pvals[[c for c in pvals.index if c in cols]]

        if len(p_exog) == 0:
            return cols, res

        worst_col = p_exog.idxmax()
        worst_p = float(p_exog.max())

        if worst_p <= p_thresh:
            return cols, res

        cols.remove(worst_col)

        if not cols:
            mod2 = sm.tsa.SARIMAX(
                y_tr, exog=None, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False
            )
            res2 = mod2.fit(disp=False)
            return [], res2

    return cols, res


# =============================================================================
# FUTURE EXOG LOGIC (B1 - deterministic)
# =============================================================================

def build_future_exog_B1(store_data, selected_cols, future_dates):
    """
    Deterministic future exog:
    - Open: Sundays open if historical Sunday open-rate >= 0.5, else closed on Sundays; non-Sunday open=1
    - Promo: promo on a weekday if historical promo-rate for that weekday >= 0.5
    - SchoolHoliday: use last-year same date if available, else 0
    - StateHoliday: 0 (unknown future)
    - IsPromoMonth: months in [3,6,9,12]
    - DOW_x: generated from future dates
    """
    future_exog = pd.DataFrame(index=future_dates)

    sd = store_data.copy()
    if "Date" not in sd.columns:
        raise ValueError("store_data must include a 'Date' column.")
    sd["Date"] = pd.to_datetime(sd["Date"], errors="coerce")
    sd = sd.sort_values("Date")

    sunday_data = sd[sd['DayOfWeek'] == 7]
    sunday_open_rate = float((sunday_data['Open'] == 1).mean()) if len(sunday_data) > 0 else 0.0

    promo_by_dow = sd.groupby("DayOfWeek")["Promo"].mean()  # 1..7

    for col in selected_cols:
        if col.startswith('DOW_'):
            dow_num = int(col.split('_')[1])
            future_exog[col] = [(1 if (d.dayofweek + 1) == dow_num else 0) for d in future_dates]

        elif col == 'Open':
            vals = []
            for d in future_dates:
                if d.weekday() == 6:  # Sunday
                    vals.append(1 if sunday_open_rate >= 0.5 else 0)
                else:
                    vals.append(1)
            future_exog[col] = vals

        elif col == 'Promo':
            vals = []
            for d in future_dates:
                dow = d.dayofweek + 1
                hist_rate = float(promo_by_dow.get(dow, 0))
                vals.append(1 if hist_rate >= 0.5 else 0)
            future_exog[col] = vals

        elif col == 'SchoolHoliday':
            sd_idx = sd.set_index("Date")
            last_year_dates = [d - pd.DateOffset(years=1) for d in future_dates]
            vals = []
            for d0 in last_year_dates:
                d0n = pd.Timestamp(d0).normalize()
                if d0n in sd_idx.index:
                    vals.append(int(sd_idx.loc[d0n, 'SchoolHoliday']))
                else:
                    vals.append(0)
            future_exog[col] = vals

        elif col == 'IsPromoMonth':
            future_exog[col] = [1 if d.month in [3, 6, 9, 12] else 0 for d in future_dates]

        elif col == 'StateHoliday':
            future_exog[col] = [0] * len(future_dates)

        else:
            future_exog[col] = [0.0] * len(future_dates)

    return future_exog, sunday_open_rate


def force_ci_zero_on_closed(pred_mean, ci_df, open_series):
    """Force predictions and CI to 0 when Open==0."""
    open_arr = np.array(open_series).astype(int)
    closed = open_arr == 0
    if closed.any():
        idx = pred_mean.index
        pred_mean.loc[idx[closed]] = 0.0
        ci_df.iloc[closed, 0] = 0.0
        ci_df.iloc[closed, 1] = 0.0
    return pred_mean, ci_df


def sunday_category(rate):
    if rate == 0:
        return "Always Closed (0%)"
    if rate == 1:
        return "Always Open (100%)"
    return "Sometimes Open (1-99%)"


# =============================================================================
# FORECAST GENERATION FOR A SINGLE STORE (FIXED WINDOW)
# =============================================================================

def generate_forecast_for_store(
    df: pd.DataFrame,
    store_id: int,
    window_weeks: int = 9,
    test_days: int = 14,
    forecast_days: int = 7,
    seasonal_on: bool = True,
    m: int = 7,
    p_thresh: float = 0.05,
    auto_arima_trace: bool = False
):
    y_full, X_full, store_data_raw = prepare_store_data(df, store_id)

    end_date = y_full.index[-1]
    test_start = end_date - pd.Timedelta(days=test_days - 1)
    test_end = end_date

    train_end = test_start - pd.Timedelta(days=1)
    train_start = train_end - pd.Timedelta(weeks=window_weeks) + pd.Timedelta(days=1)

    if train_start < y_full.index[0]:
        raise ValueError(f"Insufficient data for store {store_id}: need {window_weeks} weeks before test start.")

    y_train = y_full.loc[train_start:train_end]
    y_test = y_full.loc[test_start:test_end]
    X_train = X_full.loc[y_train.index]
    X_test = X_full.loc[y_test.index]

    order, seas, aic, bic = select_orders_with_auto_arima(
        y_train, X_train, seasonal_on=seasonal_on, m=m, trace=auto_arima_trace
    )

    selected_cols, _ = backward_elimination_exog(
        y_train, X_train, order, seas, p_thresh=p_thresh
    )

    X_tr = X_train[selected_cols] if selected_cols else None
    X_te = X_test[selected_cols] if selected_cols else None

    mod = sm.tsa.SARIMAX(
        y_train, exog=X_tr, order=order, seasonal_order=seas,
        enforce_stationarity=False, enforce_invertibility=False
    )
    res = mod.fit(disp=False)

    start_i = len(y_train)
    end_i = len(y_train) + len(y_test) - 1

    pred_test_raw = res.predict(start=start_i, end=end_i, exog=X_te)
    pred_test_raw.index = y_test.index

    pred_test = enforce_non_negative(
        pred_test_raw,
        X_test['Open'] if 'Open' in X_test.columns else None,
        index=pred_test_raw.index
    )

    test_metrics, _ = calculate_all_metrics(y_test, pred_test, X_test)

    last_7_start = test_end - pd.Timedelta(days=6)
    y_test_last7 = y_test.loc[last_7_start:test_end]
    pred_last7_raw = pred_test_raw.loc[last_7_start:test_end]
    pred_last7 = pred_test.loc[last_7_start:test_end]

    last7_metrics, _ = calculate_all_metrics(
        y_test_last7, pred_last7,
        X_test.loc[last_7_start:test_end] if X_test is not None else None
    )

    future_dates = pd.date_range(
        start=test_end + pd.Timedelta(days=1),
        periods=forecast_days,
        freq='D'
    )

    if selected_cols:
        future_exog, _ = build_future_exog_B1(store_data_raw, selected_cols, future_dates)
        future_exog_sel = future_exog[selected_cols]
    else:
        future_exog = pd.DataFrame(index=future_dates)
        future_exog_sel = None

    future_forecast = res.get_forecast(steps=forecast_days, exog=future_exog_sel)
    pred_future_raw = future_forecast.predicted_mean.copy()
    ci_future = future_forecast.conf_int().copy()

    if "Open" in future_exog.columns:
        pred_future_raw, ci_future = force_ci_zero_on_closed(pred_future_raw, ci_future, future_exog["Open"])

    pred_future = enforce_non_negative(
        pred_future_raw,
        future_exog['Open'] if 'Open' in future_exog.columns else None,
        index=pred_future_raw.index
    )

    sunday_rate_full = float(
        (store_data_raw[store_data_raw['DayOfWeek'] == 7]['Open'] == 1).mean()
    ) if len(store_data_raw[store_data_raw['DayOfWeek'] == 7]) > 0 else 0.0

    avg_sales = float(y_train.mean())
    promo_rate = float((store_data_raw['Promo'] == 1).mean())

    return {
        'store_id': store_id,
        'success': True,
        'forecast_results': {
            'store_id': store_id,
            'optimal_window_weeks': window_weeks,
            'order': order,
            'seasonal_order': seas,
            'selected_cols': selected_cols,
            'aic': aic,
            'bic': bic,

            'train_start_date': train_start.strftime('%Y-%m-%d'),
            'train_end_date': train_end.strftime('%Y-%m-%d'),
            'test_start_date': test_start.strftime('%Y-%m-%d'),
            'test_end_date': test_end.strftime('%Y-%m-%d'),

            'last_7_start': last_7_start.strftime('%Y-%m-%d'),
            'last_7_end': test_end.strftime('%Y-%m-%d'),

            'future_start_date': future_dates[0].strftime('%Y-%m-%d'),
            'future_end_date': future_dates[-1].strftime('%Y-%m-%d'),

            'y_train': y_train,
            'y_test': y_test,

            'test_pred_raw': pred_test_raw,
            'test_pred': pred_test,
            'test_metrics': test_metrics,

            'last_7_dates': y_test_last7.index,
            'last_7_actual': y_test_last7,
            'last_7_pred_raw': pred_last7_raw,
            'last_7_pred': pred_last7,
            'last_7_metrics': last7_metrics,

            'future_dates': future_dates,
            'future_pred_raw': pred_future_raw,
            'future_pred': pred_future,
            'future_ci': ci_future,
            'future_exog': future_exog,

            'X_test': X_test
        },
        'store_characteristics': {
            'store_id': store_id,
            'sunday_open_rate': sunday_rate_full,
            'sunday_category': sunday_category(sunday_rate_full),
            'avg_sales': avg_sales,
            'promo_rate': promo_rate,
            'model_order': str(order),
            'seasonal_order': str(seas),
            'selected_cols': selected_cols,
            'rmse': test_metrics['rmse'],
            'mae': test_metrics['mae'],
            'mape': test_metrics['mape'],
            'smape': test_metrics['smape'],
            'bias': test_metrics['bias'],
            'r2': test_metrics['r2_adj'],
            'aic': aic,
            'bic': bic
        }
    }


# =============================================================================
# PRINT FORECAST TABLES (TO FILE)
# =============================================================================

def print_forecast_tables(forecast_results, output_file=None):
    """Write compact last-7 + future forecast table to file."""
    output = []

    output.append("\n" + "=" * 80)
    output.append(f"STORE {forecast_results['store_id']} - FORECAST RESULTS")
    output.append("=" * 80)
    output.append(f"Model: {forecast_results['order']}{forecast_results['seasonal_order']}")
    output.append(f"Exog : {forecast_results['selected_cols']}")
    output.append(f"RMSE : {forecast_results['test_metrics']['rmse']:.2f}")

    output.append("\nLast 7 Test Days + Next 7 Forecast Days:")
    output.append("-" * 80)
    output.append(f"{'Date':<12} {'Day':<6} {'Actual':>10} {'Predicted':>12}")
    output.append("-" * 80)

    day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

    # Last 7 test days
    last_7_dates = forecast_results['last_7_dates']
    last_7_actual = forecast_results['last_7_actual']
    last_7_pred = forecast_results['last_7_pred']

    for i, date in enumerate(last_7_dates):
        day_name = day_names[date.dayofweek]
        actual_val = float(last_7_actual.iloc[i]) if hasattr(last_7_actual, 'iloc') else float(last_7_actual[i])
        pred_val = float(last_7_pred.iloc[i]) if hasattr(last_7_pred, 'iloc') else float(last_7_pred[i])
        output.append(f"{date.strftime('%Y-%m-%d'):<12} {day_name:<6} {actual_val:>10.0f} {pred_val:>12.2f}")

    # Future days
    future_dates = forecast_results['future_dates']
    future_pred = forecast_results['future_pred']
    for i, date in enumerate(future_dates):
        day_name = day_names[date.dayofweek]
        pred_val = float(future_pred.iloc[i]) if hasattr(future_pred, 'iloc') else float(future_pred[i])
        output.append(f"{date.strftime('%Y-%m-%d'):<12} {day_name:<6} {'-':>10} {pred_val:>12.2f}")

    output.append("-" * 80)
    output.append("")

    if output_file is not None:
        for line in output:
            output_file.write(line + "\n")
    else:
        for line in output:
            print(line)


# =============================================================================
# STEP 2 RUNNER (2..1115) - SAVES TO TEXT FILE + CSV
# =============================================================================

def run_step2_stores_2_to_1115(
    df,
    window_weeks=81,
    output_filename='forecast_results_stores_2_1115.txt',
    progress_every=50
):
    """
    Run Step 2 for stores 2..1115 and save results to text file + CSV.

    Output behavior:
    - No plots
    - No per-store printing to notebook
    - Writes per-store compact tables to a text file
    - Writes store_characteristics.csv and failed_stores.csv
    - Writes a progress line to the text file every `progress_every` stores
    """
    store_ids = list(range(2, 1116))  # 2..1115 inclusive

    print("\n" + "=" * 80)
    print(f"STEP 2: PROCESSING STORES 2-1115 WITH FIXED {window_weeks}-WEEK WINDOW")
    print(f"Writing detailed output to: {output_filename}")
    print("=" * 80)

    all_results = []
    failures = []

    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("FORECAST RESULTS FOR STORES 2-1115\n")
        f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Window weeks: {window_weeks}\n")
        f.write("=" * 80 + "\n\n")

        for i, store_id in enumerate(store_ids, 1):
            try:
                result = generate_forecast_for_store(
                    df=df,
                    store_id=store_id,
                    window_weeks=window_weeks,
                    test_days=14,
                    forecast_days=7,
                    seasonal_on=True,
                    m=7,
                    p_thresh=0.05,
                    auto_arima_trace=False
                )

                if result['success']:
                    print_forecast_tables(result['forecast_results'], output_file=f)
                    all_results.append(result['store_characteristics'])
                else:
                    failures.append({'store_id': store_id, 'error': 'Unknown failure (success=False)'})
                    f.write(f"\n❌ Store {store_id} returned success=False\n")

            except Exception as e:
                failures.append({'store_id': store_id, 'error': str(e)})
                f.write(f"\n❌ Store {store_id} failed: {str(e)}\n")
                f.write(traceback.format_exc() + "\n")

            # progress marker (file) + flush
            if progress_every and (i % progress_every == 0):
                f.write(f"\n--- Progress: {i}/{len(store_ids)} stores processed ---\n")
                f.flush()

    # Save CSVs
    if all_results:
        results_df = pd.DataFrame(all_results)
        results_df.to_csv('store_characteristics.csv', index=False)

        if failures:
            pd.DataFrame(failures).to_csv('failed_stores.csv', index=False)

        print("\n" + "=" * 60)
        print("PROCESSING COMPLETE")
        print("=" * 60)
        print(f"Successful: {len(all_results)}/{len(store_ids)}")
        print(f"Failed: {len(failures)}/{len(store_ids)}")
        print("\nFiles written:")
        print(f"  - {output_filename}")
        print("  - store_characteristics.csv")
        if failures:
            print("  - failed_stores.csv")

        return results_df, failures

    print("\n❌ No stores processed successfully.")
    return None, failures


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    """
    IMPORTANT:
    - This script expects you ALREADY have your cleaned/processed dataframe named `df`
    - This will run stores 2..1115 and write output to files (no notebook spam)
    """

    # Quick validation
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    required = ["Store","Date","Sales","Open","Promo","SchoolHoliday","StateHoliday","IsPromoMonth","DayOfWeek"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in df: {missing}")

    results, failures = run_step2_stores_2_to_1115(
        df,
        window_weeks=81,
        output_filename='forecast_results_stores_2_1115.txt',
        progress_every=50
    )


In [ ]:
## analysis of all stores

In [ ]:
import pandas as pd

# =========================
# ANALYSIS HELPERS (same logic as before, but OK to run now)
# =========================

def analyze_sunday_patterns(results_df):
    print(f"\n{'='*60}")
    print("SUNDAY PATTERN ANALYSIS")
    print('='*60)

    sunday_stats = results_df.groupby('sunday_category').agg({
        'store_id': 'count',
        'rmse': ['mean', 'std', 'min', 'max'],
        'mape': 'mean'
    }).round(2)

    sunday_stats.columns = ['Count', 'Avg_RMSE', 'Std_RMSE', 'Min_RMSE', 'Max_RMSE', 'Avg_MAPE']
    sunday_stats = sunday_stats.reset_index()
    print(sunday_stats.to_string(index=False))
    return sunday_stats


def analyze_volume_impact(results_df):
    print(f"\n{'='*60}")
    print("STORE VOLUME ANALYSIS")
    print('='*60)

    results_df = results_df.copy()
    results_df['volume_quartile'] = pd.qcut(
        results_df['avg_sales'],
        q=4,
        labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'],
        duplicates='drop'
    )

    volume_stats = results_df.groupby('volume_quartile').agg({
        'store_id': 'count',
        'avg_sales': 'mean',
        'rmse': ['mean', 'std'],
        'mape': 'mean'
    }).round(2)

    volume_stats.columns = ['Count', 'Avg_Sales', 'Avg_RMSE', 'Std_RMSE', 'Avg_MAPE']
    volume_stats = volume_stats.reset_index()
    print(volume_stats.to_string(index=False))
    return volume_stats


def analyze_model_architectures(results_df):
    print(f"\n{'='*60}")
    print("MODEL ARCHITECTURE ANALYSIS")
    print('='*60)

    results_df = results_df.copy()
    results_df['model_type'] = results_df.apply(
        lambda row: f"{row['model_order']}{row['seasonal_order']}", axis=1
    )

    model_stats = results_df.groupby('model_type').agg({
        'store_id': 'count',
        'rmse': 'mean'
    }).round(2)

    model_stats.columns = ['Count', 'Avg_RMSE']
    model_stats = model_stats.reset_index()
    model_stats['Pct'] = (model_stats['Count'] / len(results_df) * 100).round(1)
    model_stats = model_stats.sort_values('Count', ascending=False)

    print(model_stats.head(30).to_string(index=False))
    return model_stats


def analyze_exogenous_variables(results_df):
    print(f"\n{'='*60}")
    print("EXOGENOUS VARIABLE IMPORTANCE")
    print('='*60)

    # selected_cols may be stored as list OR as string like "['Open','Promo',...]"
    all_exog = []
    for v in results_df['selected_cols'].dropna():
        if isinstance(v, list):
            all_exog.extend(v)
        elif isinstance(v, str):
            s = v.strip()
            if s.startswith('[') and s.endswith(']'):
                try:
                    parsed = eval(s, {"__builtins__": {}})
                    if isinstance(parsed, list):
                        all_exog.extend(parsed)
                except Exception:
                    pass

    if not all_exog:
        print("No exogenous variable data available")
        return None

    exog_counts = pd.Series(all_exog).value_counts()
    exog_stats = pd.DataFrame({
        'Variable': exog_counts.index,
        'Count': exog_counts.values,
        'Pct_Stores': (exog_counts.values / len(results_df) * 100).round(1)
    })
    print(exog_stats.head(30).to_string(index=False))
    return exog_stats


# =========================
# RUN ANALYSIS FROM YOUR CSV
# =========================

CSV_PATH = "C:/Users/shrut/Downloads/store_characteristics.csv"

results_df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(results_df)} stores from {CSV_PATH}")

_ = analyze_sunday_patterns(results_df)
_ = analyze_volume_impact(results_df)
_ = analyze_model_architectures(results_df)
_ = analyze_exogenous_variables(results_df)


# 1-1115 store with plots

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
import traceback

import pmdarima as pm
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def enforce_non_negative(predictions, open_status=None, index=None):
    """Ensure predictions are non-negative and zero when store closed."""
    predictions = np.array(predictions)
    predictions = np.maximum(predictions, 0)

    if open_status is not None:
        open_status = np.array(open_status)
        predictions[open_status == 0] = 0

    if index is not None:
        return pd.Series(predictions, index=index)
    return predictions


def calculate_mape_safe(actual, predicted):
    """Safe MAPE calculation that handles zero values."""
    actual = np.array(actual)
    predicted = np.array(predicted)

    mask = actual != 0
    if mask.sum() == 0:
        return np.nan

    predicted = enforce_non_negative(predicted)
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


def calculate_smape(actual, predicted):
    """Symmetric MAPE - handles zero values better."""
    actual = np.array(actual)
    predicted = np.array(predicted)

    predicted = enforce_non_negative(predicted)

    denominator = (np.abs(actual) + np.abs(predicted))
    mask = denominator != 0
    if mask.sum() == 0:
        return np.nan

    return 100 / len(actual) * np.sum(
        2 * np.abs(predicted[mask] - actual[mask]) / denominator[mask]
    )


def calculate_all_metrics(y_true, y_pred, X_test=None):
    """Calculate all performance metrics safely."""
    metrics = {}

    if X_test is not None and 'Open' in X_test.columns:
        y_pred_clean = enforce_non_negative(
            y_pred, X_test['Open'],
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )
    else:
        y_pred_clean = enforce_non_negative(
            y_pred,
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )

    metrics['rmse'] = float(np.sqrt(mean_squared_error(y_true, y_pred_clean)))
    metrics['mae'] = float(mean_absolute_error(y_true, y_pred_clean))
    metrics['mape'] = calculate_mape_safe(y_true, y_pred_clean)
    metrics['smape'] = calculate_smape(y_true, y_pred_clean)
    metrics['bias'] = float(np.mean(y_pred_clean - y_true))

    ss_res = np.sum((y_true - y_pred_clean) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    metrics['r2_adj'] = float(1 - (ss_res / ss_tot)) if ss_tot > 0 else np.nan

    return metrics, y_pred_clean


# =============================================================================
# DATA PREP PER STORE
# =============================================================================

def prepare_store_data(df, store_id):
    """Prepare data for a specific store. Assumes df is your REAL cleaned dataset."""
    s = df[df["Store"] == store_id].copy()
    s["Date"] = pd.to_datetime(s["Date"], errors="coerce")  # robust
    s = s.sort_values("Date").set_index("Date")

    y = s["Sales"].astype(float).sort_index()

    # Required exog columns
    X = s[["Open", "Promo", "SchoolHoliday", "StateHoliday", "IsPromoMonth", "DayOfWeek"]].copy().loc[y.index]

    # types
    X["Open"] = X["Open"].astype(int)
    X["Promo"] = X["Promo"].astype(int)
    X["SchoolHoliday"] = X["SchoolHoliday"].astype(int)
    X["IsPromoMonth"] = X["IsPromoMonth"].astype(int)

    # StateHoliday may be '0','a','b','c' or numeric; convert to 0/1
    X["StateHoliday"] = (X["StateHoliday"].astype(str) != "0").astype(int)

    # Day-of-week dummies (1..7)
    dow = pd.get_dummies(X["DayOfWeek"], prefix="DOW", drop_first=True).astype(float)

    X_num = X.drop(columns=["DayOfWeek"]).astype(float)
    Xcand = pd.concat([X_num, dow], axis=1).astype(float)

    # return store_data with Date column for future-exog logic
    return y, Xcand, s.reset_index()


# =============================================================================
# MODEL SELECTION + EXOG SELECTION
# =============================================================================

def select_orders_with_auto_arima(y_tr, X_tr, seasonal_on=True, m=7, trace=False):
    """Select SARIMAX orders using auto_arima (AIC)."""
    if seasonal_on:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=True,
            m=m,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            D=None,
            max_p=3, max_q=3,
            max_P=2, max_Q=2,
            max_d=0,
            max_D=1,
            start_p=1, start_q=1,
            start_P=0, start_Q=0,
            test='adf',
            seasonal_test='ocsb',
            n_fits=20,
            with_intercept=True
        )
        return sel.order, sel.seasonal_order, float(sel.aic()), float(sel.bic())
    else:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=False,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            max_p=3, max_q=3,
            max_d=0,
            start_p=1, start_q=1,
            test='adf',
            n_fits=15,
            with_intercept=True
        )
        return sel.order, (0, 0, 0, 0), float(sel.aic()), float(sel.bic())


def backward_elimination_exog(y_tr, X_tr, order, seasonal_order, p_thresh=0.05, max_iter=50):
    """Select significant exogenous variables using p-values (backward elimination)."""
    cols = list(X_tr.columns)

    if len(cols) == 0:
        mod = sm.tsa.SARIMAX(
            y_tr, exog=None, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        return [], res

    for _ in range(max_iter):
        X_use = X_tr[cols] if cols else None

        mod = sm.tsa.SARIMAX(
            y_tr, exog=X_use, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        if not cols:
            return [], res

        pvals = res.pvalues
        p_exog = pvals[[c for c in pvals.index if c in cols]]

        if len(p_exog) == 0:
            return cols, res

        worst_col = p_exog.idxmax()
        worst_p = float(p_exog.max())

        if worst_p <= p_thresh:
            return cols, res

        cols.remove(worst_col)

        if not cols:
            mod2 = sm.tsa.SARIMAX(
                y_tr, exog=None, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False
            )
            res2 = mod2.fit(disp=False)
            return [], res2

    return cols, res


# =============================================================================
# FUTURE EXOG LOGIC (B1 - deterministic)
# =============================================================================

def build_future_exog_B1(store_data, selected_cols, future_dates):
    """
    Deterministic future exog:
    - Open: Sundays open if historical Sunday open-rate >= 0.5, else closed on Sundays; non-Sunday open=1
    - Promo: promo on a weekday if historical promo-rate for that weekday >= 0.5
    - SchoolHoliday: use last-year same date if available, else 0
    - StateHoliday: 0 
    - IsPromoMonth: months in [3,6,9,12]
    - DOW_x: generated from future dates
    """
    future_exog = pd.DataFrame(index=future_dates)

    sd = store_data.copy()
    if "Date" not in sd.columns:
        raise ValueError("store_data must include a 'Date' column.")
    sd["Date"] = pd.to_datetime(sd["Date"], errors="coerce")
    sd = sd.sort_values("Date")

    sunday_data = sd[sd['DayOfWeek'] == 7]
    sunday_open_rate = float((sunday_data['Open'] == 1).mean()) if len(sunday_data) > 0 else 0.0

    promo_by_dow = sd.groupby("DayOfWeek")["Promo"].mean()  # 1..7

    for col in selected_cols:
        if col.startswith('DOW_'):
            dow_num = int(col.split('_')[1])
            future_exog[col] = [(1 if (d.dayofweek + 1) == dow_num else 0) for d in future_dates]

        elif col == 'Open':
            vals = []
            for d in future_dates:
                if d.weekday() == 6:  # Sunday
                    vals.append(1 if sunday_open_rate >= 0.5 else 0)
                else:
                    vals.append(1)
            future_exog[col] = vals

        elif col == 'Promo':
            vals = []
            for d in future_dates:
                dow = d.dayofweek + 1
                hist_rate = float(promo_by_dow.get(dow, 0))
                vals.append(1 if hist_rate >= 0.5 else 0)
            future_exog[col] = vals

        elif col == 'SchoolHoliday':
            sd_idx = sd.set_index("Date")
            last_year_dates = [d - pd.DateOffset(years=1) for d in future_dates]
            vals = []
            for d0 in last_year_dates:
                d0n = pd.Timestamp(d0).normalize()
                if d0n in sd_idx.index:
                    vals.append(int(sd_idx.loc[d0n, 'SchoolHoliday']))
                else:
                    vals.append(0)
            future_exog[col] = vals

        elif col == 'IsPromoMonth':
            future_exog[col] = [1 if d.month in [3, 6, 9, 12] else 0 for d in future_dates]

        elif col == 'StateHoliday':
            future_exog[col] = [0] * len(future_dates)

        else:
            future_exog[col] = [0.0] * len(future_dates)

    return future_exog, sunday_open_rate


def force_ci_zero_on_closed(pred_mean, ci_df, open_series):
    """Force predictions and CI to 0 when Open==0."""
    open_arr = np.array(open_series).astype(int)
    closed = open_arr == 0
    if closed.any():
        idx = pred_mean.index
        pred_mean.loc[idx[closed]] = 0.0
        ci_df.iloc[closed, 0] = 0.0
        ci_df.iloc[closed, 1] = 0.0
    return pred_mean, ci_df


def sunday_category(rate):
    if rate == 0:
        return "Always Closed (0%)"
    if rate == 1:
        return "Always Open (100%)"
    return "Sometimes Open (1-99%)"


# =============================================================================
# FORECAST GENERATION FOR A SINGLE STORE (FIXED WINDOW)
# =============================================================================

def generate_forecast_for_store(
    df: pd.DataFrame,
    store_id: int,
    window_weeks: int = 9,
    test_days: int = 14,
    forecast_days: int = 7,
    seasonal_on: bool = True,
    m: int = 7,
    p_thresh: float = 0.05,
    auto_arima_trace: bool = False
):
    y_full, X_full, store_data_raw = prepare_store_data(df, store_id)

    end_date = y_full.index[-1]
    test_start = end_date - pd.Timedelta(days=test_days - 1)
    test_end = end_date

    train_end = test_start - pd.Timedelta(days=1)
    train_start = train_end - pd.Timedelta(weeks=window_weeks) + pd.Timedelta(days=1)

    if train_start < y_full.index[0]:
        raise ValueError(f"Insufficient data for store {store_id}: need {window_weeks} weeks before test start.")

    y_train = y_full.loc[train_start:train_end]
    y_test = y_full.loc[test_start:test_end]
    X_train = X_full.loc[y_train.index]
    X_test = X_full.loc[y_test.index]

    order, seas, aic, bic = select_orders_with_auto_arima(
        y_train, X_train, seasonal_on=seasonal_on, m=m, trace=auto_arima_trace
    )

    selected_cols, _ = backward_elimination_exog(
        y_train, X_train, order, seas, p_thresh=p_thresh
    )

    X_tr = X_train[selected_cols] if selected_cols else None
    X_te = X_test[selected_cols] if selected_cols else None

    mod = sm.tsa.SARIMAX(
        y_train, exog=X_tr, order=order, seasonal_order=seas,
        enforce_stationarity=False, enforce_invertibility=False
    )
    res = mod.fit(disp=False)

    start_i = len(y_train)
    end_i = len(y_train) + len(y_test) - 1

    pred_test_raw = res.predict(start=start_i, end=end_i, exog=X_te)
    pred_test_raw.index = y_test.index

    pred_test = enforce_non_negative(
        pred_test_raw,
        X_test['Open'] if 'Open' in X_test.columns else None,
        index=pred_test_raw.index
    )

    test_metrics, _ = calculate_all_metrics(y_test, pred_test, X_test)

    last_7_start = test_end - pd.Timedelta(days=6)
    y_test_last7 = y_test.loc[last_7_start:test_end]
    pred_last7_raw = pred_test_raw.loc[last_7_start:test_end]
    pred_last7 = pred_test.loc[last_7_start:test_end]

    _ = calculate_all_metrics(
        y_test_last7, pred_last7,
        X_test.loc[last_7_start:test_end] if X_test is not None else None
    )

    future_dates = pd.date_range(
        start=test_end + pd.Timedelta(days=1),
        periods=forecast_days,
        freq='D'
    )

    if selected_cols:
        future_exog, _sunday_rate = build_future_exog_B1(store_data_raw, selected_cols, future_dates)
        future_exog_sel = future_exog[selected_cols]
    else:
        future_exog = pd.DataFrame(index=future_dates)
        future_exog_sel = None

    future_forecast = res.get_forecast(steps=forecast_days, exog=future_exog_sel)
    pred_future_raw = future_forecast.predicted_mean.copy()
    ci_future = future_forecast.conf_int().copy()

    if "Open" in future_exog.columns:
        pred_future_raw, ci_future = force_ci_zero_on_closed(pred_future_raw, ci_future, future_exog["Open"])

    pred_future = enforce_non_negative(
        pred_future_raw,
        future_exog['Open'] if 'Open' in future_exog.columns else None,
        index=pred_future_raw.index
    )

    sunday_rate_full = float(
        (store_data_raw[store_data_raw['DayOfWeek'] == 7]['Open'] == 1).mean()
    ) if len(store_data_raw[store_data_raw['DayOfWeek'] == 7]) > 0 else 0.0

    avg_sales = float(y_train.mean())
    promo_rate = float((store_data_raw['Promo'] == 1).mean())

    return {
        'store_id': store_id,
        'success': True,
        'forecast_results': {
            'store_id': store_id,
            'optimal_window_weeks': window_weeks,
            'order': order,
            'seasonal_order': seas,
            'selected_cols': selected_cols,
            'aic': aic,
            'bic': bic,

            'train_start_date': train_start.strftime('%Y-%m-%d'),
            'train_end_date': train_end.strftime('%Y-%m-%d'),
            'test_start_date': test_start.strftime('%Y-%m-%d'),
            'test_end_date': test_end.strftime('%Y-%m-%d'),

            'last_7_start': last_7_start.strftime('%Y-%m-%d'),
            'last_7_end': test_end.strftime('%Y-%m-%d'),

            'future_start_date': future_dates[0].strftime('%Y-%m-%d'),
            'future_end_date': future_dates[-1].strftime('%Y-%m-%d'),

            'y_train': y_train,
            'y_test': y_test,

            'test_pred_raw': pred_test_raw,
            'test_pred': pred_test,
            'test_metrics': test_metrics,

            'last_7_dates': y_test_last7.index,
            'last_7_actual': y_test_last7,
            'last_7_pred_raw': pred_last7_raw,
            'last_7_pred': pred_last7,

            'future_dates': future_dates,
            'future_pred_raw': pred_future_raw,
            'future_pred': pred_future,
            'future_ci': ci_future,
            'future_exog': future_exog,

            'X_test': X_test
        },
        'store_characteristics': {
            'store_id': store_id,
            'sunday_open_rate': sunday_rate_full,
            'sunday_category': sunday_category(sunday_rate_full),
            'avg_sales': avg_sales,
            'promo_rate': promo_rate,
            'model_order': str(order),
            'seasonal_order': str(seas),
            'selected_cols': selected_cols,
            'rmse': test_metrics['rmse'],
            'mae': test_metrics['mae'],
            'mape': test_metrics['mape'],
            'smape': test_metrics['smape'],
            'bias': test_metrics['bias'],
            'r2': test_metrics['r2_adj'],
            'aic': aic,
            'bic': bic
        }
    }


# =============================================================================
# PRINT FORECAST TABLES (TO FILE)
# =============================================================================

def print_forecast_tables(forecast_results, output_file=None):
    """Write compact last-7 + future forecast table to file."""
    output = []

    output.append("\n" + "=" * 80)
    output.append(f"STORE {forecast_results['store_id']} - FORECAST RESULTS")
    output.append("=" * 80)
    output.append(f"Model: {forecast_results['order']}{forecast_results['seasonal_order']}")
    output.append(f"Exog : {forecast_results['selected_cols']}")
    output.append(f"RMSE : {forecast_results['test_metrics']['rmse']:.2f}")

    output.append("\nLast 7 Test Days + Next 7 Forecast Days:")
    output.append("-" * 80)
    output.append(f"{'Date':<12} {'Day':<6} {'Actual':>10} {'Predicted':>12}")
    output.append("-" * 80)

    day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

    last_7_dates = forecast_results['last_7_dates']
    last_7_actual = forecast_results['last_7_actual']
    last_7_pred = forecast_results['last_7_pred']

    for i, date in enumerate(last_7_dates):
        day_name = day_names[date.dayofweek]
        actual_val = float(last_7_actual.iloc[i]) if hasattr(last_7_actual, 'iloc') else float(last_7_actual[i])
        pred_val = float(last_7_pred.iloc[i]) if hasattr(last_7_pred, 'iloc') else float(last_7_pred[i])
        output.append(f"{date.strftime('%Y-%m-%d'):<12} {day_name:<6} {actual_val:>10.0f} {pred_val:>12.2f}")

    future_dates = forecast_results['future_dates']
    future_pred = forecast_results['future_pred']
    for i, date in enumerate(future_dates):
        day_name = day_names[date.dayofweek]
        pred_val = float(future_pred.iloc[i]) if hasattr(future_pred, 'iloc') else float(future_pred[i])
        output.append(f"{date.strftime('%Y-%m-%d'):<12} {day_name:<6} {'-':>10} {pred_val:>12.2f}")

    output.append("-" * 80)
    output.append("")

    if output_file is not None:
        for line in output:
            output_file.write(line + "\n")
    else:
        for line in output:
            print(line)


# =============================================================================
# PDF GRID PLOTTING (6 or 8 plots per page)
# =============================================================================

def plot_combined_on_axis(ax, forecast_results, store_id):
    """Draw the SINGLE combined plot into an existing axis (no figure creation)."""
    ax.plot(
        forecast_results['y_test'].index,
        forecast_results['y_test'].values,
        label="Actual",
        marker='o',
        linewidth=1
    )

    test_pred_series = pd.Series(
        np.asarray(forecast_results['test_pred']),
        index=forecast_results['y_test'].index
    )
    future_pred_series = pd.Series(
        np.asarray(forecast_results['future_pred']),
        index=forecast_results['future_dates']
    )
    pred_all = pd.concat([test_pred_series, future_pred_series])

    ax.plot(
        pred_all.index,
        pred_all.values,
        label="Pred",
        linestyle="--",
        marker='s',
        linewidth=1
    )

    ci = forecast_results['future_ci']
    ax.fill_between(
        forecast_results['future_dates'],
        ci.iloc[:, 0].values,
        ci.iloc[:, 1].values,
        alpha=0.2,
        label="95% CI"
    )

    ax.set_title(f"Store {store_id}", fontsize=10)
    ax.grid(True, alpha=0.2)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)


def pdf_page_layout(plots_per_page):
    """Supported: 6 -> 3x2, 8 -> 4x2."""
    if plots_per_page == 6:
        return 3, 2
    if plots_per_page == 8:
        return 4, 2
    raise ValueError("plots_per_page must be 6 or 8")


def new_pdf_page_figure(plots_per_page):
    """Create one PDF page figure with a grid of axes."""
    nrows, ncols = pdf_page_layout(plots_per_page)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 22))
    axes = np.array(axes).reshape(-1)
    return fig, axes


# =============================================================================
# STEP 2 RUNNER (1..1115) - SAVES TEXT + CSV + PDF GRID PLOTS
# =============================================================================

def run_step2_stores_1_to_1115(
    df,
    window_weeks=81,
    output_filename='forecast_results_stores_1_1115.txt',
    pdf_filename='Appendix_Forecast_Plots_Stores_1_1115.pdf',
    progress_every=50,
    plots_per_page=8
):
    store_ids = list(range(1, 1116))

    print("\n" + "=" * 80)
    print(f"STEP 2: PROCESSING STORES 1-1115 WITH FIXED {window_weeks}-WEEK WINDOW")
    print(f"Writing text output to: {output_filename}")
    print(f"Writing PDF appendix to: {pdf_filename} ({plots_per_page} plots/page)")
    print("=" * 80)

    all_results = []
    failures = []

    pdf = PdfPages(pdf_filename)

    fig_page, axes_page = new_pdf_page_figure(plots_per_page)
    slot = 0

    try:
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write("=" * 80 + "\n")
            f.write("FORECAST RESULTS FOR STORES 1-1115\n")
            f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Window weeks: {window_weeks}\n")
            f.write("=" * 80 + "\n\n")

            for i, store_id in enumerate(store_ids, 1):
                try:
                    result = generate_forecast_for_store(
                        df=df,
                        store_id=store_id,
                        window_weeks=window_weeks,
                        test_days=14,
                        forecast_days=7,
                        seasonal_on=True,
                        m=7,
                        p_thresh=0.05,
                        auto_arima_trace=False
                    )

                    if result['success']:
                        print_forecast_tables(result['forecast_results'], output_file=f)
                        all_results.append(result['store_characteristics'])

                        ax = axes_page[slot]
                        plot_combined_on_axis(ax, result['forecast_results'], store_id)

                        if slot == 0:
                            ax.legend(fontsize=8)
                        else:
                            leg = ax.get_legend()
                            if leg is not None:
                                leg.remove()

                        slot += 1

                        if slot == plots_per_page:
                            fig_page.tight_layout()
                            pdf.savefig(fig_page)
                            plt.close(fig_page)
                            fig_page, axes_page = new_pdf_page_figure(plots_per_page)
                            slot = 0

                    else:
                        failures.append({'store_id': store_id, 'error': 'Unknown failure (success=False)'})
                        f.write(f"\n❌ Store {store_id} returned success=False\n")

                except Exception as e:
                    failures.append({'store_id': store_id, 'error': str(e)})
                    f.write(f"\n❌ Store {store_id} failed: {str(e)}\n")
                    f.write(traceback.format_exc() + "\n")

                if progress_every and (i % progress_every == 0):
                    f.write(f"\n--- Progress: {i}/{len(store_ids)} stores processed ---\n")
                    f.flush()

            if slot > 0:
                for j in range(slot, len(axes_page)):
                    axes_page[j].axis("off")
                fig_page.tight_layout()
                pdf.savefig(fig_page)
                plt.close(fig_page)

    finally:
        pdf.close()

    if all_results:
        results_df = pd.DataFrame(all_results)

        # ✅ CHANGED NAME HERE
        results_df.to_csv('store_characteristics_1_1115.csv', index=False)

        if failures:
            pd.DataFrame(failures).to_csv('failed_stores.csv', index=False)

        print("\n" + "=" * 60)
        print("PROCESSING COMPLETE")
        print("=" * 60)
        print(f"Successful: {len(all_results)}/{len(store_ids)}")
        print(f"Failed: {len(failures)}/{len(store_ids)}")
        print("\nFiles written:")
        print(f"  - {output_filename}")
        print("  - store_characteristics_1_1115.csv")
        if failures:
            print("  - failed_stores.csv")
        print(f"  - {pdf_filename}")

        return results_df, failures

    print("\n❌ No stores processed successfully.")
    return None, failures


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    """
    IMPORTANT:
    - This script expects you ALREADY have your cleaned/processed dataframe named `df`
    - This will run stores 1..1115 and write output to files (no notebook spam)
    """

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    required = ["Store","Date","Sales","Open","Promo","SchoolHoliday","StateHoliday","IsPromoMonth","DayOfWeek"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in df: {missing}")

    results, failures = run_step2_stores_1_to_1115(
        df,
        window_weeks=81,
        output_filename='forecast_results_stores_1_1115.txt',
        pdf_filename='Appendix_Forecast_Plots_Stores_1_1115.pdf',
        progress_every=50,
        plots_per_page=8
    )


# analysis of 1-1115 stores 

In [ ]:
print("df columns:", df.columns.tolist())

In [ ]:
import pandas as pd

# =========================
# ANALYSIS HELPERS
# =========================

def analyze_sunday_patterns(results_df):
    print(f"\n{'='*60}")
    print("SUNDAY PATTERN ANALYSIS")
    print('='*60)

    sunday_stats = results_df.groupby('sunday_category').agg({
        'store_id': 'count',
        'rmse': ['mean', 'std', 'min', 'max'],
        'mape': 'mean'
    }).round(2)

    sunday_stats.columns = ['Count', 'Avg_RMSE', 'Std_RMSE', 'Min_RMSE', 'Max_RMSE', 'Avg_MAPE']
    sunday_stats = sunday_stats.reset_index()
    print(sunday_stats.to_string(index=False))
    return sunday_stats


def analyze_model_architectures(results_df):
    print(f"\n{'='*60}")
    print("MODEL ARCHITECTURE ANALYSIS")
    print('='*60)

    results_df = results_df.copy()
    results_df['model_type'] = results_df.apply(
        lambda row: f"{row['model_order']}{row['seasonal_order']}", axis=1
    )

    model_stats = results_df.groupby('model_type').agg({
        'store_id': 'count',
        'rmse': 'mean',
        'mape': 'mean'
    }).round(2)

    model_stats.columns = ['Count', 'Avg_RMSE', 'Avg_MAPE']
    model_stats = model_stats.reset_index()
    model_stats['Pct'] = (model_stats['Count'] / len(results_df) * 100).round(1)
    model_stats = model_stats.sort_values('Count', ascending=False)

    print(model_stats.head(30).to_string(index=False))
    return model_stats


def analyze_exogenous_variables(results_df):
    print(f"\n{'='*60}")
    print("EXOGENOUS VARIABLE IMPORTANCE")
    print('='*60)

    all_exog = []
    for v in results_df['selected_cols'].dropna():
        if isinstance(v, list):
            all_exog.extend(v)
        elif isinstance(v, str):
            s = v.strip()
            if s.startswith('[') and s.endswith(']'):
                try:
                    parsed = eval(s, {"__builtins__": {}})
                    if isinstance(parsed, list):
                        all_exog.extend(parsed)
                except:
                    pass

    if not all_exog:
        print("No exogenous variable data available")
        return None

    exog_counts = pd.Series(all_exog).value_counts()
    exog_stats = pd.DataFrame({
        'Variable': exog_counts.index,
        'Count': exog_counts.values,
        'Pct_Stores': (exog_counts.values / len(results_df) * 100).round(1)
    })

    print(exog_stats.head(30).to_string(index=False))
    return exog_stats


# =========================
# BEST & WORST STORES
# =========================

def analyze_best_worst_stores(results_df, top_n=10):
    print(f"\n{'='*60}")
    print("BEST PERFORMING STORES ")
    print('='*60)

    best = results_df.sort_values("mape").head(top_n)
    print(best[['store_id','avg_sales','mape','rmse','r2']].to_string(index=False))

    print(f"\n{'='*60}")
    print("WORST PERFORMING STORES ")
    print('='*60)

    worst = results_df.sort_values("mape", ascending=False).head(top_n)
    print(worst[['store_id','avg_sales','mape','rmse','r2']].to_string(index=False))

    return best, worst

#-----------------------------------------------------------
    
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# Helper: rebuild StoreType
# -------------------------
def add_storetype_from_dummies(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    b = df.get("StoreType_b", 0)
    c = df.get("StoreType_c", 0)
    d = df.get("StoreType_d", 0)

    # make sure they behave like 0/1 arrays
    b = pd.Series(b).fillna(0).astype(int).values
    c = pd.Series(c).fillna(0).astype(int).values
    d = pd.Series(d).fillna(0).astype(int).values

    # baseline 'a' when none of b/c/d is 1
    st = np.where(b == 1, "b",
         np.where(c == 1, "c",
         np.where(d == 1, "d", "a")))

    df["StoreType"] = st
    return df


# -------------------------
# Main: Store type analysis
# -------------------------
def analyze_by_store_type(results_df: pd.DataFrame, df: pd.DataFrame, save_plot=True):
    print(f"\n{'='*60}")
    print("STORE TYPE ANALYSIS")
    print('='*60)

    # rebuild StoreType
    df2 = add_storetype_from_dummies(df)

    # Store -> StoreType mapping
    store_types = df2[["Store", "StoreType"]].drop_duplicates()

    merged = results_df.merge(
        store_types,
        left_on="store_id",
        right_on="Store",
        how="left"
    )

    # stats
    type_stats = merged.groupby("StoreType").agg(
        Count=("store_id", "count"),
        Avg_RMSE=("rmse", "mean"),
        Std_RMSE=("rmse", "std"),
        Avg_MAPE=("mape", "mean"),
        Avg_Sales=("avg_sales", "mean"),
        Avg_PromoRate=("promo_rate", "mean"),
        Avg_SundayOpenRate=("sunday_open_rate", "mean"),
        Avg_R2=("r2", "mean"),
    ).round(2).reset_index()

    # nice order a,b,c,d
    order = pd.Categorical(type_stats["StoreType"], categories=["a", "b", "c", "d"], ordered=True)
    type_stats = type_stats.assign(StoreType=order).sort_values("StoreType").reset_index(drop=True)

    print(type_stats.to_string(index=False))

    # plots
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    axes[0, 0].bar(type_stats["StoreType"].astype(str), type_stats["Avg_RMSE"])
    axes[0, 0].set_title("Average RMSE by Store Type")
    axes[0, 0].set_ylabel("RMSE")
    axes[0, 0].grid(True, alpha=0.3, axis="y")

    axes[0, 1].bar(type_stats["StoreType"].astype(str), type_stats["Avg_MAPE"])
    axes[0, 1].set_title("Average MAPE by Store Type")
    axes[0, 1].set_ylabel("MAPE (%)")
    axes[0, 1].grid(True, alpha=0.3, axis="y")

    axes[1, 0].bar(type_stats["StoreType"].astype(str), type_stats["Avg_Sales"])
    axes[1, 0].set_title("Average Sales by Store Type")
    axes[1, 0].set_ylabel("Avg Sales")
    axes[1, 0].grid(True, alpha=0.3, axis="y")

    axes[1, 1].bar(type_stats["StoreType"].astype(str), type_stats["Avg_SundayOpenRate"])
    axes[1, 1].set_title("Average Sunday Open Rate by Store Type")
    axes[1, 1].set_ylabel("Sunday Open Rate")
    axes[1, 1].grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    if save_plot:
        plt.savefig("store_type_analysis.png", dpi=150)
    plt.show()

    return merged, type_stats
    

# =========================
# RUN ANALYSIS FROM NEW CSV
# =========================

CSV_PATH = "C:/Users/shrut/Downloads/store_characteristics_1_1115.csv"

results_df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(results_df)} stores from {CSV_PATH}")

_ = analyze_sunday_patterns(results_df)
_ = analyze_model_architectures(results_df)
_ = analyze_exogenous_variables(results_df)
best, worst = analyze_best_worst_stores(results_df, top_n=10)
_ = analyze_by_store_type(results_df, df)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── load ──────────────────────────────────────────────────
CSV_PATH = r"C:/Users/shrut/Downloads/store_characteristics_1_1115.csv"
results_df = pd.read_csv(CSV_PATH)

# ── stats ──────────────────────────────────────────────────
mean_rmse = results_df['rmse'].mean()
std_rmse  = results_df['rmse'].std()
med_rmse  = results_df['rmse'].median()
threshold = mean_rmse + 2 * std_rmse

outliers  = results_df[results_df['rmse'] > threshold].sort_values('rmse', ascending=False)

print(f"Mean RMSE   : {mean_rmse:.2f}")
print(f"Median RMSE : {med_rmse:.2f}")
print(f"Std RMSE    : {std_rmse:.2f}")
print(f"Threshold   : {threshold:.2f}  (mean + 2*std)")
print(f"Outliers    : {len(outliers)} stores above threshold")
print()
print(outliers[['store_id', 'rmse', 'mape', 'avg_sales', 
                'sunday_category', 'model_order']].to_string(index=False))

# ── plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# histogram
axes[0].hist(results_df['rmse'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].aCSV_PATH = r"C:/Users/shrut/Downloads/store_characteristics_1_1115.csv"
results_df = pd.read_csv(CSV_PATH)

# ── stats ──────────────────────────────────────────────────
mean_rmse = results_df['rmse'].mean()
std_rmse  = results_df['rmse'].std()
med_rmse  = results_df['rmse'].median()
threshold = mean_rmse + 2 * std_rmse

outliers  = results_df[results_df['rmse'] > threshold].sort_values('rmse', ascending=False)

print(f"Mean RMSE   : {mean_rmse:.2f}")
print(f"Median RMSE : {med_rmse:.2f}")
print(f"Std RMSE    : {std_rmse:.2f}")
print(f"Threshold   : {threshold:.2f}  (mean + 2*std)")
print(f"Outliers    : {len(outliers)} stores above threshold")
print()
print(outliers[['store_id', 'rmse', 'mape', 'avg_sales', 
                'sunday_category', 'model_order']].to_string(index=False))

# ── plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# histogram
axes[0].hist(results_df['rmse'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(mean_rmse,  color='red',    linestyle='--', linewidth=1.5, label=f'Mean: {mean_rmse:.0f}')
axes[0].axvline(med_rmse,   color='green',  linestyle='--', linewidth=1.5, label=f'Median: {med_rmse:.0f}')
axes[0].axvline(threshold,  color='orange', linestyle='--', linewidth=1.5, label=f'Threshold: {threshold:.0f}')
axes[0].set_title('RMSE Distribution Across 1,115 Stores')
axes[0].set_xlabel('RMSE')
axes[0].set_ylabel('Number of Stores')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# boxplot
axes[1].boxplot(results_df['rmse'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('RMSE Boxplot')
axes[1].set_ylabel('RMSE')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rmse_outlier_analysis.png', dpi=150)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load data
CSV_PATH = r"C:/Users/shrut/Downloads/store_characteristics_1_1115.csv"
results_df = pd.read_csv(CSV_PATH)

rmse = results_df['rmse'].dropna()

# ── Statistics ─────────────────────────────
mean_rmse = rmse.mean()
median_rmse = rmse.median()
std_rmse = rmse.std()

Q1 = rmse.quantile(0.25)
Q3 = rmse.quantile(0.75)
IQR = Q3 - Q1

threshold = Q3 + 1.5 * IQR

outliers = results_df[results_df['rmse'] > threshold].sort_values('rmse', ascending=False)

print(f"Mean RMSE   : {mean_rmse:.2f}")
print(f"Median RMSE : {median_rmse:.2f}")
print(f"Q1          : {Q1:.2f}")
print(f"Q3          : {Q3:.2f}")
print(f"IQR         : {IQR:.2f}")
print(f"Threshold   : {threshold:.2f}  (Q3 + 1.5*IQR)")
print(f"Outliers    : {len(outliers)} stores")
print(outliers[['store_id', 'rmse', 'mape', 'avg_sales', 
                'sunday_category', 'model_order']].to_string(index=False))
# ── Plot ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Histogram
axes[0].hist(rmse, bins=50, color='steelblue', edgecolor='white', alpha=0.85)

axes[0].axvline(mean_rmse,
                color='red',
                linestyle='--',
                linewidth=2,
                label=f"Mean: {mean_rmse:.0f}")

axes[0].axvline(median_rmse,
                color='green',
                linestyle='--',
                linewidth=2,
                label=f"Median: {median_rmse:.0f}")

axes[0].axvline(threshold,
                color='orange',
                linestyle='--',
                linewidth=2,
                label=f"Outlier threshold: {threshold:.0f}")

axes[0].set_title("RMSE Distribution Across 1,115 Stores")
axes[0].set_xlabel("RMSE")
axes[0].set_ylabel("Number of Stores")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Boxplot
axes[1].boxplot(rmse,
                vert=True,
                patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))

axes[1].set_title("RMSE Boxplot")
axes[1].set_ylabel("RMSE")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

plt.savefig("rmse_distribution_analysis.png", dpi=300)

plt.show()

In [ ]:
# 1. Log transformation (makes data more symmetric)
rmse_log = np.log1p(rmse)  # log(1+rmse)
# Then apply outlier detection on transformed data

# 2. More conservative multiplier for skewed data
threshold_conservative = Q3 + 2.0 * IQR  # Less aggressive
threshold_aggressive = Q3 + 1.0 * IQR    # More aggressive

# 3. Percentile-based method
p95 = rmse.quantile(0.95)  # Top 5% as outliers
p99 = rmse.quantile(0.99)  # Top 1% as outliers

# 4. Modified Z-score using median
from scipy import stats
median = rmse.median()
mad = stats.median_abs_deviation(rmse)  # Median Absolute Deviation
modified_z_scores = 0.6745 * (rmse - median) / mad
outliers_mad = rmse[abs(modified_z_scores) > 3.5]

In [ ]:
# Compare different methods
print(f"Your threshold (Q3 + 1.5×IQR): {threshold:.2f}")
print(f"99th percentile: {rmse.quantile(0.99):.2f}")
print(f"95th percentile: {rmse.quantile(0.95):.2f}")
print(f"Mean + 2×std: {rmse.mean() + 2*rmse.std():.2f}")
print(f"Mean + 3×std: {rmse.mean() + 3*rmse.std():.2f}")

In [ ]:
#dont use this

In [ ]:
import re
import numpy as np
import pandas as pd

# =========================
# METRICS (7-day)
# =========================
def mape_safe(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred))
    mask = denom != 0
    if mask.sum() == 0:
        return np.nan
    return 100 * np.mean(2 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask])

def compute_metrics_7(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    bias = float(np.mean(y_pred - y_true))

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return {
        "rmse": rmse,
        "mae": mae,
        "mape": mape_safe(y_true, y_pred),
        "smape": smape(y_true, y_pred),
        "bias": bias,
        "r2": r2
    }

# =========================
# PARSE THE TXT OUTPUT
# =========================
def parse_last7_from_txt(txt_path):
    """
    Parses Step 2 text file and extracts last-7 test (Actual, Predicted) for each store.
    Returns DataFrame with 7-day metrics per store_id.
    """
    store_id = None
    collecting = False
    rows = {}  # store_id -> list of (actual, pred)

    date_line = re.compile(r"^\d{4}-\d{2}-\d{2}\s+")

    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")

            # Detect new store block
            m = re.search(r"STORE\s+(\d+)\s+-\s+FORECAST RESULTS", line)
            if m:
                store_id = int(m.group(1))
                rows[store_id] = []
                collecting = False
                continue

            # Detect the table header line that starts the date rows
            if "Last 7 Test Days + Next 7 Forecast Days" in line:
                collecting = True
                continue

            if not collecting or store_id is None:
                continue

            # Stop collecting when the separator ends and next store begins naturally
            if line.startswith("-" * 10) or line.strip() == "":
                continue

            # Parse data rows
            if date_line.match(line):
                parts = line.split()
                # expected: [date, day, actual, predicted]
                if len(parts) >= 4:
                    actual_str = parts[2]
                    pred_str = parts[3]

                    # Only keep TEST rows (actual is numeric). Forecast rows have actual = '-'
                    if actual_str != "-":
                        try:
                            actual = float(actual_str)
                            pred = float(pred_str)
                            rows[store_id].append((actual, pred))
                        except:
                            pass

                # We only need last 7 test rows
                if len(rows[store_id]) >= 7:
                    collecting = False

    # Build metrics table
    out = []
    for sid, ap in rows.items():
        if len(ap) == 7:
            y_true = [x[0] for x in ap]
            y_pred = [x[1] for x in ap]
            met = compute_metrics_7(y_true, y_pred)
            met["store_id"] = sid
            out.append(met)

    return pd.DataFrame(out)

# =========================
# BUILD results_df_7
# =========================
CSV_PATH = r"C:/Users/shrut/Downloads/store_characteristics_1_1115.csv"
TXT_PATH = r"C:/Users/shrut/Downloads/forecast_results_stores_1_1115.txt"  # <-- set where your txt is

results_df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(results_df)} stores from {CSV_PATH}")

last7_metrics_df = parse_last7_from_txt(TXT_PATH)
print(f"Parsed last-7 metrics for {len(last7_metrics_df)} stores from {TXT_PATH}")

# Keep old (14-day) metrics for reference
results_df_7 = results_df.copy()
for c in ["rmse","mae","mape","smape","bias","r2"]:
    if c in results_df_7.columns:
        results_df_7 = results_df_7.rename(columns={c: f"{c}_14"})

# Merge in 7-day metrics (these become the default columns your analysis uses)
results_df_7 = results_df_7.merge(last7_metrics_df, on="store_id", how="left")

# Optional: drop stores where parsing failed
results_df_7 = results_df_7.dropna(subset=["rmse","mape"])
print(f"Final results_df_7 stores available for 7-day analysis: {len(results_df_7)}")

In [ ]:
_ = analyze_sunday_patterns(results_df_7)
_ = analyze_model_architectures(results_df_7)
_ = analyze_exogenous_variables(results_df_7)
best, worst = analyze_best_worst_stores(results_df_7, top_n=10)
_ = analyze_by_store_type(results_df_7, df)

In [ ]:
results_df['selected_cols'].isna().sum()

In [ ]:
results_df[results_df['selected_cols'].isin(['[]',''])].shape

## store 1 rolling window

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from collections import Counter
import textwrap

import pmdarima as pm
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =============================================================================
# IMPROVED UTILITY FUNCTIONS
# =============================================================================

def enforce_non_negative(predictions, open_status=None, index=None):
    """
    Ensure predictions are non-negative.
    If store is closed (Open=0), sales should be 0.
    Returns a pandas Series if index is provided, otherwise numpy array.
    """
    predictions = np.array(predictions)
    predictions = np.maximum(predictions, 0)

    if open_status is not None:
        open_status = np.array(open_status)
        predictions[open_status == 0] = 0

    if index is not None:
        return pd.Series(predictions, index=index)
    return predictions


def calculate_mape_safe(actual, predicted):
    """
    Safe MAPE calculation that handles zero values.
    Returns NaN if all actual values are zero.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)

    mask = actual != 0
    if mask.sum() == 0:
        return np.nan

    predicted = enforce_non_negative(predicted)
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


def calculate_smape(actual, predicted):
    """
    Symmetric MAPE - handles zero values better than MAPE.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)

    predicted = enforce_non_negative(predicted)

    denominator = (np.abs(actual) + np.abs(predicted))
    mask = denominator != 0
    if mask.sum() == 0:
        return np.nan

    return 100 / len(actual) * np.sum(2 * np.abs(predicted[mask] - actual[mask]) / denominator[mask])


def calculate_all_metrics(y_true, y_pred, X_test=None):
    """
    Calculate all performance metrics safely.
    """
    metrics = {}

    if X_test is not None and 'Open' in X_test.columns:
        y_pred_clean = enforce_non_negative(
            y_pred, X_test['Open'],
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )
    else:
        y_pred_clean = enforce_non_negative(
            y_pred,
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )

    metrics['rmse'] = float(np.sqrt(mean_squared_error(y_true, y_pred_clean)))
    metrics['mae'] = float(mean_absolute_error(y_true, y_pred_clean))

    metrics['mape'] = calculate_mape_safe(y_true, y_pred_clean)
    metrics['smape'] = calculate_smape(y_true, y_pred_clean)

    metrics['bias'] = float(np.mean(y_pred_clean - y_true))
    metrics['std_error'] = float(np.std(y_pred_clean - y_true))

    ss_res = np.sum((y_true - y_pred_clean) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    metrics['r2_adj'] = float(1 - (ss_res / ss_tot)) if ss_tot > 0 else np.nan

    return metrics, y_pred_clean


# =============================================================================
# DATA PREPARATION
# =============================================================================

def prepare_store_data(
    df: pd.DataFrame,
    store_id: int,
    exogenous_columns=("Open", "Promo", "SchoolHoliday", "StateHoliday", "IsPromoMonth", "DayOfWeek"),
):
    s = df[df["Store"] == store_id].copy()
    s["Date"] = pd.to_datetime(s["Date"], dayfirst=True)
    s = s.sort_values("Date").set_index("Date")

    y = s["Sales"].astype(float).sort_index()
    X = s[list(exogenous_columns)].copy().loc[y.index]

    X["Open"] = X["Open"].astype(int)
    X["Promo"] = X["Promo"].astype(int)
    X["SchoolHoliday"] = X["SchoolHoliday"].astype(int)
    X["IsPromoMonth"] = X["IsPromoMonth"].astype(int)
    X["StateHoliday"] = (X["StateHoliday"].astype(str) != "0").astype(int)

    dow = pd.get_dummies(X["DayOfWeek"], prefix="DOW", drop_first=True).astype(float)

    X_num = X.drop(columns=["DayOfWeek"]).astype(float)
    Xcand = pd.concat([X_num, dow], axis=1).astype(float)

    return y, Xcand


def select_orders_with_auto_arima(y_tr, X_tr, seasonal_on=True, m=7, trace=False):
    if seasonal_on:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=True,
            m=m,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            D=None,
            max_p=3, max_q=3,
            max_P=2, max_Q=2,
            max_d=0,
            max_D=1,
            start_p=1,
            start_q=1,
            start_P=0,
            start_Q=0,
            test='adf',
            seasonal_test='ocsb',
            n_fits=20,
            with_intercept=True
        )
        return sel.order, sel.seasonal_order, float(sel.aic()), float(sel.bic())
    else:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=False,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            max_p=3, max_q=3,
            max_d=0,
            start_p=1,
            start_q=1,
            test='adf',
            n_fits=15,
            with_intercept=True
        )
        return sel.order, (0, 0, 0, 0), float(sel.aic()), float(sel.bic())


def backward_elimination_exog(y_tr, X_tr, order, seasonal_order, p_thresh=0.05, max_iter=50):
    cols = list(X_tr.columns)

    if len(cols) == 0:
        mod = sm.tsa.SARIMAX(
            y_tr, exog=None, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        return [], res

    for _ in range(max_iter):
        X_use = X_tr[cols] if cols else None

        mod = sm.tsa.SARIMAX(
            y_tr, exog=X_use, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        if not cols:
            return [], res

        pvals = res.pvalues
        p_exog = pvals[[c for c in pvals.index if c in cols]]

        if len(p_exog) == 0:
            return cols, res

        worst_col = p_exog.idxmax()
        worst_p = float(p_exog.max())

        if worst_p <= p_thresh:
            return cols, res

        cols.remove(worst_col)

        if not cols:
            mod2 = sm.tsa.SARIMAX(
                y_tr, exog=None, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False
            )
            res2 = mod2.fit(disp=False)
            return [], res2

    return cols, res


# =============================================================================
# CORRECT SLIDING WINDOW VALIDATION
# =============================================================================

def create_sliding_splits(
    initial_train_start='2013-01-01',
    initial_train_end='2014-07-31',
    horizon_days=14,
    n_splits=26
):
    """
    Create sliding window splits where BOTH training and test windows slide forward.
    """
    initial_train_start = pd.to_datetime(initial_train_start)
    initial_train_end = pd.to_datetime(initial_train_end)
    
    splits = []
    
    for i in range(n_splits):
        shift_days = i * horizon_days
        train_start = initial_train_start + pd.Timedelta(days=shift_days)
        train_end = initial_train_end + pd.Timedelta(days=shift_days)
        test_start = train_end + pd.Timedelta(days=1)
        test_end = test_start + pd.Timedelta(days=horizon_days - 1)
        
        splits.append({
            'split_id': i + 1,
            'train_start': train_start,
            'train_end': train_end,
            'test_start': test_start,
            'test_end': test_end,
            'shift_days': shift_days
        })
    
    return splits


def print_splits_summary(splits):
    """
    Print a nice summary of all splits.
    """
    print(f"\n{'='*100}")
    print(f"SLIDING WINDOW SPLITS SUMMARY")
    print(f"{'='*100}")
    print(f"{'Block':<6} {'Training Period':<35} {'Test Period':<35} {'Shift'}")
    print(f"{'-'*100}")
    
    for split in splits[:5]:
        train_period = f"{split['train_start'].date()} to {split['train_end'].date()}"
        test_period = f"{split['test_start'].date()} to {split['test_end'].date()}"
        print(f"{split['split_id']:<6} {train_period:<35} {test_period:<35} {split['shift_days']} days")
    
    if len(splits) > 10:
        print("...")
    
    for split in splits[-5:]:
        train_period = f"{split['train_start'].date()} to {split['train_end'].date()}"
        test_period = f"{split['test_start'].date()} to {split['test_end'].date()}"
        print(f"{split['split_id']:<6} {train_period:<35} {test_period:<35} {split['shift_days']} days")
    
    print(f"{'='*100}")


def evaluate_single_split(y_full, X_full, split, seasonal_on=True, m=7, 
                          p_thresh=0.05, auto_arima_trace=False):
    """
    Evaluate a single split.
    """
    train_mask = (y_full.index >= split['train_start']) & (y_full.index <= split['train_end'])
    test_mask = (y_full.index >= split['test_start']) & (y_full.index <= split['test_end'])
    
    y_train = y_full[train_mask]
    y_test = y_full[test_mask]
    
    if len(y_test) == 0:
        print(f"    Warning: No test data available for split {split['split_id']}")
        return None
    
    X_train = X_full.loc[y_train.index]
    X_test = X_full.loc[y_test.index]
    
    try:
        order, seas, aic, bic = select_orders_with_auto_arima(
            y_train, X_train, seasonal_on=seasonal_on, m=m, trace=auto_arima_trace
        )
        
        selected_cols, _ = backward_elimination_exog(
            y_train, X_train, order, seas, p_thresh=p_thresh
        )
        
        if selected_cols:
            X_tr_final = X_train[selected_cols]
            X_te_final = X_test[selected_cols]
        else:
            X_tr_final = None
            X_te_final = None
            selected_cols = []
        
        mod = sm.tsa.SARIMAX(
            y_train, exog=X_tr_final, order=order, seasonal_order=seas,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        
        pred_raw = res.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_te_final)
        pred = enforce_non_negative(
            pred_raw,
            X_test['Open'] if 'Open' in X_test.columns else None,
            index=pred_raw.index
        )
        
        rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
        mae = float(mean_absolute_error(y_test, pred))
        
        metrics, _ = calculate_all_metrics(y_test, pred, X_test)
        
        return {
            'split_id': split['split_id'],
            'train_start': split['train_start'],
            'train_end': split['train_end'],
            'test_start': split['test_start'],
            'test_end': split['test_end'],
            'rmse': rmse,
            'mae': mae,
            'mape': metrics['mape'],
            'smape': metrics['smape'],
            'order': order,
            'seasonal_order': seas,
            'selected_cols': selected_cols,
            'aic': aic,
            'bic': bic,
            'n_train': len(y_train),
            'n_test': len(y_test)
        }
        
    except Exception as e:
        print(f"    Error in split {split['split_id']}: {str(e)[:100]}...")
        return None


def run_exog_selection_sliding_window(
    df: pd.DataFrame,
    store_id: int,
    initial_train_start='2013-01-01',
    initial_train_end='2014-07-31',
    horizon_days=14,
    n_splits=26,
    seasonal_on=True,
    m=7,
    p_thresh=0.05,
    auto_arima_trace=False
):
    """
    Main function that implements CORRECT sliding window.
    """
    
    print(f"\n{'='*80}")
    print(f"EXOGENOUS VARIABLE SELECTION - STORE {store_id}")
    print(f"{'='*80}")
    print(f"Initial training window: {initial_train_start} to {initial_train_end} (1 year 7 months)")
    print(f"Forecast horizon: {horizon_days} days")
    print(f"Number of test blocks: {n_splits}")
    print(f"\n{'='*80}")
    print(f"SLIDING WINDOW MECHANISM:")
    print(f"{'='*80}")
    print(f"  * BOTH training AND test windows slide forward by {horizon_days} days each block")
    print(f"  * Training window always 1 year 7 months (577 days)")
    print(f"  * Test window always 14 days")
    print(f"  * 26 blocks covering from 2014-08-01 to 2015-07-31")
    
    y_full, X_full = prepare_store_data(df, store_id)
    print(f"\nData loaded: {len(y_full)} days from {y_full.index[0].date()} to {y_full.index[-1].date()}")
    print(f"Total exogenous variables available: {len(X_full.columns)}")
    print(f"Variables: {list(X_full.columns)}")
    
    splits = create_sliding_splits(
        initial_train_start=initial_train_start,
        initial_train_end=initial_train_end,
        horizon_days=horizon_days,
        n_splits=n_splits
    )
    
    print_splits_summary(splits)
    
    all_results = []
    
    print(f"\n{'='*80}")
    print(f"EVALUATING EACH SPLIT")
    print(f"{'='*80}")
    
    for split in splits:
        print(f"\nSplit {split['split_id']}:")
        print(f"  Train: {split['train_start'].date()} to {split['train_end'].date()}")
        print(f"  Test:  {split['test_start'].date()} to {split['test_end'].date()}")
        
        result = evaluate_single_split(
            y_full=y_full,
            X_full=X_full,
            split=split,
            seasonal_on=seasonal_on,
            m=m,
            p_thresh=p_thresh,
            auto_arima_trace=auto_arima_trace
        )
        
        if result:
            all_results.append(result)
            print(f"  * RMSE: {result['rmse']:.2f}")
            print(f"    Selected exog: {result['selected_cols'] if result['selected_cols'] else 'None'}")
            print(f"    Model order: {result['order']}{result['seasonal_order']}")
    
    if not all_results:
        print("\n*** No successful splits! ***")
        return None
    
    results_df = pd.DataFrame(all_results)
    
    return results_df, splits


def analyze_selection_frequency(results_df):
    """
    Analyze how often each exogenous variable is selected across splits.
    """
    var_selection_count = {}
    total_splits = len(results_df)
    
    all_vars = []
    for selected in results_df['selected_cols']:
        if selected:
            all_vars.extend(selected)
    all_vars = sorted(set(all_vars))
    
    for var in all_vars:
        count = 0
        for selected in results_df['selected_cols']:
            if selected and var in selected:
                count += 1
        var_selection_count[var] = count
    
    print(f"\n{'='*80}")
    print(f"VARIABLE SELECTION FREQUENCY ANALYSIS")
    print(f"{'='*80}")
    print(f"Across {total_splits} splits:")
    print()
    print(f"{'Variable':<20} {'Times Selected':<15} {'Frequency':<10}")
    print("-" * 45)
    
    for var, count in sorted(var_selection_count.items(), key=lambda x: x[1], reverse=True):
        freq = (count / total_splits) * 100
        print(f"  * {var:<18} {count:>5}/{total_splits:<9} {freq:>6.1f}%")
    
    consistent_vars = [var for var, count in var_selection_count.items() 
                      if count / total_splits > 0.5]
    
    print(f"\nConsistently selected variables (>50% of splits):")
    if consistent_vars:
        for var in consistent_vars:
            freq = (var_selection_count[var] / total_splits) * 100
            print(f"  - {var} ({freq:.1f}%)")
    else:
        print(f"  No variables selected consistently")
    
    return var_selection_count, consistent_vars


def generate_stability_report(results_df):
    """
    Generate detailed stability report.
    """
    rmse_values = results_df['rmse'].values
    
    print(f"\n{'='*80}")
    print(f"STABILITY REPORT")
    print(f"{'='*80}")
    print(f"\nPerformance across {len(results_df)} splits:")
    print(f"  * Average RMSE: {np.mean(rmse_values):.2f}")
    print(f"  * Std Deviation: {np.std(rmse_values):.2f}")
    print(f"  * Coefficient of Variation: {(np.std(rmse_values)/np.mean(rmse_values))*100:.1f}%")
    print(f"  * Min RMSE: {np.min(rmse_values):.2f}")
    print(f"  * Max RMSE: {np.max(rmse_values):.2f}")
    print(f"  * Range: {np.max(rmse_values) - np.min(rmse_values):.2f}")
    
    print(f"\nSplit-by-split performance:")
    print("-" * 100)
    print(f"{'Split':<6} {'Test Period':<25} {'RMSE':<10} {'MAE':<10} {'MAPE':<8} {'Selected Exog'}")
    print("-" * 100)
    
    for _, row in results_df.sort_values('split_id').iterrows():
        test_period = f"{row['test_start'].date()} to {row['test_end'].date()}"
        selected = str(row['selected_cols'])[:30] if row['selected_cols'] else 'None'
        mape = f"{row['mape']:.1f}%" if not np.isnan(row['mape']) else "N/A"
        print(f"{row['split_id']:<6} {test_period:<25} {row['rmse']:<10.2f} {row['mae']:<10.2f} {mape:<8} {selected}")
    
    return results_df


def select_best_model_based_on_criteria(results_df):
    """
    Select the best model based on:
    1. Lowest Avg RMSE among models that appear in at least 2 splits
    2. If no model appears twice, then take the best single appearance
    """
    
    print(f"\n{'='*80}")
    print(f"MODEL SELECTION BASED ON CRITERIA")
    print(f"{'='*80}")
    
    # Create a unique key for EACH COMPLETE MODEL CONFIGURATION
    results_df['model_key'] = results_df.apply(
        lambda row: (
            tuple(sorted(row['selected_cols'])) if row['selected_cols'] else tuple(),
            row['order'],
            row['seasonal_order']
        ), axis=1
    )
    
    # Calculate metrics for each unique model configuration
    model_performance = []
    
    for model_key in results_df['model_key'].unique():
        subset = results_df[results_df['model_key'] == model_key]
        
        exog_tuple, order, seasonal_order = model_key
        exog_list = list(exog_tuple) if exog_tuple else []
        
        avg_rmse = subset['rmse'].mean()
        std_rmse = subset['rmse'].std()
        frequency = len(subset)
        frequency_pct = (frequency / len(results_df)) * 100
        
        split_ids = sorted(subset['split_id'].tolist())
        rmse_values = subset['rmse'].tolist()
        
        model_performance.append({
            'exog_vars': exog_list,
            'order': order,
            'seasonal_order': seasonal_order,
            'model_str': f"{order}{seasonal_order}",
            'avg_rmse': avg_rmse,
            'std_rmse': std_rmse,
            'frequency': frequency,
            'frequency_pct': frequency_pct,
            'min_rmse': subset['rmse'].min(),
            'max_rmse': subset['rmse'].max(),
            'split_ids': split_ids,
            'rmse_values': rmse_values
        })
    
    model_df = pd.DataFrame(model_performance)
    model_df = model_df.sort_values(['avg_rmse', 'std_rmse'])
    
    # Select best model (prefer models with at least 2 appearances)
    reliable_models = model_df[model_df['frequency'] >= 2]
    if len(reliable_models) > 0:
        best_model = reliable_models.iloc[0]
        print(f"\nNOTE: Selected from models appearing in at least 2 splits for reliability")
    else:
        best_model = model_df.iloc[0]
        print(f"\nNOTE: No model appears in multiple splits - selecting best single appearance")
    
    print(f"\n{'='*80}")
    print(f"BEST MODEL SELECTED BASED ON CRITERIA")
    print(f"{'='*80}")
    print(f"\nSelection Criteria:")
    print(f"  1. Primary: Lowest Average RMSE among models with ≥2 appearances")
    print(f"  2. Secondary: Lowest Std RMSE (stability)")
    print(f"\nSelected Model:")
    
    # Convert order tuples to strings for display
    order_str = str(best_model['order'])
    seasonal_str = str(best_model['seasonal_order'])
    
    if best_model['exog_vars']:
        exog_str = '[' + ', '.join([f"'{v}'" for v in best_model['exog_vars']]) + ']'
        wrapped_exog = textwrap.wrap(exog_str, width=70, subsequent_indent=' ' * 19)
        print(f"  ARIMA Order: {order_str}{seasonal_str}")
        print(f"  Exogenous Variables: {wrapped_exog[0]}")
        for line in wrapped_exog[1:]:
            print(f"                     {line}")
    else:
        print(f"  ARIMA Order: {order_str}{seasonal_str}")
        print(f"  Exogenous Variables: None")
    
    print(f"  Average RMSE: {best_model['avg_rmse']:.2f}")
    print(f"  Std RMSE (stability): {best_model['std_rmse']:.2f}")
    print(f"  Frequency: {best_model['frequency']}/{len(results_df)} splits ({best_model['frequency_pct']:.1f}%)")
    print(f"  RMSE Range: {best_model['min_rmse']:.2f} - {best_model['max_rmse']:.2f}")
    print(f"  Appears in splits: {best_model['split_ids']}")
    
    if best_model['frequency'] > 1:
        print(f"  Individual RMSEs: {[round(r, 2) for r in best_model['rmse_values']]}")
    
    # Show top 3 recommended models
    print(f"\nTOP 3 RECOMMENDED MODELS:")
    print("-" * 100)
    print(f"{'Rank':<6} {'ARIMA Order':<20} {'Avg RMSE':<10} {'Std':<8} {'Freq':<6} {'Exogenous Variables'}")
    print("-" * 100)
    
    for rank in range(min(3, len(model_df))):
        row = model_df.iloc[rank]
        order_display = f"{row['order']}{row['seasonal_order']}"
        exog_display = str(row['exog_vars'])[:40] + "..." if len(str(row['exog_vars'])) > 40 else str(row['exog_vars'])
        print(f"{rank+1:<6} {order_display:<20} {row['avg_rmse']:<10.2f} {row['std_rmse']:<8.2f} {row['frequency']}/{len(results_df):<6} {exog_display}")
    
    return best_model, model_df


def print_block_summary(results_df):
    """
    Print a summary of each block showing SARIMA model and FULL exogenous variables.
    """
    print(f"\n{'='*80}")
    print(f"BLOCK-BY-BLOCK MODEL SUMMARY")
    print(f"{'='*80}")
    
    all_configs = []
    
    for _, row in results_df.sort_values('split_id').iterrows():
        block_num = row['split_id']
        model_order = f"{row['order']}{row['seasonal_order']}"
        exog_vars = row['selected_cols'] if row['selected_cols'] else ['None']
        
        config = {
            'block': block_num,
            'model': model_order,
            'exog': exog_vars,
            'exog_tuple': tuple(sorted(exog_vars)) if exog_vars != ['None'] else ('None',)
        }
        all_configs.append(config)
    
    print(f"\n{'='*100}")
    print(f"{'Block':<8} {'SARIMA Model':<30} {'Exogenous Variables'}")
    print(f"{'='*100}")
    
    for config in all_configs:
        block_num = config['block']
        model_order = config['model']
        exog_vars = config['exog']
        exog_str = str(exog_vars)
        print(f"Block {block_num:<3}   {model_order:<30} {exog_str}")
    
    print(f"{'='*100}")
    
    # Show frequency of each configuration
    print(f"\nCONFIGURATION FREQUENCY:")
    print(f"{'-'*100}")
    
    config_summary = {}
    for config in all_configs:
        key = (config['model'], config['exog_tuple'])
        if key not in config_summary:
            config_summary[key] = {
                'model': config['model'],
                'exog': config['exog'],
                'blocks': [],
                'count': 0
            }
        config_summary[key]['blocks'].append(config['block'])
        config_summary[key]['count'] += 1
    
    for idx, (key, info) in enumerate(sorted(config_summary.items(), 
                                              key=lambda x: x[1]['blocks'][0])):
        blocks_str = ', '.join([str(b) for b in info['blocks']])
        exog_str = str(info['exog'])
        print(f"\nConfig {idx+1}: {info['model']}")
        print(f"  Exog: {exog_str}")
        print(f"  Appears in blocks: {blocks_str} ({info['count']}/{len(results_df)} blocks, {info['count']/len(results_df)*100:.1f}%)")
    
    print(f"{'-'*100}")
    
    # Find most common configuration
    most_common_key = max(config_summary.items(), key=lambda x: x[1]['count'])[0]
    most_common_info = config_summary[most_common_key]
    
    print(f"\nMOST COMMON MODEL CONFIGURATION:")
    print(f"  Appeared in {most_common_info['count']}/{len(results_df)} blocks ({most_common_info['count']/len(results_df)*100:.1f}%)")
    print(f"  SARIMA Model: {most_common_info['model']}")
    print(f"  Exogenous Variables: {most_common_info['exog']}")
    print(f"  Blocks: {most_common_info['blocks']}")
    
    # Show top 3 most common configurations
    print(f"\nTOP 3 MOST COMMON CONFIGURATIONS:")
    print(f"{'-'*100}")
    print(f"{'Rank':<6} {'Frequency':<15} {'SARIMA Model':<30} {'Exogenous Variables'}")
    print(f"{'-'*100}")
    
    sorted_configs = sorted(config_summary.items(), key=lambda x: x[1]['count'], reverse=True)
    
    for rank, (key, info) in enumerate(sorted_configs[:3], 1):
        pct = (info['count'] / len(results_df)) * 100
        exog_str = str(info['exog'])
        print(f"{rank:<6} {info['count']}/{len(results_df)} ({pct:5.1f}%)   {info['model']:<30} {exog_str}")
    
    print(f"{'='*100}")
    
    return all_configs


def save_exog_selection_results(results_df, store_id, var_selection_count, consistent_vars, best_model, block_configs):
    """
    Save all results to files.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save detailed results
    results_file = f"exog_selection_results_store_{store_id}_{timestamp}.csv"
    
    save_data = []
    for _, row in results_df.iterrows():
        save_data.append({
            'split_id': row['split_id'],
            'train_start': row['train_start'].date(),
            'train_end': row['train_end'].date(),
            'test_start': row['test_start'].date(),
            'test_end': row['test_end'].date(),
            'rmse': row['rmse'],
            'mae': row['mae'],
            'mape': row['mape'],
            'smape': row['smape'],
            'order': str(row['order']),
            'seasonal_order': str(row['seasonal_order']),
            'selected_cols': str(row['selected_cols']),
            'aic': row['aic'],
            'bic': row['bic'],
            'n_train': row['n_train'],
            'n_test': row['n_test']
        })
    
    save_df = pd.DataFrame(save_data)
    save_df.to_csv(results_file, index=False, encoding='utf-8')
    
    # Save summary
    summary_file = f"exog_selection_summary_store_{store_id}_{timestamp}.txt"
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(f"EXOGENOUS VARIABLE SELECTION RESULTS - STORE {store_id}\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Selection method: Backward elimination (p-value threshold = 0.05)\n")
        f.write(f"Number of sliding windows: {len(results_df)}\n\n")
        
        f.write("SLIDING WINDOW CONFIGURATION:\n")
        f.write(f"  Initial training: 2013-01-01 to 2014-07-31\n")
        f.write(f"  Each window shifts forward by 14 days\n")
        f.write(f"  Training window length: 1 year 7 months (577 days)\n")
        f.write(f"  Test window length: 14 days\n\n")
        
        f.write("PERFORMANCE METRICS:\n")
        f.write(f"  Average RMSE: {results_df['rmse'].mean():.2f}\n")
        f.write(f"  Std RMSE: {results_df['rmse'].std():.2f}\n")
        f.write(f"  Min RMSE: {results_df['rmse'].min():.2f}\n")
        f.write(f"  Max RMSE: {results_df['rmse'].max():.2f}\n")
        f.write(f"  Average MAE: {results_df['mae'].mean():.2f}\n\n")
        
        f.write("=" * 60 + "\n")
        f.write("BLOCK-BY-BLOCK MODEL SUMMARY\n")
        f.write("=" * 60 + "\n")
        f.write(f"{'Block':<8} {'SARIMA Model':<30} {'Exogenous Variables'}\n")
        f.write("-" * 90 + "\n")
        
        for _, row in results_df.sort_values('split_id').iterrows():
            block_num = row['split_id']
            model_order = f"{row['order']}{row['seasonal_order']}"
            exog_vars = row['selected_cols'] if row['selected_cols'] else ['None']
            f.write(f"Block {block_num:<3}   {model_order:<30} {exog_vars}\n")
        
        f.write("-" * 90 + "\n\n")
        
        f.write("VARIABLE SELECTION FREQUENCY:\n")
        for var, count in sorted(var_selection_count.items(), key=lambda x: x[1], reverse=True):
            freq = (count / len(results_df)) * 100
            f.write(f"  * {var:20s}: {count:2d}/{len(results_df)} ({freq:5.1f}%)\n")
        
        f.write(f"\nCONSISTENTLY SELECTED VARIABLES (>50% of splits):\n")
        if consistent_vars:
            for var in consistent_vars:
                freq = (var_selection_count[var] / len(results_df)) * 100
                f.write(f"  - {var} ({freq:.1f}%)\n")
        else:
            f.write(f"  No variables selected consistently\n")
        
        f.write(f"\n{'='*60}\n")
        f.write(f"BEST MODEL ACCORDING TO SELECTION CRITERIA\n")
        f.write(f"{'='*60}\n")
        f.write(f"\nSelection Criteria Applied:\n")
        f.write(f"  1. Primary: Lowest Average RMSE among models with ≥2 appearances\n")
        f.write(f"  2. Secondary: Lowest Std RMSE (stability)\n\n")
        
        # Convert order tuple to string for display
        order_str = str(best_model['order'])
        seasonal_str = str(best_model['seasonal_order'])
        
        f.write(f"SELECTED MODEL:\n")
        f.write(f"  ARIMA Order: {order_str}{seasonal_str}\n")
        f.write(f"  Exogenous Variables: {best_model['exog_vars'] if best_model['exog_vars'] else 'None'}\n")
        f.write(f"  Average RMSE: {best_model['avg_rmse']:.2f}\n")
        f.write(f"  Std RMSE (stability): {best_model['std_rmse']:.2f}\n")
        f.write(f"  Frequency: {best_model['frequency']}/{len(results_df)} splits ({best_model['frequency_pct']:.1f}%)\n")
        f.write(f"  RMSE Range: {best_model['min_rmse']:.2f} - {best_model['max_rmse']:.2f}\n")
        f.write(f"  Appears in splits: {best_model['split_ids']}\n\n")
        
        if best_model['frequency'] > 1:
            f.write(f"  Individual RMSEs: {[round(r, 2) for r in best_model['rmse_values']]}\n\n")
        
        f.write(f"MODEL FORMULATION:\n")
        f.write(f"  SARIMAX({best_model['order']}{best_model['seasonal_order']}) with exogenous variables:\n")
        if best_model['exog_vars']:
            for var in best_model['exog_vars']:
                f.write(f"    * {var}\n")
        else:
            f.write(f"    * No exogenous variables\n")
        
        f.write(f"\nPERFORMANCE METRICS (across {len(results_df)} sliding windows):\n")
        f.write(f"  * Average RMSE: {best_model['avg_rmse']:.2f}\n")
        f.write(f"  * RMSE Stability: +/-{best_model['std_rmse']:.2f}\n")
        f.write(f"  * Best RMSE: {best_model['min_rmse']:.2f}\n")
        f.write(f"  * Worst RMSE: {best_model['max_rmse']:.2f}\n\n")
        
        f.write(f"\n{'='*60}\n")
        f.write(f"END OF MODEL SELECTION REPORT\n")
        f.write(f"{'='*60}\n")
    
    print(f"\nResults saved:")
    print(f"  * Detailed results: {results_file}")
    print(f"  * Summary with model specification: {summary_file}")
    
    return results_file, summary_file


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def run_exog_selection_for_store(
    df: pd.DataFrame,
    store_id: int = 1,
    initial_train_start='2013-01-01',
    initial_train_end='2014-07-31',
    horizon_days=14,
    n_splits=26,
    seasonal_on=True,
    m=7,
    p_thresh=0.05,
    auto_arima_trace=False
):
    """
    Main function with CORRECT sliding window implementation.
    """
    
    print("\n" + "=" * 80)
    print("MASTER THESIS - TASK 1: EXOGENOUS VARIABLE SELECTION")
    print(f"Store ID: {store_id}")
    print("=" * 80)
    print("\nIMPLEMENTING CORRECT SLIDING WINDOW:")
    print("  * BOTH training AND test windows slide forward by 14 days each block")
    print("  * Training window always 1 year 7 months")
    print("  * 26 blocks from 2014-08-01 to 2015-07-31")
    print("\nYOUR EXACT EXOG LOGIC:")
    print("  * Using ALL available exogenous variables initially")
    print("  * Backward elimination based on p-values (statistical significance)")
    print("  * Selection done PER SPLIT")
    
    results_df, splits = run_exog_selection_sliding_window(
        df=df,
        store_id=store_id,
        initial_train_start=initial_train_start,
        initial_train_end=initial_train_end,
        horizon_days=horizon_days,
        n_splits=n_splits,
        seasonal_on=seasonal_on,
        m=m,
        p_thresh=p_thresh,
        auto_arima_trace=auto_arima_trace
    )
    
    if results_df is None or len(results_df) == 0:
        print("\n*** Exogenous selection failed. ***")
        return None
    
    var_selection_count, consistent_vars = analyze_selection_frequency(results_df)
    
    generate_stability_report(results_df)
    
    block_configs = print_block_summary(results_df)
    
    best_model, model_rankings = select_best_model_based_on_criteria(results_df)
    
    save_exog_selection_results(results_df, store_id, var_selection_count, consistent_vars, best_model, block_configs)
    
    print("\n" + "=" * 80)
    print("TASK 1 COMPLETED SUCCESSFULLY!")
    print("=" * 80)
    print("\nFINAL OUTPUT:")
    print(f"  * Selection method: Backward elimination (p <= {p_thresh})")
    print(f"  * Number of successful splits: {len(results_df)}/{n_splits}")
    print(f"  * Average RMSE: {results_df['rmse'].mean():.2f}")
    print(f"  * Std RMSE: {results_df['rmse'].std():.2f}")
    
    print(f"\n  * BEST MODEL ACCORDING TO CRITERIA:")
    if best_model['exog_vars']:
        exog_str = '[' + ', '.join([f"'{v}'" for v in best_model['exog_vars']]) + ']'
        if len(exog_str) > 60:
            print(f"    - Exogenous variables: {exog_str[:60]}...")
            print(f"      Full list: {exog_str}")
        else:
            print(f"    - Exogenous variables: {exog_str}")
    else:
        print(f"    - Exogenous variables: None")
    print(f"    - Average RMSE: {best_model['avg_rmse']:.2f}")
    print(f"    - Std RMSE: {best_model['std_rmse']:.2f}")
    print(f"    - Frequency: {best_model['frequency']}/{len(results_df)} splits ({best_model['frequency_pct']:.1f}%)")
    
    if consistent_vars:
        print(f"\n  * CONSISTENTLY SELECTED VARIABLES (>50% of splits):")
        for var in consistent_vars:
            count = var_selection_count[var]
            freq = (count / len(results_df)) * 100
            print(f"    - {var}: {count}/{len(results_df)} times ({freq:.1f}%)")
        print(f"\n  -> RECOMMENDED MODEL: SARIMAX with {consistent_vars}")
    
    return {
        'store_id': store_id,
        'results_df': results_df,
        'splits': splits,
        'var_selection_count': var_selection_count,
        'consistent_vars': consistent_vars,
        'best_model': best_model,
        'model_rankings': model_rankings,
        'block_configs': block_configs,
        'avg_rmse': results_df['rmse'].mean(),
        'std_rmse': results_df['rmse'].std()
    }


# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    
    print("\n" + "=" * 80)
    print("VERIFYING REAL DATA BEFORE ANALYSIS")
    print("=" * 80)
    
    print(f"Data shape: {df.shape}")
    print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"Number of unique stores: {df['Store'].nunique()}")
    print(f"\nColumns in dataframe:")
    print(df.columns.tolist())
    
    required_cols = ['Store', 'Date', 'Sales', 'Open', 'Promo', 'SchoolHoliday', 
                     'StateHoliday', 'DayOfWeek', 'IsPromoMonth']
    
    print("\nChecking required columns:")
    all_good = True
    for col in required_cols:
        if col in df.columns:
            print(f"  * {col}")
        else:
            print(f"  X {col} - MISSING!")
            all_good = False
    
    if not all_good:
        print("\n*** Missing required columns. Please check your data loading code. ***")
        exit()
    
    p_thresh = 0.05
    STORE_ID = 1
    
    print(f"\n{'-'*50}")
    print(f"Analyzing Store {STORE_ID}")
    print('-'*50)
    
    if STORE_ID not in df['Store'].unique():
        print(f"*** Store {STORE_ID} not found in data! ***")
        print(f"Available stores: {sorted(df['Store'].unique())[:10]}...")
        exit()
    
    print("\n" + "=" * 80)
    print(f"STARTING TASK 1 FOR STORE {STORE_ID}")
    print("=" * 80)
    print("\nCORRECT SLIDING WINDOW CONFIGURATION:")
    print("  * Block 1: Train 2013-01-01 to 2014-07-31 | Test 2014-08-01 to 2014-08-14")
    print("  * Block 2: Train 2013-01-15 to 2014-08-14 | Test 2014-08-15 to 2014-08-28")
    print("  * Block 3: Train 2013-01-29 to 2014-08-28 | Test 2014-08-29 to 2014-09-11")
    print("  * ... and so on for 26 blocks")
    
    results = run_exog_selection_for_store(
        df=df,
        store_id=STORE_ID,
        initial_train_start='2013-01-01',
        initial_train_end='2014-07-31',
        horizon_days=14,
        n_splits=26,
        seasonal_on=True,
        m=7,
        p_thresh=p_thresh,
        auto_arima_trace=True
    )
    
    if results:
        print(f"\n{'='*80}")
        print(f"TASK 1 COMPLETE - FINAL ANSWER FOR STORE {STORE_ID}")
        print(f"{'='*80}")
        
        best = results['best_model']
        print(f"\nBEST MODEL SELECTED (based on lowest Avg RMSE):")
        
        # Convert order tuples to strings for display
        order_str = str(best['order'])
        seasonal_str = str(best['seasonal_order'])
        
        if best['exog_vars']:
            exog_str = '[' + ', '.join([f"'{v}'" for v in best['exog_vars']]) + ']'
            wrapped = textwrap.wrap(exog_str, width=70, subsequent_indent=' ' * 19)
            print(f"  ARIMA Order: {order_str}{seasonal_str}")
            print(f"  Exogenous Variables: {wrapped[0]}")
            for line in wrapped[1:]:
                print(f"                     {line}")
        else:
            print(f"  ARIMA Order: {order_str}{seasonal_str}")
            print(f"  Exogenous Variables: None")
        
        print(f"  Average RMSE: {best['avg_rmse']:.2f}")
        print(f"  Std RMSE (stability): {best['std_rmse']:.2f}")
        print(f"  Frequency: {best['frequency']}/{len(results['results_df'])} splits ({best['frequency_pct']:.1f}%)")
        
        print(f"\nRECOMMENDED MODEL SPECIFICATION for Store {STORE_ID}:")
        print(f"  SARIMAX({best['order']}{best['seasonal_order']}) with exogenous variables:")
        if best['exog_vars']:
            for var in best['exog_vars']:
                count = results['var_selection_count'].get(var, 0)
                freq = (count / len(results['results_df'])) * 100
                print(f"    * {var} (selected {count}/{len(results['results_df'])} times, {freq:.1f}%)")
        else:
            print(f"    * No exogenous variables")
        
        print(f"\n  This model was validated across {len(results['results_df'])} sliding windows")
        print(f"  Average RMSE: {best['avg_rmse']:.2f}")
        print(f"  RMSE stability: +/-{best['std_rmse']:.2f}")
        
        print(f"\nThis recommendation is based on:")
        print(f"  * Backward elimination with p-value threshold = {p_thresh}")
        print(f"  * {len(results['results_df'])} successful sliding windows out of 26")
        print(f"  * Primary criterion: Lowest Average RMSE ({best['avg_rmse']:.2f})")
        print(f"  * Secondary criterion: Lowest Std RMSE ({best['std_rmse']:.2f})")

## store 1 rolling window for stability

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from collections import Counter
import textwrap

import pmdarima as pm
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =============================================================================
# IMPROVED UTILITY FUNCTIONS
# =============================================================================

def enforce_non_negative(predictions, open_status=None, index=None):
    """
    Ensure predictions are non-negative.
    If store is closed (Open=0), sales should be 0.
    Returns a pandas Series if index is provided, otherwise numpy array.
    """
    predictions = np.array(predictions)
    predictions = np.maximum(predictions, 0)

    if open_status is not None:
        open_status = np.array(open_status)
        predictions[open_status == 0] = 0

    if index is not None:
        return pd.Series(predictions, index=index)
    return predictions


def calculate_mape_safe(actual, predicted):
    """
    Safe MAPE calculation that handles zero values.
    Returns NaN if all actual values are zero.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)

    mask = actual != 0
    if mask.sum() == 0:
        return np.nan

    predicted = enforce_non_negative(predicted)
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


def calculate_smape(actual, predicted):
    """
    Symmetric MAPE - handles zero values better than MAPE.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)

    predicted = enforce_non_negative(predicted)

    denominator = (np.abs(actual) + np.abs(predicted))
    mask = denominator != 0
    if mask.sum() == 0:
        return np.nan

    return 100 / len(actual) * np.sum(2 * np.abs(predicted[mask] - actual[mask]) / denominator[mask])


def calculate_all_metrics(y_true, y_pred, X_test=None):
    """
    Calculate all performance metrics safely.
    """
    metrics = {}

    if X_test is not None and 'Open' in X_test.columns:
        y_pred_clean = enforce_non_negative(
            y_pred, X_test['Open'],
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )
    else:
        y_pred_clean = enforce_non_negative(
            y_pred,
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )

    metrics['rmse'] = float(np.sqrt(mean_squared_error(y_true, y_pred_clean)))
    metrics['mae'] = float(mean_absolute_error(y_true, y_pred_clean))

    metrics['mape'] = calculate_mape_safe(y_true, y_pred_clean)
    metrics['smape'] = calculate_smape(y_true, y_pred_clean)

    metrics['bias'] = float(np.mean(y_pred_clean - y_true))
    metrics['std_error'] = float(np.std(y_pred_clean - y_true))

    ss_res = np.sum((y_true - y_pred_clean) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    metrics['r2_adj'] = float(1 - (ss_res / ss_tot)) if ss_tot > 0 else np.nan

    return metrics, y_pred_clean


# =============================================================================
# DATA PREPARATION
# =============================================================================

def prepare_store_data(
    df: pd.DataFrame,
    store_id: int,
    exogenous_columns=("Open", "Promo", "SchoolHoliday", "StateHoliday", "IsPromoMonth", "DayOfWeek"),
):
    s = df[df["Store"] == store_id].copy()
    s["Date"] = pd.to_datetime(s["Date"], dayfirst=True)
    s = s.sort_values("Date").set_index("Date")

    y = s["Sales"].astype(float).sort_index()
    X = s[list(exogenous_columns)].copy().loc[y.index]

    X["Open"] = X["Open"].astype(int)
    X["Promo"] = X["Promo"].astype(int)
    X["SchoolHoliday"] = X["SchoolHoliday"].astype(int)
    X["IsPromoMonth"] = X["IsPromoMonth"].astype(int)
    X["StateHoliday"] = (X["StateHoliday"].astype(str) != "0").astype(int)

    dow = pd.get_dummies(X["DayOfWeek"], prefix="DOW", drop_first=True).astype(float)

    X_num = X.drop(columns=["DayOfWeek"]).astype(float)
    Xcand = pd.concat([X_num, dow], axis=1).astype(float)

    return y, Xcand


def select_orders_with_auto_arima(y_tr, X_tr, seasonal_on=True, m=7, trace=False):
    if seasonal_on:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=True,
            m=m,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            D=None,
            max_p=3, max_q=3,
            max_P=2, max_Q=2,
            max_d=0,
            max_D=1,
            start_p=1,
            start_q=1,
            start_P=0,
            start_Q=0,
            test='adf',
            seasonal_test='ocsb',
            n_fits=20,
            with_intercept=True
        )
        return sel.order, sel.seasonal_order, float(sel.aic()), float(sel.bic())
    else:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=False,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            max_p=3, max_q=3,
            max_d=0,
            start_p=1,
            start_q=1,
            test='adf',
            n_fits=15,
            with_intercept=True
        )
        return sel.order, (0, 0, 0, 0), float(sel.aic()), float(sel.bic())


def backward_elimination_exog(y_tr, X_tr, order, seasonal_order, p_thresh=0.05, max_iter=50):
    cols = list(X_tr.columns)

    if len(cols) == 0:
        mod = sm.tsa.SARIMAX(
            y_tr, exog=None, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        return [], res

    for _ in range(max_iter):
        X_use = X_tr[cols] if cols else None

        mod = sm.tsa.SARIMAX(
            y_tr, exog=X_use, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        if not cols:
            return [], res

        pvals = res.pvalues
        p_exog = pvals[[c for c in pvals.index if c in cols]]

        if len(p_exog) == 0:
            return cols, res

        worst_col = p_exog.idxmax()
        worst_p = float(p_exog.max())

        if worst_p <= p_thresh:
            return cols, res

        cols.remove(worst_col)

        if not cols:
            mod2 = sm.tsa.SARIMAX(
                y_tr, exog=None, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False
            )
            res2 = mod2.fit(disp=False)
            return [], res2

    return cols, res


# =============================================================================
# CORRECT SLIDING WINDOW VALIDATION
# =============================================================================

def create_sliding_splits(
    initial_train_start='2013-01-01',
    initial_train_end='2014-07-31',
    horizon_days=14,
    n_splits=26
):
    """
    Create sliding window splits where BOTH training and test windows slide forward.
    """
    initial_train_start = pd.to_datetime(initial_train_start)
    initial_train_end = pd.to_datetime(initial_train_end)

    splits = []

    for i in range(n_splits):
        shift_days = i * horizon_days
        train_start = initial_train_start + pd.Timedelta(days=shift_days)
        train_end = initial_train_end + pd.Timedelta(days=shift_days)
        test_start = train_end + pd.Timedelta(days=1)
        test_end = test_start + pd.Timedelta(days=horizon_days - 1)

        splits.append({
            'split_id': i + 1,
            'train_start': train_start,
            'train_end': train_end,
            'test_start': test_start,
            'test_end': test_end,
            'shift_days': shift_days
        })

    return splits


def print_splits_summary(splits):
    """
    Print a nice summary of all splits.
    """
    print(f"\n{'='*100}")
    print(f"SLIDING WINDOW SPLITS SUMMARY")
    print(f"{'='*100}")
    print(f"{'Block':<6} {'Training Period':<35} {'Test Period':<35} {'Shift'}")
    print(f"{'-'*100}")

    for split in splits[:5]:
        train_period = f"{split['train_start'].date()} to {split['train_end'].date()}"
        test_period = f"{split['test_start'].date()} to {split['test_end'].date()}"
        print(f"{split['split_id']:<6} {train_period:<35} {test_period:<35} {split['shift_days']} days")

    if len(splits) > 10:
        print("...")

    for split in splits[-5:]:
        train_period = f"{split['train_start'].date()} to {split['train_end'].date()}"
        test_period = f"{split['test_start'].date()} to {split['test_end'].date()}"
        print(f"{split['split_id']:<6} {train_period:<35} {test_period:<35} {split['shift_days']} days")

    print(f"{'='*100}")


def evaluate_single_split(y_full, X_full, split, seasonal_on=True, m=7,
                          p_thresh=0.05, auto_arima_trace=False):
    """
    Evaluate a single split.
    """
    train_mask = (y_full.index >= split['train_start']) & (y_full.index <= split['train_end'])
    test_mask = (y_full.index >= split['test_start']) & (y_full.index <= split['test_end'])

    y_train = y_full[train_mask]
    y_test = y_full[test_mask]

    if len(y_test) == 0:
        print(f"    Warning: No test data available for split {split['split_id']}")
        return None

    X_train = X_full.loc[y_train.index]
    X_test = X_full.loc[y_test.index]

    try:
        order, seas, aic, bic = select_orders_with_auto_arima(
            y_train, X_train, seasonal_on=seasonal_on, m=m, trace=auto_arima_trace
        )

        selected_cols, _ = backward_elimination_exog(
            y_train, X_train, order, seas, p_thresh=p_thresh
        )

        # =========================
        # FIX 1: keep original column order (do NOT rely on elimination order)
        # =========================
        selected_cols = [c for c in X_train.columns if c in (selected_cols or [])]

        if selected_cols:
            X_tr_final = X_train[selected_cols]
            X_te_final = X_test[selected_cols]
        else:
            X_tr_final = None
            X_te_final = None
            selected_cols = []

        mod = sm.tsa.SARIMAX(
            y_train, exog=X_tr_final, order=order, seasonal_order=seas,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        pred_raw = res.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_te_final)
        pred = enforce_non_negative(
            pred_raw,
            X_test['Open'] if 'Open' in X_test.columns else None,
            index=pred_raw.index
        )

        rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
        mae = float(mean_absolute_error(y_test, pred))

        metrics, _ = calculate_all_metrics(y_test, pred, X_test)

        return {
            'split_id': split['split_id'],
            'train_start': split['train_start'],
            'train_end': split['train_end'],
            'test_start': split['test_start'],
            'test_end': split['test_end'],
            'rmse': rmse,
            'mae': mae,
            'mape': metrics['mape'],
            'smape': metrics['smape'],
            'order': order,
            'seasonal_order': seas,
            'selected_cols': selected_cols,
            'aic': aic,
            'bic': bic,
            'n_train': len(y_train),
            'n_test': len(y_test)
        }

    except Exception as e:
        print(f"    Error in split {split['split_id']}: {str(e)[:100]}...")
        return None


def run_exog_selection_sliding_window(
    df: pd.DataFrame,
    store_id: int,
    initial_train_start='2013-01-01',
    initial_train_end='2014-07-31',
    horizon_days=14,
    n_splits=26,
    seasonal_on=True,
    m=7,
    p_thresh=0.05,
    auto_arima_trace=False
):
    """
    Main function that implements CORRECT sliding window.
    """

    print(f"\n{'='*80}")
    print(f"EXOGENOUS VARIABLE SELECTION - STORE {store_id}")
    print(f"{'='*80}")
    print(f"Initial training window: {initial_train_start} to {initial_train_end} (1 year 7 months)")
    print(f"Forecast horizon: {horizon_days} days")
    print(f"Number of test blocks: {n_splits}")
    print(f"\n{'='*80}")
    print(f"SLIDING WINDOW MECHANISM:")
    print(f"{'='*80}")
    print(f"  * BOTH training AND test windows slide forward by {horizon_days} days each block")
    print(f"  * Training window always 1 year 7 months (577 days)")
    print(f"  * Test window always 14 days")
    print(f"  * 26 blocks covering from 2014-08-01 to 2015-07-31")

    y_full, X_full = prepare_store_data(df, store_id)
    print(f"\nData loaded: {len(y_full)} days from {y_full.index[0].date()} to {y_full.index[-1].date()}")
    print(f"Total exogenous variables available: {len(X_full.columns)}")
    print(f"Variables: {list(X_full.columns)}")

    splits = create_sliding_splits(
        initial_train_start=initial_train_start,
        initial_train_end=initial_train_end,
        horizon_days=horizon_days,
        n_splits=n_splits
    )

    print_splits_summary(splits)

    all_results = []

    print(f"\n{'='*80}")
    print(f"EVALUATING EACH SPLIT")
    print(f"{'='*80}")

    for split in splits:
        print(f"\nSplit {split['split_id']}:")
        print(f"  Train: {split['train_start'].date()} to {split['train_end'].date()}")
        print(f"  Test:  {split['test_start'].date()} to {split['test_end'].date()}")

        result = evaluate_single_split(
            y_full=y_full,
            X_full=X_full,
            split=split,
            seasonal_on=seasonal_on,
            m=m,
            p_thresh=p_thresh,
            auto_arima_trace=auto_arima_trace
        )

        if result:
            all_results.append(result)
            print(f"  * RMSE: {result['rmse']:.2f}")
            print(f"    Selected exog: {result['selected_cols'] if result['selected_cols'] else 'None'}")
            print(f"    Model order: {result['order']}{result['seasonal_order']}")

    if not all_results:
        print("\n*** No successful splits! ***")
        return None

    results_df = pd.DataFrame(all_results)

    return results_df, splits


def analyze_selection_frequency(results_df):
    """
    Analyze how often each exogenous variable is selected across splits.
    """
    var_selection_count = {}
    total_splits = len(results_df)

    all_vars = []
    for selected in results_df['selected_cols']:
        if selected:
            all_vars.extend(selected)
    all_vars = sorted(set(all_vars))

    for var in all_vars:
        count = 0
        for selected in results_df['selected_cols']:
            if selected and var in selected:
                count += 1
        var_selection_count[var] = count

    print(f"\n{'='*80}")
    print(f"VARIABLE SELECTION FREQUENCY ANALYSIS")
    print(f"{'='*80}")
    print(f"Across {total_splits} splits:")
    print()
    print(f"{'Variable':<20} {'Times Selected':<15} {'Frequency':<10}")
    print("-" * 45)

    for var, count in sorted(var_selection_count.items(), key=lambda x: x[1], reverse=True):
        freq = (count / total_splits) * 100
        print(f"  * {var:<18} {count:>5}/{total_splits:<9} {freq:>6.1f}%")

    consistent_vars = [var for var, count in var_selection_count.items()
                      if count / total_splits > 0.5]

    print(f"\nConsistently selected variables (>50% of splits):")
    if consistent_vars:
        for var in consistent_vars:
            freq = (var_selection_count[var] / total_splits) * 100
            print(f"  - {var} ({freq:.1f}%)")
    else:
        print(f"  No variables selected consistently")

    return var_selection_count, consistent_vars


def generate_stability_report(results_df):
    """
    Generate detailed stability report.
    """
    rmse_values = results_df['rmse'].values

    print(f"\n{'='*80}")
    print(f"STABILITY REPORT")
    print(f"{'='*80}")
    print(f"\nPerformance across {len(results_df)} splits:")
    print(f"  * Average RMSE: {np.mean(rmse_values):.2f}")
    print(f"  * Std Deviation: {np.std(rmse_values):.2f}")
    print(f"  * Coefficient of Variation: {(np.std(rmse_values)/np.mean(rmse_values))*100:.1f}%")
    print(f"  * Min RMSE: {np.min(rmse_values):.2f}")
    print(f"  * Max RMSE: {np.max(rmse_values):.2f}")
    print(f"  * Range: {np.max(rmse_values) - np.min(rmse_values):.2f}")

    print(f"\nSplit-by-split performance:")
    print("-" * 100)
    print(f"{'Split':<6} {'Test Period':<25} {'RMSE':<10} {'MAE':<10} {'MAPE':<8} {'Selected Exog'}")
    print("-" * 100)

    for _, row in results_df.sort_values('split_id').iterrows():
        test_period = f"{row['test_start'].date()} to {row['test_end'].date()}"
        selected = str(row['selected_cols'])[:30] if row['selected_cols'] else 'None'
        mape = f"{row['mape']:.1f}%" if not np.isnan(row['mape']) else "N/A"
        print(f"{row['split_id']:<6} {test_period:<25} {row['rmse']:<10.2f} {row['mae']:<10.2f} {mape:<8} {selected}")

    return results_df


def select_best_model_based_on_criteria(results_df):
    """
    Select the best model based on:
    1. Lowest Avg RMSE among models that appear in at least 2 splits
    2. If no model appears twice, then take the best single appearance
    """

    print(f"\n{'='*80}")
    print(f"MODEL SELECTION BASED ON CRITERIA")
    print(f"{'='*80}")

    # =========================
    # FIX 2: model_key must preserve exog ORDER (no sorting)
    # =========================
    results_df = results_df.copy()
    results_df['model_key'] = results_df.apply(
        lambda row: (
            tuple(row['selected_cols']) if row['selected_cols'] else tuple(),
            row['order'],
            row['seasonal_order']
        ), axis=1
    )

    model_performance = []

    for model_key in results_df['model_key'].unique():
        subset = results_df[results_df['model_key'] == model_key]

        exog_tuple, order, seasonal_order = model_key
        exog_list = list(exog_tuple) if exog_tuple else []

        avg_rmse = subset['rmse'].mean()
        std_rmse = subset['rmse'].std()
        frequency = len(subset)
        frequency_pct = (frequency / len(results_df)) * 100

        split_ids = sorted(subset['split_id'].tolist())
        rmse_values = subset['rmse'].tolist()

        model_performance.append({
            'exog_vars': exog_list,
            'order': order,
            'seasonal_order': seasonal_order,
            'model_str': f"{order}{seasonal_order}",
            'avg_rmse': avg_rmse,
            'std_rmse': std_rmse,
            'frequency': frequency,
            'frequency_pct': frequency_pct,
            'min_rmse': subset['rmse'].min(),
            'max_rmse': subset['rmse'].max(),
            'split_ids': split_ids,
            'rmse_values': rmse_values
        })

    model_df = pd.DataFrame(model_performance)
    model_df = model_df.sort_values(['avg_rmse', 'std_rmse'])

    reliable_models = model_df[model_df['frequency'] >= 2]
    if len(reliable_models) > 0:
        best_model = reliable_models.iloc[0]
        print(f"\nNOTE: Selected from models appearing in at least 2 splits for reliability")
    else:
        best_model = model_df.iloc[0]
        print(f"\nNOTE: No model appears in multiple splits - selecting best single appearance")

    print(f"\n{'='*80}")
    print(f"BEST MODEL SELECTED BASED ON CRITERIA")
    print(f"{'='*80}")
    print(f"\nSelection Criteria:")
    print(f"  1. Primary: Lowest Average RMSE among models with ≥2 appearances")
    print(f"  2. Secondary: Lowest Std RMSE (stability)")
    print(f"\nSelected Model:")

    order_str = str(best_model['order'])
    seasonal_str = str(best_model['seasonal_order'])

    if best_model['exog_vars']:
        exog_str = '[' + ', '.join([f"'{v}'" for v in best_model['exog_vars']]) + ']'
        wrapped_exog = textwrap.wrap(exog_str, width=70, subsequent_indent=' ' * 19)
        print(f"  ARIMA Order: {order_str}{seasonal_str}")
        print(f"  Exogenous Variables: {wrapped_exog[0]}")
        for line in wrapped_exog[1:]:
            print(f"                     {line}")
    else:
        print(f"  ARIMA Order: {order_str}{seasonal_str}")
        print(f"  Exogenous Variables: None")

    print(f"  Average RMSE: {best_model['avg_rmse']:.2f}")
    print(f"  Std RMSE (stability): {best_model['std_rmse']:.2f}")
    print(f"  Frequency: {best_model['frequency']}/{len(results_df)} splits ({best_model['frequency_pct']:.1f}%)")
    print(f"  RMSE Range: {best_model['min_rmse']:.2f} - {best_model['max_rmse']:.2f}")
    print(f"  Appears in splits: {best_model['split_ids']}")

    if best_model['frequency'] > 1:
        print(f"  Individual RMSEs: {[round(r, 2) for r in best_model['rmse_values']]}")

    print(f"\nTOP 3 RECOMMENDED MODELS:")
    print("-" * 100)
    print(f"{'Rank':<6} {'ARIMA Order':<20} {'Avg RMSE':<10} {'Std':<8} {'Freq':<6} {'Exogenous Variables'}")
    print("-" * 100)

    for rank in range(min(3, len(model_df))):
        row = model_df.iloc[rank]
        order_display = f"{row['order']}{row['seasonal_order']}"
        exog_display = str(row['exog_vars'])[:40] + "..." if len(str(row['exog_vars'])) > 40 else str(row['exog_vars'])
        print(f"{rank+1:<6} {order_display:<20} {row['avg_rmse']:<10.2f} {row['std_rmse']:<8.2f} {row['frequency']}/{len(results_df):<6} {exog_display}")

    return best_model, model_df


def print_block_summary(results_df):
    """
    Print a summary of each block showing SARIMA model and FULL exogenous variables.
    """
    print(f"\n{'='*80}")
    print(f"BLOCK-BY-BLOCK MODEL SUMMARY")
    print(f"{'='*80}")

    all_configs = []

    for _, row in results_df.sort_values('split_id').iterrows():
        block_num = row['split_id']
        model_order = f"{row['order']}{row['seasonal_order']}"

        # display None, but store empty list
        exog_vars = row['selected_cols'] if row['selected_cols'] else []
        config = {
            'block': block_num,
            'model': model_order,
            'exog': exog_vars,
            'exog_tuple': tuple(exog_vars)  # keep ORDER
        }
        all_configs.append(config)

    print(f"\n{'='*100}")
    print(f"{'Block':<8} {'SARIMA Model':<30} {'Exogenous Variables'}")
    print(f"{'='*100}")

    for config in all_configs:
        block_num = config['block']
        model_order = config['model']
        exog_vars = config['exog'] if config['exog'] else ['None']
        print(f"Block {block_num:<3}   {model_order:<30} {exog_vars}")

    print(f"{'='*100}")

    print(f"\nCONFIGURATION FREQUENCY:")
    print(f"{'-'*100}")

    config_summary = {}
    for config in all_configs:
        key = (config['model'], config['exog_tuple'])
        if key not in config_summary:
            config_summary[key] = {
                'model': config['model'],
                'exog': config['exog'],
                'blocks': [],
                'count': 0
            }
        config_summary[key]['blocks'].append(config['block'])
        config_summary[key]['count'] += 1

    for idx, (key, info) in enumerate(sorted(config_summary.items(),
                                              key=lambda x: x[1]['blocks'][0])):
        blocks_str = ', '.join([str(b) for b in info['blocks']])
        exog_str = str(info['exog'] if info['exog'] else ['None'])
        print(f"\nConfig {idx+1}: {info['model']}")
        print(f"  Exog: {exog_str}")
        print(f"  Appears in blocks: {blocks_str} ({info['count']}/{len(results_df)} blocks, {info['count']/len(results_df)*100:.1f}%)")

    print(f"{'-'*100}")

    most_common_key = max(config_summary.items(), key=lambda x: x[1]['count'])[0]
    most_common_info = config_summary[most_common_key]

    print(f"\nMOST COMMON MODEL CONFIGURATION:")
    print(f"  Appeared in {most_common_info['count']}/{len(results_df)} blocks ({most_common_info['count']/len(results_df)*100:.1f}%)")
    print(f"  SARIMA Model: {most_common_info['model']}")
    print(f"  Exogenous Variables: {most_common_info['exog'] if most_common_info['exog'] else ['None']}")
    print(f"  Blocks: {most_common_info['blocks']}")

    print(f"\nTOP 3 MOST COMMON CONFIGURATIONS:")
    print(f"{'-'*100}")
    print(f"{'Rank':<6} {'Frequency':<15} {'SARIMA Model':<30} {'Exogenous Variables'}")
    print(f"{'-'*100}")

    sorted_configs = sorted(config_summary.items(), key=lambda x: x[1]['count'], reverse=True)

    for rank, (key, info) in enumerate(sorted_configs[:3], 1):
        pct = (info['count'] / len(results_df)) * 100
        exog_str = str(info['exog'] if info['exog'] else ['None'])
        print(f"{rank:<6} {info['count']}/{len(results_df)} ({pct:5.1f}%)   {info['model']:<30} {exog_str}")

    print(f"{'='*100}")

    return all_configs


def save_exog_selection_results(results_df, store_id, var_selection_count, consistent_vars, best_model, block_configs):
    """
    Save all results to files.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    results_file = f"exog_selection_results_store_{store_id}_{timestamp}.csv"

    save_data = []
    for _, row in results_df.iterrows():
        save_data.append({
            'split_id': row['split_id'],
            'train_start': row['train_start'].date(),
            'train_end': row['train_end'].date(),
            'test_start': row['test_start'].date(),
            'test_end': row['test_end'].date(),
            'rmse': row['rmse'],
            'mae': row['mae'],
            'mape': row['mape'],
            'smape': row['smape'],
            'order': str(row['order']),
            'seasonal_order': str(row['seasonal_order']),
            'selected_cols': str(row['selected_cols']),
            'aic': row['aic'],
            'bic': row['bic'],
            'n_train': row['n_train'],
            'n_test': row['n_test']
        })

    save_df = pd.DataFrame(save_data)
    save_df.to_csv(results_file, index=False, encoding='utf-8')

    summary_file = f"exog_selection_summary_store_{store_id}_{timestamp}.txt"
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(f"EXOGENOUS VARIABLE SELECTION RESULTS - STORE {store_id}\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Selection method: Backward elimination (p-value threshold = 0.05)\n")
        f.write(f"Number of sliding windows: {len(results_df)}\n\n")

        f.write("SLIDING WINDOW CONFIGURATION:\n")
        f.write(f"  Initial training: 2013-01-01 to 2014-07-31\n")
        f.write(f"  Each window shifts forward by 14 days\n")
        f.write(f"  Training window length: 1 year 7 months (577 days)\n")
        f.write(f"  Test window length: 14 days\n\n")

        f.write("PERFORMANCE METRICS:\n")
        f.write(f"  Average RMSE: {results_df['rmse'].mean():.2f}\n")
        f.write(f"  Std RMSE: {results_df['rmse'].std():.2f}\n")
        f.write(f"  Min RMSE: {results_df['rmse'].min():.2f}\n")
        f.write(f"  Max RMSE: {results_df['rmse'].max():.2f}\n")
        f.write(f"  Average MAE: {results_df['mae'].mean():.2f}\n\n")

        f.write("=" * 60 + "\n")
        f.write("BLOCK-BY-BLOCK MODEL SUMMARY\n")
        f.write("=" * 60 + "\n")
        f.write(f"{'Block':<8} {'SARIMA Model':<30} {'Exogenous Variables'}\n")
        f.write("-" * 90 + "\n")

        for _, row in results_df.sort_values('split_id').iterrows():
            block_num = row['split_id']
            model_order = f"{row['order']}{row['seasonal_order']}"
            exog_vars = row['selected_cols'] if row['selected_cols'] else []
            f.write(f"Block {block_num:<3}   {model_order:<30} {exog_vars if exog_vars else ['None']}\n")

        f.write("-" * 90 + "\n\n")

        f.write("VARIABLE SELECTION FREQUENCY:\n")
        for var, count in sorted(var_selection_count.items(), key=lambda x: x[1], reverse=True):
            freq = (count / len(results_df)) * 100
            f.write(f"  * {var:20s}: {count:2d}/{len(results_df)} ({freq:5.1f}%)\n")

        f.write(f"\nCONSISTENTLY SELECTED VARIABLES (>50% of splits):\n")
        if consistent_vars:
            for var in consistent_vars:
                freq = (var_selection_count[var] / len(results_df)) * 100
                f.write(f"  - {var} ({freq:.1f}%)\n")
        else:
            f.write(f"  No variables selected consistently\n")

        f.write(f"\n{'='*60}\n")
        f.write(f"BEST MODEL ACCORDING TO SELECTION CRITERIA\n")
        f.write(f"{'='*60}\n")
        f.write(f"\nSelection Criteria Applied:\n")
        f.write(f"  1. Primary: Lowest Average RMSE among models with ≥2 appearances\n")
        f.write(f"  2. Secondary: Lowest Std RMSE (stability)\n\n")

        order_str = str(best_model['order'])
        seasonal_str = str(best_model['seasonal_order'])

        f.write(f"SELECTED MODEL:\n")
        f.write(f"  ARIMA Order: {order_str}{seasonal_str}\n")
        f.write(f"  Exogenous Variables: {best_model['exog_vars'] if best_model['exog_vars'] else 'None'}\n")
        f.write(f"  Average RMSE: {best_model['avg_rmse']:.2f}\n")
        f.write(f"  Std RMSE (stability): {best_model['std_rmse']:.2f}\n")
        f.write(f"  Frequency: {best_model['frequency']}/{len(results_df)} splits ({best_model['frequency_pct']:.1f}%)\n")
        f.write(f"  RMSE Range: {best_model['min_rmse']:.2f} - {best_model['max_rmse']:.2f}\n")
        f.write(f"  Appears in splits: {best_model['split_ids']}\n\n")

        if best_model['frequency'] > 1:
            f.write(f"  Individual RMSEs: {[round(r, 2) for r in best_model['rmse_values']]}\n\n")

        f.write(f"MODEL FORMULATION:\n")
        f.write(f"  SARIMAX({best_model['order']}{best_model['seasonal_order']}) with exogenous variables:\n")
        if best_model['exog_vars']:
            for var in best_model['exog_vars']:
                f.write(f"    * {var}\n")
        else:
            f.write(f"    * No exogenous variables\n")

        f.write(f"\nPERFORMANCE METRICS (across {len(results_df)} sliding windows):\n")
        f.write(f"  * Average RMSE: {best_model['avg_rmse']:.2f}\n")
        f.write(f"  * RMSE Stability: +/-{best_model['std_rmse']:.2f}\n")
        f.write(f"  * Best RMSE: {best_model['min_rmse']:.2f}\n")
        f.write(f"  * Worst RMSE: {best_model['max_rmse']:.2f}\n\n")

        f.write(f"\n{'='*60}\n")
        f.write(f"END OF MODEL SELECTION REPORT\n")
        f.write(f"{'='*60}\n")

    print(f"\nResults saved:")
    print(f"  * Detailed results: {results_file}")
    print(f"  * Summary with model specification: {summary_file}")

    return results_file, summary_file


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def run_exog_selection_for_store(
    df: pd.DataFrame,
    store_id: int = 1,
    initial_train_start='2013-01-01',
    initial_train_end='2014-07-31',
    horizon_days=14,
    n_splits=26,
    seasonal_on=True,
    m=7,
    p_thresh=0.05,
    auto_arima_trace=False
):
    """
    Main function with CORRECT sliding window implementation.
    """

    print("\n" + "=" * 80)
    print("MASTER THESIS - TASK 1: EXOGENOUS VARIABLE SELECTION")
    print(f"Store ID: {store_id}")
    print("=" * 80)
    print("\nIMPLEMENTING CORRECT SLIDING WINDOW:")
    print("  * BOTH training AND test windows slide forward by 14 days each block")
    print("  * Training window always 1 year 7 months")
    print("  * 26 blocks from 2014-08-01 to 2015-07-31")
    print("\nYOUR EXACT EXOG LOGIC:")
    print("  * Using ALL available exogenous variables initially")
    print("  * Backward elimination based on p-values (statistical significance)")
    print("  * Selection done PER SPLIT")

    results_df, splits = run_exog_selection_sliding_window(
        df=df,
        store_id=store_id,
        initial_train_start=initial_train_start,
        initial_train_end=initial_train_end,
        horizon_days=horizon_days,
        n_splits=n_splits,
        seasonal_on=seasonal_on,
        m=m,
        p_thresh=p_thresh,
        auto_arima_trace=auto_arima_trace
    )

    if results_df is None or len(results_df) == 0:
        print("\n*** Exogenous selection failed. ***")
        return None

    var_selection_count, consistent_vars = analyze_selection_frequency(results_df)

    generate_stability_report(results_df)

    block_configs = print_block_summary(results_df)

    best_model, model_rankings = select_best_model_based_on_criteria(results_df)

    save_exog_selection_results(results_df, store_id, var_selection_count, consistent_vars, best_model, block_configs)

    print("\n" + "=" * 80)
    print("TASK 1 COMPLETED SUCCESSFULLY!")
    print("=" * 80)
    print("\nFINAL OUTPUT:")
    print(f"  * Selection method: Backward elimination (p <= {p_thresh})")
    print(f"  * Number of successful splits: {len(results_df)}/{n_splits}")
    print(f"  * Average RMSE: {results_df['rmse'].mean():.2f}")
    print(f"  * Std RMSE: {results_df['rmse'].std():.2f}")

    print(f"\n  * BEST MODEL ACCORDING TO CRITERIA:")
    if best_model['exog_vars']:
        exog_str = '[' + ', '.join([f"'{v}'" for v in best_model['exog_vars']]) + ']'
        if len(exog_str) > 60:
            print(f"    - Exogenous variables: {exog_str[:60]}...")
            print(f"      Full list: {exog_str}")
        else:
            print(f"    - Exogenous variables: {exog_str}")
    else:
        print(f"    - Exogenous variables: None")
    print(f"    - Average RMSE: {best_model['avg_rmse']:.2f}")
    print(f"    - Std RMSE: {best_model['std_rmse']:.2f}")
    print(f"    - Frequency: {best_model['frequency']}/{len(results_df)} splits ({best_model['frequency_pct']:.1f}%)")

    if consistent_vars:
        print(f"\n  * CONSISTENTLY SELECTED VARIABLES (>50% of splits):")
        for var in consistent_vars:
            count = var_selection_count[var]
            freq = (count / len(results_df)) * 100
            print(f"    - {var}: {count}/{len(results_df)} times ({freq:.1f}%)")
        print(f"\n  -> RECOMMENDED MODEL: SARIMAX with {consistent_vars}")

    return {
        'store_id': store_id,
        'results_df': results_df,
        'splits': splits,
        'var_selection_count': var_selection_count,
        'consistent_vars': consistent_vars,
        'best_model': best_model,
        'model_rankings': model_rankings,
        'block_configs': block_configs,
        'avg_rmse': results_df['rmse'].mean(),
        'std_rmse': results_df['rmse'].std()
    }


# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":

    print("\n" + "=" * 80)
    print("VERIFYING REAL DATA BEFORE ANALYSIS")
    print("=" * 80)

    print(f"Data shape: {df.shape}")
    print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"Number of unique stores: {df['Store'].nunique()}")
    print(f"\nColumns in dataframe:")
    print(df.columns.tolist())

    required_cols = ['Store', 'Date', 'Sales', 'Open', 'Promo', 'SchoolHoliday',
                     'StateHoliday', 'DayOfWeek', 'IsPromoMonth']

    print("\nChecking required columns:")
    all_good = True
    for col in required_cols:
        if col in df.columns:
            print(f"  * {col}")
        else:
            print(f"  X {col} - MISSING!")
            all_good = False

    if not all_good:
        print("\n*** Missing required columns. Please check your data loading code. ***")
        exit()

    p_thresh = 0.05
    STORE_ID = 1

    print(f"\n{'-'*50}")
    print(f"Analyzing Store {STORE_ID}")
    print('-'*50)

    if STORE_ID not in df['Store'].unique():
        print(f"*** Store {STORE_ID} not found in data! ***")
        print(f"Available stores: {sorted(df['Store'].unique())[:10]}...")
        exit()

    print("\n" + "=" * 80)
    print(f"STARTING TASK 1 FOR STORE {STORE_ID}")
    print("=" * 80)
    print("\nCORRECT SLIDING WINDOW CONFIGURATION:")
    print("  * Block 1: Train 2013-01-01 to 2014-07-31 | Test 2014-08-01 to 2014-08-14")
    print("  * Block 2: Train 2013-01-15 to 2014-08-14 | Test 2014-08-15 to 2014-08-28")
    print("  * Block 3: Train 2013-01-29 to 2014-08-28 | Test 2014-08-29 to 2014-09-11")
    print("  * ... and so on for 26 blocks")

    results = run_exog_selection_for_store(
        df=df,
        store_id=STORE_ID,
        initial_train_start='2013-01-01',
        initial_train_end='2014-07-31',
        horizon_days=14,
        n_splits=26,
        seasonal_on=True,
        m=7,
        p_thresh=p_thresh,
        auto_arima_trace=True
    )

    if results:
        print(f"\n{'='*80}")
        print(f"TASK 1 COMPLETE - FINAL ANSWER FOR STORE {STORE_ID}")
        print(f"{'='*80}")

        best = results['best_model']
        print(f"\nBEST MODEL SELECTED (based on lowest Avg RMSE):")

        order_str = str(best['order'])
        seasonal_str = str(best['seasonal_order'])

        if best['exog_vars']:
            exog_str = '[' + ', '.join([f"'{v}'" for v in best['exog_vars']]) + ']'
            wrapped = textwrap.wrap(exog_str, width=70, subsequent_indent=' ' * 19)
            print(f"  ARIMA Order: {order_str}{seasonal_str}")
            print(f"  Exogenous Variables: {wrapped[0]}")
            for line in wrapped[1:]:
                print(f"                     {line}")
        else:
            print(f"  ARIMA Order: {order_str}{seasonal_str}")
            print(f"  Exogenous Variables: None")

        print(f"  Average RMSE: {best['avg_rmse']:.2f}")
        print(f"  Std RMSE (stability): {best['std_rmse']:.2f}")
        print(f"  Frequency: {best['frequency']}/{len(results['results_df'])} splits ({best['frequency_pct']:.1f}%)")

        print(f"\nRECOMMENDED MODEL SPECIFICATION for Store {STORE_ID}:")
        print(f"  SARIMAX({best['order']}{best['seasonal_order']}) with exogenous variables:")
        if best['exog_vars']:
            for var in best['exog_vars']:
                count = results['var_selection_count'].get(var, 0)
                freq = (count / len(results['results_df'])) * 100
                print(f"    * {var} (selected {count}/{len(results['results_df'])} times, {freq:.1f}%)")
        else:
            print(f"    * No exogenous variables")

        print(f"\n  This model was validated across {len(results['results_df'])} sliding windows")
        print(f"  Average RMSE: {best['avg_rmse']:.2f}")
        print(f"  RMSE stability: +/-{best['std_rmse']:.2f}")

        print(f"\nThis recommendation is based on:")
        print(f"  * Backward elimination with p-value threshold = {p_thresh}")
        print(f"  * {len(results['results_df'])} successful sliding windows out of 26")
        print(f"  * Primary criterion: Lowest Average RMSE ({best['avg_rmse']:.2f})")
        print(f"  * Secondary criterion: Lowest Std RMSE ({best['std_rmse']:.2f})")

In [ ]:
# task 2

In [ ]:
## all in one tast 2

In [ ]:
# ============================================================
# TASK 2 — ONE-CELL COMPLETE CODE (prints + RMSE matrix + plots + final 14-day prediction)
# Fixed model from Task 1:
#   SARIMAX (0,0,1)(2,0,2,7)
#   Exog: Open + Promo + DOW_2..DOW_6
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from sklearn.metrics import mean_squared_error


# ============================================================
# FIXED SETTINGS (from Task 1)
# ============================================================
FIXED_ORDER = (0, 0, 1)
FIXED_SEASONAL_ORDER = (2, 0, 2, 7)
FIXED_EXOG_COLS = ["Open", "Promo", "DOW_2", "DOW_3", "DOW_4", "DOW_5", "DOW_6"]

HORIZON_DAYS = 14
N_BLOCKS = 26
TRAIN_MONTHS_LIST = list(range(1, 13))  # 1..12


# ============================================================
# UTILITIES
# ============================================================
def enforce_non_negative(predictions, open_status=None, index=None):
    preds = np.array(predictions, dtype=float)
    preds = np.maximum(preds, 0)

    if open_status is not None:
        open_status = np.array(open_status, dtype=int)
        preds[open_status == 0] = 0

    if index is not None:
        return pd.Series(preds, index=index)
    return preds


def prepare_store_data_fixed_exog(df: pd.DataFrame, store_id: int):
    """
    Same exog construction logic as Task 1, but returns only fixed exogs.
    Creates dummies with drop_first=True.
    Ensures all FIXED_EXOG_COLS exist.
    """
    s = df[df["Store"] == store_id].copy()
    s["Date"] = pd.to_datetime(s["Date"], dayfirst=True, errors="coerce")
    s = s.dropna(subset=["Date"]).sort_values("Date").set_index("Date")

    y = s["Sales"].astype(float).sort_index()

    X = s[["Open", "Promo", "SchoolHoliday", "StateHoliday", "IsPromoMonth", "DayOfWeek"]].copy().loc[y.index]
    X["Open"] = X["Open"].astype(int)
    X["Promo"] = X["Promo"].astype(int)
    X["SchoolHoliday"] = X["SchoolHoliday"].astype(int)
    X["IsPromoMonth"] = X["IsPromoMonth"].astype(int)
    X["StateHoliday"] = (X["StateHoliday"].astype(str) != "0").astype(int)

    dow = pd.get_dummies(X["DayOfWeek"], prefix="DOW", drop_first=True).astype(float)
    X_num = X.drop(columns=["DayOfWeek"]).astype(float)
    X_all = pd.concat([X_num, dow], axis=1).astype(float)

    # Ensure fixed columns exist
    for c in FIXED_EXOG_COLS:
        if c not in X_all.columns:
            X_all[c] = 0.0

    X_fixed = X_all[FIXED_EXOG_COLS].astype(float)
    return y, X_fixed


def create_backward_test_blocks(last_date, horizon_days=14, n_blocks=26):
    """
    Block 1 = most recent 14 days ending at last_date
    Block 2 = previous 14 days, etc. (moving backward)
    """
    last_date = pd.to_datetime(last_date)
    blocks = []
    for t in range(1, n_blocks + 1):
        test_end = last_date - pd.Timedelta(days=(t - 1) * horizon_days)
        test_start = test_end - pd.Timedelta(days=horizon_days - 1)
        blocks.append({"block_id": t, "test_start": test_start, "test_end": test_end})
    return blocks


def fit_predict_rmse_and_model(y, X_fixed, train_start, train_end, test_start, test_end):
    """
    Fit fixed SARIMAX on train, predict test, compute RMSE.
    Returns: rmse, pred_series, fitted_results (res) or (nan, None, None).
    """
    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_start:test_end]

    if len(y_train) < 30 or len(y_test) == 0:
        return np.nan, None, None

    X_train = X_fixed.loc[y_train.index]
    X_test = X_fixed.loc[y_test.index]

    try:
        mod = sm.tsa.SARIMAX(
            y_train,
            exog=X_train,
            order=FIXED_ORDER,
            seasonal_order=FIXED_SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        pred_raw = res.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_test)
        pred = enforce_non_negative(pred_raw, open_status=X_test["Open"], index=pred_raw.index)

        rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
        return rmse, pred, res

    except Exception:
        return np.nan, None, None


def build_future_exog_fixed(store_df, future_dates):
    """
    Future exog for FIXED_EXOG_COLS.

    Open rule:
      - Non-Sunday: Open=1
      - Sunday: Open=1 if historical Sunday open-rate >= 0.5 else 0

    Promo rule:
      - Promo=1 on a weekday if historical promo-rate for that weekday >= 0.5
    """
    store_df = store_df.copy()
    store_df["Date"] = pd.to_datetime(store_df["Date"], dayfirst=True, errors="coerce")
    store_df = store_df.dropna(subset=["Date"])

    sunday = store_df[store_df["DayOfWeek"] == 7]
    sunday_open_rate = float((sunday["Open"] == 1).mean()) if len(sunday) > 0 else 0.0

    promo_by_dow = store_df.groupby("DayOfWeek")["Promo"].mean()

    future_exog = pd.DataFrame(index=future_dates)

    # Open
    open_vals = []
    for d in future_dates:
        if d.weekday() == 6:  # Sunday
            open_vals.append(1 if sunday_open_rate >= 0.5 else 0)
        else:
            open_vals.append(1)
    future_exog["Open"] = open_vals

    # Promo
    promo_vals = []
    for d in future_dates:
        dow = d.weekday() + 1  # 1..7
        hist_rate = float(promo_by_dow.get(dow, 0.0))
        promo_vals.append(1 if hist_rate >= 0.5 else 0)
    future_exog["Promo"] = promo_vals

    # DOW_2..DOW_6
    for k in [2, 3, 4, 5, 6]:
        future_exog[f"DOW_{k}"] = [(1 if (d.weekday() + 1) == k else 0) for d in future_dates]

    for c in FIXED_EXOG_COLS:
        if c not in future_exog.columns:
            future_exog[c] = 0.0

    return future_exog[FIXED_EXOG_COLS].astype(float)


def plot_final_like_example(store_id, y, test_start, test_end, pred_test,
                            future_dates, pred_future, ci_future,
                            history_days=45):
    hist_start = test_start - pd.Timedelta(days=history_days)
    y_hist = y.loc[hist_start:test_end]
    y_test = y.loc[test_start:test_end]

    plt.figure(figsize=(12, 5))

    # History actual
    plt.plot(y_hist.index, y_hist.values, 'b-', linewidth=1.5, label='Actual (History)')

    # Test actual points
    plt.plot(y_test.index, y_test.values, 'bo', markersize=4, label='Test Actual')

    # Test predicted
    plt.plot(pred_test.index, pred_test.values, 'r--', linewidth=2, label='Test Predicted')

    # Future forecast
    plt.plot(future_dates, pred_future.values, 'g-', linewidth=2.5, marker='o', markersize=4, label='Future Forecast')

    # CI
    plt.fill_between(future_dates, ci_future.iloc[:, 0], ci_future.iloc[:, 1], alpha=0.2, label='95% CI')

    # Forecast start
    plt.axvline(x=future_dates[0], color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='Forecast Start')

    plt.xlabel("Date")
    plt.ylabel("Sales")
    plt.title(f"Store {store_id}: Test Prediction (Last 14 Days) + Next 14-Day Forecast")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# ============================================================
# TASK 2 — RUN EVERYTHING
# ============================================================
def run_task2_all_in_one(df: pd.DataFrame, store_id: int):
    print("\n" + "="*90)
    print(f"TASK 2 — TRAINING LENGTH SELECTION (Store {store_id})")
    print("="*90)
    print(f"Fixed model: SARIMAX{FIXED_ORDER}{FIXED_SEASONAL_ORDER}")
    print(f"Fixed exogs: {FIXED_EXOG_COLS}")
    print(f"Horizon: {HORIZON_DAYS} days | Blocks: {N_BLOCKS} | Training lengths: 1..12 months")

    # Load store data
    y, X_fixed = prepare_store_data_fixed_exog(df, store_id)
    last_date = y.index.max()

    print("\nData info:")
    print(f"  Date range: {y.index.min().date()} → {y.index.max().date()}")
    print(f"  Total days: {len(y)}")

    # Build blocks
    blocks = create_backward_test_blocks(last_date, horizon_days=HORIZON_DAYS, n_blocks=N_BLOCKS)
    print("\nEvaluation period (last 26 blocks):")
    print(f"  Oldest test: {blocks[-1]['test_start'].date()} → {blocks[-1]['test_end'].date()}")
    print(f"  Newest test: {blocks[0]['test_start'].date()} → {blocks[0]['test_end'].date()}")

    # RMSE matrix
    rmse_mat = pd.DataFrame(index=TRAIN_MONTHS_LIST, columns=[b["block_id"] for b in blocks], dtype=float)

    # Outer loop
    for b in blocks:
        test_start, test_end = b["test_start"], b["test_end"]
        train_end = test_start - pd.Timedelta(days=1)

        # Inner loop
        for L in TRAIN_MONTHS_LIST:
            train_start = test_start - pd.DateOffset(months=L)
            rmse, _, _ = fit_predict_rmse_and_model(y, X_fixed, train_start, train_end, test_start, test_end)
            rmse_mat.loc[L, b["block_id"]] = rmse

    # Summary per L
    summary = pd.DataFrame({
        "Mean_RMSE": rmse_mat.mean(axis=1, skipna=True),
        "Std_RMSE": rmse_mat.std(axis=1, skipna=True),
        "Valid_Blocks": rmse_mat.notna().sum(axis=1)
    }).sort_index()

    # Choose best L
    best_L = int(summary.sort_values(["Mean_RMSE", "Std_RMSE"]).index[0])

    print("\n" + "-"*90)
    print("RMSE SUMMARY BY TRAINING LENGTH (L months)")
    print("-"*90)
    print(summary.round(2))

    print("\nSelected training length (L*):", best_L,
          f"(Mean RMSE={summary.loc[best_L,'Mean_RMSE']:.2f}, Std={summary.loc[best_L,'Std_RMSE']:.2f})")

    print("\nRMSE matrix shape:", rmse_mat.shape, "(should be 12 x 26)")

    # Plot RMSE vs L (boxplot)
    plt.figure(figsize=(12, 5))
    data_for_box = [rmse_mat.loc[L].dropna().values for L in TRAIN_MONTHS_LIST]
    plt.boxplot(data_for_box, labels=TRAIN_MONTHS_LIST, showfliers=False)
    plt.xlabel("Training length L (months)")
    plt.ylabel("RMSE over 14-day test blocks")
    plt.title(f"Task 2: RMSE vs Training Length (Store {store_id})")
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    # Final step: Fit using L* on most recent TRAIN before most recent TEST (Block 1)
    b1 = blocks[0]
    test_start, test_end = b1["test_start"], b1["test_end"]
    train_end = test_start - pd.Timedelta(days=1)
    train_start = test_start - pd.DateOffset(months=best_L)

    print("\n" + "-"*90)
    print("FINAL EVALUATION ON MOST RECENT 14 DAYS (Block 1)")
    print("-"*90)
    print(f"TRAIN: {train_start.date()} → {train_end.date()}  (L*={best_L} months)")
    print(f"TEST : {test_start.date()} → {test_end.date()}  (14 days)")

    final_rmse, pred_test, fitted_res = fit_predict_rmse_and_model(
        y, X_fixed, train_start, train_end, test_start, test_end
    )

    print(f"Final RMSE (last 14 days): {final_rmse:.2f}")

    # Print last 14-day table
    y_test = y.loc[test_start:test_end]
    final_pred_df = pd.DataFrame({
        "Date": y_test.index,
        "Actual": y_test.values,
        "Predicted": pred_test.values
    })
    final_pred_df["Error"] = final_pred_df["Predicted"] - final_pred_df["Actual"]

    print("\nLast 14 days: Actual vs Predicted")
    print(final_pred_df.to_string(index=False))

    # ALSO do a next-14-day forecast + CI for a plot like your example
    future_dates = pd.date_range(start=test_end + pd.Timedelta(days=1), periods=HORIZON_DAYS, freq="D")
    store_df = df[df["Store"] == store_id].copy()
    future_exog = build_future_exog_fixed(store_df, future_dates)

    fc = fitted_res.get_forecast(steps=HORIZON_DAYS, exog=future_exog)
    pred_future_raw = fc.predicted_mean.copy()
    ci_future = fc.conf_int().copy()

    # If Open=0 => force mean & CI to 0
    closed_mask = (future_exog["Open"].values == 0)
    if closed_mask.any():
        idx = pred_future_raw.index
        pred_future_raw.loc[idx[closed_mask]] = 0.0
        ci_future.iloc[closed_mask, 0] = 0.0
        ci_future.iloc[closed_mask, 1] = 0.0

    pred_future = enforce_non_negative(pred_future_raw, open_status=future_exog["Open"], index=pred_future_raw.index)

    # Plot like your example (test + future + CI)
    plot_final_like_example(
        store_id=store_id,
        y=y,
        test_start=test_start,
        test_end=test_end,
        pred_test=pred_test,
        future_dates=future_dates,
        pred_future=pred_future,
        ci_future=ci_future,
        history_days=45
    )

    return {
        "store_id": store_id,
        "rmse_matrix": rmse_mat,
        "summary": summary,
        "best_L": best_L,
        "final_rmse": final_rmse,
        "final_14day_pred_df": final_pred_df
    }


# ============================================================
# RUN IT (df must already exist)
# ============================================================
STORE_ID = 1  # change if needed
task2_results = run_task2_all_in_one(df, STORE_ID)

# If you want to save outputs (optional):
# task2_results["rmse_matrix"].to_csv(f"task2_rmse_matrix_store_{STORE_ID}.csv")
# task2_results["summary"].to_csv(f"task2_rmse_summary_store_{STORE_ID}.csv")
# task2_results["final_14day_pred_df"].to_csv(f"task2_last14_pred_store_{STORE_ID}.csv", index=False)

In [ ]:
### lets choose this model

In [ ]:
BEST MODEL SELECTED (based on lowest Avg RMSE):
  ARIMA Order: (0, 0, 1)(2, 0, 2, 7)
  Exogenous Variables: ['Open', 'Promo', 'SchoolHoliday', 'DOW_2', 'DOW_3', 'DOW_4', 'DOW_5',
                                        'DOW_6']
  Average RMSE: 399.27
  Std RMSE (stability): 231.98
  Frequency: 2/26 splits (7.7%)

In [ ]:
# ============================================================
# TASK 2 — CLEAN VERSION (NO FUTURE FORECASTING) + DTYPE FIX
# Model:
#   SARIMAX (0,0,1)(2,0,2,7)
# Exog:
#   Open + Promo + SchoolHoliday + DOW_2..DOW_6
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error


# ============================================================
# FIXED SETTINGS
# ============================================================
FIXED_ORDER = (0, 0, 1)
FIXED_SEASONAL_ORDER = (2, 0, 2, 7)

FIXED_EXOG_COLS = ["Open", "Promo", "SchoolHoliday",
                   "DOW_2", "DOW_3", "DOW_4", "DOW_5", "DOW_6"]

HORIZON_DAYS = 14
N_BLOCKS = 26
TRAIN_MONTHS_LIST = list(range(1, 13))


# ============================================================
# UTILITIES
# ============================================================
def enforce_non_negative(predictions, open_status=None, index=None):
    preds = np.array(predictions, dtype=float)
    preds = np.maximum(preds, 0)

    if open_status is not None:
        open_status = np.array(open_status, dtype=int)
        preds[open_status == 0] = 0

    if index is not None:
        return pd.Series(preds, index=index)
    return preds


def prepare_store_data_fixed_exog(df, store_id):
    s = df[df["Store"] == store_id].copy()
    s["Date"] = pd.to_datetime(s["Date"], dayfirst=True, errors="coerce")
    s = s.dropna(subset=["Date"]).sort_values("Date").set_index("Date")

    # y as float
    y = pd.to_numeric(s["Sales"], errors="coerce").astype(float).dropna()

    # base exogs
    X = s.loc[y.index, ["Open", "Promo", "SchoolHoliday", "DayOfWeek"]].copy()
    X["Open"] = pd.to_numeric(X["Open"], errors="coerce").fillna(0).astype(int)
    X["Promo"] = pd.to_numeric(X["Promo"], errors="coerce").fillna(0).astype(int)
    X["SchoolHoliday"] = pd.to_numeric(X["SchoolHoliday"], errors="coerce").fillna(0).astype(int)

    # dummies
    dow = pd.get_dummies(X["DayOfWeek"], prefix="DOW", drop_first=True)
    X_all = pd.concat([X.drop(columns=["DayOfWeek"]), dow], axis=1)

    # Ensure required cols exist
    for c in FIXED_EXOG_COLS:
        if c not in X_all.columns:
            X_all[c] = 0

    # ✅ FORCE NUMERIC FLOAT
    X_fixed = X_all[FIXED_EXOG_COLS].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(float)

    return y, X_fixed


def create_backward_test_blocks(last_date):
    last_date = pd.to_datetime(last_date)
    blocks = []
    for t in range(1, N_BLOCKS + 1):
        test_end = last_date - pd.Timedelta(days=(t - 1) * HORIZON_DAYS)
        test_start = test_end - pd.Timedelta(days=HORIZON_DAYS - 1)
        blocks.append((test_start, test_end))
    return blocks


def fit_predict_rmse(y, X, train_start, train_end, test_start, test_end):
    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_start:test_end]

    if len(y_train) < 30 or len(y_test) == 0:
        return np.nan, None

    X_train = X.loc[y_train.index].astype(float)
    X_test = X.loc[y_test.index].astype(float)

    model = sm.tsa.SARIMAX(
        y_train.astype(float),
        exog=X_train,
        order=FIXED_ORDER,
        seasonal_order=FIXED_SEASONAL_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    res = model.fit(disp=False)
    pred = res.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_test)
    pred = enforce_non_negative(pred, open_status=X_test["Open"], index=pred.index)

    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    return rmse, pred


# ============================================================
# MAIN TASK 2
# ============================================================
def run_task2(df, store_id):

    print("\n" + "="*90)
    print(f"TASK 2 — TRAINING LENGTH SELECTION (Store {store_id})")
    print("="*90)
    print(f"Fixed model: SARIMAX{FIXED_ORDER}{FIXED_SEASONAL_ORDER}")
    print(f"Fixed exogs: {FIXED_EXOG_COLS}")
    print(f"Horizon: {HORIZON_DAYS} days | Blocks: {N_BLOCKS} | Training lengths: 1..12 months")

    y, X = prepare_store_data_fixed_exog(df, store_id)
    last_date = y.index.max()

    print("\nData info:")
    print(f"  Date range: {y.index.min().date()} → {y.index.max().date()}")
    print(f"  Total days: {len(y)}")

    blocks = create_backward_test_blocks(last_date)
    print("\nEvaluation period (last 26 blocks):")
    print(f"  Oldest test: {blocks[-1][0].date()} → {blocks[-1][1].date()}")
    print(f"  Newest test: {blocks[0][0].date()} → {blocks[0][1].date()}")

    rmse_mat = pd.DataFrame(index=TRAIN_MONTHS_LIST, columns=range(1, N_BLOCKS + 1), dtype=float)

    for i, (test_start, test_end) in enumerate(blocks, start=1):
        train_end = test_start - pd.Timedelta(days=1)
        for L in TRAIN_MONTHS_LIST:
            train_start = test_start - pd.DateOffset(months=L)
            rmse, _ = fit_predict_rmse(y, X, train_start, train_end, test_start, test_end)
            rmse_mat.loc[L, i] = rmse

    summary = pd.DataFrame({
        "Mean_RMSE": rmse_mat.mean(axis=1, skipna=True),
        "Std_RMSE": rmse_mat.std(axis=1, skipna=True),
        "Valid_Blocks": rmse_mat.notna().sum(axis=1)
    })

    best_L = int(summary.sort_values(["Mean_RMSE", "Std_RMSE"]).index[0])

    print("\n" + "-"*90)
    print("RMSE SUMMARY BY TRAINING LENGTH (L months)")
    print("-"*90)
    print(summary.round(2))

    print(f"\nSelected L* = {best_L} months "
          f"(Mean RMSE={summary.loc[best_L,'Mean_RMSE']:.2f}, Std={summary.loc[best_L,'Std_RMSE']:.2f})")

    # RMSE BOXPLOT
    plt.figure(figsize=(10, 5))
    plt.boxplot([rmse_mat.loc[L].dropna().values for L in TRAIN_MONTHS_LIST],
                labels=TRAIN_MONTHS_LIST,
                showfliers=False)
    plt.xlabel("Training Length (Months)")
    plt.ylabel("RMSE (14-day blocks)")
    plt.title(f"Store {store_id}: RMSE Distribution Across Training Lengths")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # FINAL LAST 14 DAYS ONLY (Block 1)
    test_start, test_end = blocks[0]
    train_start = test_start - pd.DateOffset(months=best_L)
    train_end = test_start - pd.Timedelta(days=1)

    final_rmse, pred = fit_predict_rmse(y, X, train_start, train_end, test_start, test_end)
    if pred is None or np.isnan(final_rmse):
        raise RuntimeError("Final block prediction failed (pred is None or RMSE is NaN).")

    print(f"\nFinal 14-day RMSE (most recent block): {final_rmse:.2f}")

    y_test = y.loc[test_start:test_end]

    # ✅ Create last14_table (this is what you want to return)
    last14_table = pd.DataFrame({
        "Actual": y_test.values,
        "Predicted": pred.values
    }, index=y_test.index)
    last14_table["Error"] = last14_table["Predicted"] - last14_table["Actual"]

    print("\nLast 14 Days: Actual vs Predicted")
    print(last14_table.reset_index().rename(columns={"index": "Date"}).to_string(index=False))

    # FINAL PLOT — LAST 2 MONTHS + TEST PREDICTION
    history_start = test_start - pd.DateOffset(months=2)
    y_history = y.loc[history_start:test_end]

    plt.figure(figsize=(12, 5))

    # History actual
    plt.plot(y_history.index, y_history.values,
             color="blue", linewidth=1.5, label="Actual (Last 2 Months)")

    # Highlight test actual
    plt.plot(y_test.index, y_test.values,
             'bo', markersize=4, label="Test Actual")

    # Test prediction
    plt.plot(pred.index, pred.values,
             'r--', linewidth=2, label="Test Predicted")

    plt.title(f"Store {store_id}: Last 2 Months + 14-Day Prediction")
    plt.xlabel("Date")
    plt.ylabel("Sales")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # ✅ CRITICAL FIX: return the 3 things you are unpacking
    return rmse_mat, summary, last14_table


# RUN
rmse_mat, summary, last14_table = run_task2(df, store_id=1)

In [ ]:
# ============================================================
# TASK 2 — CLEAN VERSION (NO FUTURE FORECASTING) + DTYPE FIX
# Model:
#   SARIMAX (0,0,1)(2,0,2,7)
# Exog:
#   Open + Promo + SchoolHoliday + DOW_2..DOW_6
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error


# ============================================================
# FIXED SETTINGS
# ============================================================
FIXED_ORDER = (0, 0, 1)
FIXED_SEASONAL_ORDER = (2, 0, 2, 7)

FIXED_EXOG_COLS = ["Open", "Promo", "SchoolHoliday",
                   "DOW_2", "DOW_3", "DOW_4", "DOW_5", "DOW_6"]

HORIZON_DAYS = 14
N_BLOCKS = 26
TRAIN_MONTHS_LIST = list(range(1, 13))


# ============================================================
# UTILITIES
# ============================================================
def enforce_non_negative(predictions, open_status=None, index=None):
    preds = np.array(predictions, dtype=float)
    preds = np.maximum(preds, 0)

    if open_status is not None:
        open_status = np.array(open_status, dtype=int)
        preds[open_status == 0] = 0

    if index is not None:
        return pd.Series(preds, index=index)
    return preds


def prepare_store_data_fixed_exog(df, store_id):
    s = df[df["Store"] == store_id].copy()
    s["Date"] = pd.to_datetime(s["Date"], dayfirst=True, errors="coerce")
    s = s.dropna(subset=["Date"]).sort_values("Date").set_index("Date")

    # y as float
    y = pd.to_numeric(s["Sales"], errors="coerce").astype(float).dropna()

    # base exogs
    X = s.loc[y.index, ["Open", "Promo", "SchoolHoliday", "DayOfWeek"]].copy()
    X["Open"] = pd.to_numeric(X["Open"], errors="coerce").fillna(0).astype(int)
    X["Promo"] = pd.to_numeric(X["Promo"], errors="coerce").fillna(0).astype(int)
    X["SchoolHoliday"] = pd.to_numeric(X["SchoolHoliday"], errors="coerce").fillna(0).astype(int)

    # dummies
    dow = pd.get_dummies(X["DayOfWeek"], prefix="DOW", drop_first=True)
    X_all = pd.concat([X.drop(columns=["DayOfWeek"]), dow], axis=1)

    # Ensure required cols exist
    for c in FIXED_EXOG_COLS:
        if c not in X_all.columns:
            X_all[c] = 0

    # FORCE NUMERIC FLOAT
    X_fixed = X_all[FIXED_EXOG_COLS].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(float)

    return y, X_fixed


def create_backward_test_blocks(last_date):
    last_date = pd.to_datetime(last_date)
    blocks = []
    for t in range(1, N_BLOCKS + 1):
        test_end = last_date - pd.Timedelta(days=(t - 1) * HORIZON_DAYS)
        test_start = test_end - pd.Timedelta(days=HORIZON_DAYS - 1)
        blocks.append((test_start, test_end))
    return blocks


def fit_predict_rmse(y, X, train_start, train_end, test_start, test_end):
    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_start:test_end]

    if len(y_train) < 30 or len(y_test) == 0:
        return np.nan, None

    X_train = X.loc[y_train.index].astype(float)
    X_test = X.loc[y_test.index].astype(float)

    model = sm.tsa.SARIMAX(
        y_train.astype(float),
        exog=X_train,
        order=FIXED_ORDER,
        seasonal_order=FIXED_SEASONAL_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    res = model.fit(disp=False)
    pred = res.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_test)
    pred = enforce_non_negative(pred, open_status=X_test["Open"], index=pred.index)

    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    return rmse, pred


# ============================================================
# MAIN TASK 2
# ============================================================
def run_task2(df, store_id):

    print("\n" + "="*90)
    print(f"TASK 2 — TRAINING LENGTH SELECTION (Store {store_id})")
    print("="*90)
    print(f"Fixed model: SARIMAX{FIXED_ORDER}{FIXED_SEASONAL_ORDER}")
    print(f"Fixed exogs: {FIXED_EXOG_COLS}")
    print(f"Horizon: {HORIZON_DAYS} days | Blocks: {N_BLOCKS} | Training lengths: 1..12 months")

    y, X = prepare_store_data_fixed_exog(df, store_id)
    last_date = y.index.max()

    print("\nData info:")
    print(f"  Date range: {y.index.min().date()} → {y.index.max().date()}")
    print(f"  Total days: {len(y)}")

    blocks = create_backward_test_blocks(last_date)
    print("\nEvaluation period (last 26 blocks):")
    print(f"  Oldest test: {blocks[-1][0].date()} → {blocks[-1][1].date()}")
    print(f"  Newest test: {blocks[0][0].date()} → {blocks[0][1].date()}")

    rmse_mat = pd.DataFrame(index=TRAIN_MONTHS_LIST, columns=range(1, N_BLOCKS + 1), dtype=float)

    for i, (test_start, test_end) in enumerate(blocks, start=1):
        train_end = test_start - pd.Timedelta(days=1)
        for L in TRAIN_MONTHS_LIST:
            train_start = test_start - pd.DateOffset(months=L)
            rmse, _ = fit_predict_rmse(y, X, train_start, train_end, test_start, test_end)
            rmse_mat.loc[L, i] = rmse

    # ✅ summary without Valid_Blocks
    summary = pd.DataFrame({
        "Mean_RMSE": rmse_mat.mean(axis=1, skipna=True),
        "Std_RMSE": rmse_mat.std(axis=1, skipna=True),
    })

    best_L = int(summary.sort_values(["Mean_RMSE", "Std_RMSE"]).index[0])

    print("\n" + "-"*90)
    print("RMSE SUMMARY BY TRAINING LENGTH (L months)")
    print("-"*90)
    print(summary.round(2))

    print(f"\nSelected L* = {best_L} months "
          f"(Mean RMSE={summary.loc[best_L,'Mean_RMSE']:.2f}, Std={summary.loc[best_L,'Std_RMSE']:.2f})")

    # ✅ print all 26 RMSE values for each L (your requested loop)
    for L in TRAIN_MONTHS_LIST:
        print(f"\nTraining Length = {L} month(s)")
        print(rmse_mat.loc[L].round(2).to_string())

    # RMSE BOXPLOT
    plt.figure(figsize=(10, 5))
    plt.boxplot([rmse_mat.loc[L].dropna().values for L in TRAIN_MONTHS_LIST],
                labels=TRAIN_MONTHS_LIST,
                showfliers=False)
    plt.xlabel("Training Length (Months)")
    plt.ylabel("RMSE (14-day blocks)")
    plt.title(f"Store {store_id}: RMSE Distribution Across Training Lengths")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # FINAL LAST 14 DAYS ONLY (Block 1)
    test_start, test_end = blocks[0]
    train_start = test_start - pd.DateOffset(months=best_L)
    train_end = test_start - pd.Timedelta(days=1)

    final_rmse, pred = fit_predict_rmse(y, X, train_start, train_end, test_start, test_end)
    if pred is None or np.isnan(final_rmse):
        raise RuntimeError("Final block prediction failed (pred is None or RMSE is NaN).")

    print(f"\nFinal 14-day RMSE (most recent block): {final_rmse:.2f}")

    y_test = y.loc[test_start:test_end]

    last14_table = pd.DataFrame({
        "Actual": y_test.values,
        "Predicted": pred.values
    }, index=y_test.index)
    last14_table["Error"] = last14_table["Predicted"] - last14_table["Actual"]

    print("\nLast 14 Days: Actual vs Predicted")
    print(last14_table.reset_index().rename(columns={"index": "Date"}).to_string(index=False))

    # FINAL PLOT — LAST 2 MONTHS + TEST PREDICTION
    history_start = test_start - pd.DateOffset(months=2)
    y_history = y.loc[history_start:test_end]

    plt.figure(figsize=(12, 5))

    plt.plot(y_history.index, y_history.values,
             color="blue", linewidth=1.5, label="Actual (Last 2 Months)")

    plt.plot(y_test.index, y_test.values,
             'bo', markersize=4, label="Test Actual")

    plt.plot(pred.index, pred.values,
             'r--', linewidth=2, label="Test Predicted")

    plt.title(f"Store {store_id}: Last 2 Months + 14-Day Prediction")
    plt.xlabel("Date")
    plt.ylabel("Sales")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return rmse_mat, summary, last14_table


# RUN
rmse_mat, summary, last14_table = run_task2(df, store_id=1)

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_test(series, title):
    series = series.dropna()
    result = adfuller(series, autolag="AIC")

    print(f"\nADF Test: {title}")
    print(f"ADF Statistic : {result[0]:.6f}")
    print(f"p-value       : {result[1]:.6f}")

    if result[1] < 0.05:
        print("Decision: Stationary (reject H0)")
    else:
        print("Decision: NOT stationary (fail to reject H0)")

    return result[1]

p0 = adf_test(y, f"Original Sales Series (Store {STORE_ID})")

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(y)
plt.title(f"Original Sales Time Series (d = 0) — Store {STORE_ID}")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.grid(True, alpha=0.3)
plt.show()



In [ ]:
df.head(5)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from collections import Counter
import textwrap

import pmdarima as pm
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =============================================================================
# IMPROVED UTILITY FUNCTIONS
# =============================================================================

def enforce_non_negative(predictions, open_status=None, index=None):
    """
    Ensure predictions are non-negative.
    If store is closed (Open=0), sales should be 0.
    Returns a pandas Series if index is provided, otherwise numpy array.
    """
    predictions = np.array(predictions)
    predictions = np.maximum(predictions, 0)

    if open_status is not None:
        open_status = np.array(open_status)
        predictions[open_status == 0] = 0

    if index is not None:
        return pd.Series(predictions, index=index)
    return predictions


def calculate_mape_safe(actual, predicted):
    """
    Safe MAPE calculation that handles zero values.
    Returns NaN if all actual values are zero.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)

    mask = actual != 0
    if mask.sum() == 0:
        return np.nan

    predicted = enforce_non_negative(predicted)
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


def calculate_smape(actual, predicted):
    """
    Symmetric MAPE - handles zero values better than MAPE.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)

    predicted = enforce_non_negative(predicted)

    denominator = (np.abs(actual) + np.abs(predicted))
    mask = denominator != 0
    if mask.sum() == 0:
        return np.nan

    return 100 / len(actual) * np.sum(2 * np.abs(predicted[mask] - actual[mask]) / denominator[mask])


def calculate_all_metrics(y_true, y_pred, X_test=None):
    """
    Calculate all performance metrics safely.
    """
    metrics = {}

    if X_test is not None and 'Open' in X_test.columns:
        y_pred_clean = enforce_non_negative(
            y_pred, X_test['Open'],
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )
    else:
        y_pred_clean = enforce_non_negative(
            y_pred,
            index=y_pred.index if hasattr(y_pred, 'index') else None
        )

    metrics['rmse'] = float(np.sqrt(mean_squared_error(y_true, y_pred_clean)))
    metrics['mae'] = float(mean_absolute_error(y_true, y_pred_clean))

    metrics['mape'] = calculate_mape_safe(y_true, y_pred_clean)
    metrics['smape'] = calculate_smape(y_true, y_pred_clean)

    metrics['bias'] = float(np.mean(y_pred_clean - y_true))
    metrics['std_error'] = float(np.std(y_pred_clean - y_true))

    ss_res = np.sum((y_true - y_pred_clean) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    metrics['r2_adj'] = float(1 - (ss_res / ss_tot)) if ss_tot > 0 else np.nan

    return metrics, y_pred_clean


# =============================================================================
# DATA PREPARATION
# =============================================================================

def prepare_store_data(
    df: pd.DataFrame,
    store_id: int,
    exogenous_columns=("Open", "Promo", "SchoolHoliday", "StateHoliday", "IsPromoMonth", "DayOfWeek"),
):
    s = df[df["Store"] == store_id].copy()
    s["Date"] = pd.to_datetime(s["Date"], dayfirst=True)
    s = s.sort_values("Date").set_index("Date")

    y = s["Sales"].astype(float).sort_index()
    X = s[list(exogenous_columns)].copy().loc[y.index]

    X["Open"] = X["Open"].astype(int)
    X["Promo"] = X["Promo"].astype(int)
    X["SchoolHoliday"] = X["SchoolHoliday"].astype(int)
    X["IsPromoMonth"] = X["IsPromoMonth"].astype(int)
    X["StateHoliday"] = (X["StateHoliday"].astype(str) != "0").astype(int)

    dow = pd.get_dummies(X["DayOfWeek"], prefix="DOW", drop_first=True).astype(float)

    X_num = X.drop(columns=["DayOfWeek"]).astype(float)
    Xcand = pd.concat([X_num, dow], axis=1).astype(float)

    return y, Xcand


def select_orders_with_auto_arima(y_tr, X_tr, seasonal_on=True, m=7, trace=False):
    if seasonal_on:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=True,
            m=m,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            D=None,
            max_p=3, max_q=3,
            max_P=2, max_Q=2,
            max_d=0,
            max_D=1,
            start_p=1,
            start_q=1,
            start_P=0,
            start_Q=0,
            test='adf',
            seasonal_test='ocsb',
            n_fits=20,
            with_intercept=True
        )
        return sel.order, sel.seasonal_order, float(sel.aic()), float(sel.bic())
    else:
        sel = pm.auto_arima(
            y_tr,
            exogenous=X_tr,
            seasonal=False,
            stepwise=True,
            trace=trace,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic",
            d=0,
            max_p=3, max_q=3,
            max_d=0,
            start_p=1,
            start_q=1,
            test='adf',
            n_fits=15,
            with_intercept=True
        )
        return sel.order, (0, 0, 0, 0), float(sel.aic()), float(sel.bic())


def backward_elimination_exog(y_tr, X_tr, order, seasonal_order, p_thresh=0.05, max_iter=50):
    cols = list(X_tr.columns)

    if len(cols) == 0:
        mod = sm.tsa.SARIMAX(
            y_tr, exog=None, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        return [], res

    for _ in range(max_iter):
        X_use = X_tr[cols] if cols else None

        mod = sm.tsa.SARIMAX(
            y_tr, exog=X_use, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        if not cols:
            return [], res

        pvals = res.pvalues
        p_exog = pvals[[c for c in pvals.index if c in cols]]

        if len(p_exog) == 0:
            return cols, res

        worst_col = p_exog.idxmax()
        worst_p = float(p_exog.max())

        if worst_p <= p_thresh:
            return cols, res

        cols.remove(worst_col)

        if not cols:
            mod2 = sm.tsa.SARIMAX(
                y_tr, exog=None, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False
            )
            res2 = mod2.fit(disp=False)
            return [], res2

    return cols, res


# =============================================================================
# CORRECT SLIDING WINDOW VALIDATION
# =============================================================================

def create_sliding_splits(
    initial_train_start='2013-01-01',
    initial_train_end='2014-07-31',
    horizon_days=14,
    n_splits=26
):
    """
    Create sliding window splits where BOTH training and test windows slide forward.
    """
    initial_train_start = pd.to_datetime(initial_train_start)
    initial_train_end = pd.to_datetime(initial_train_end)

    splits = []

    for i in range(n_splits):
        shift_days = i * horizon_days
        train_start = initial_train_start + pd.Timedelta(days=shift_days)
        train_end = initial_train_end + pd.Timedelta(days=shift_days)
        test_start = train_end + pd.Timedelta(days=1)
        test_end = test_start + pd.Timedelta(days=horizon_days - 1)

        splits.append({
            'split_id': i + 1,
            'train_start': train_start,
            'train_end': train_end,
            'test_start': test_start,
            'test_end': test_end,
            'shift_days': shift_days
        })

    return splits


def print_splits_summary(splits):
    """
    Print a nice summary of all splits.
    """
    print(f"\n{'='*100}")
    print(f"SLIDING WINDOW SPLITS SUMMARY")
    print(f"{'='*100}")
    print(f"{'Block':<6} {'Training Period':<35} {'Test Period':<35} {'Shift'}")
    print(f"{'-'*100}")

    for split in splits[:5]:
        train_period = f"{split['train_start'].date()} to {split['train_end'].date()}"
        test_period = f"{split['test_start'].date()} to {split['test_end'].date()}"
        print(f"{split['split_id']:<6} {train_period:<35} {test_period:<35} {split['shift_days']} days")

    if len(splits) > 10:
        print("...")

    for split in splits[-5:]:
        train_period = f"{split['train_start'].date()} to {split['train_end'].date()}"
        test_period = f"{split['test_start'].date()} to {split['test_end'].date()}"
        print(f"{split['split_id']:<6} {train_period:<35} {test_period:<35} {split['shift_days']} days")

    print(f"{'='*100}")


def evaluate_single_split(y_full, X_full, split, seasonal_on=True, m=7,
                          p_thresh=0.05, auto_arima_trace=False):
    """
    Evaluate a single split.
    """
    train_mask = (y_full.index >= split['train_start']) & (y_full.index <= split['train_end'])
    test_mask = (y_full.index >= split['test_start']) & (y_full.index <= split['test_end'])

    y_train = y_full[train_mask]
    y_test = y_full[test_mask]

    if len(y_test) == 0:
        print(f"    Warning: No test data available for split {split['split_id']}")
        return None

    X_train = X_full.loc[y_train.index]
    X_test = X_full.loc[y_test.index]

    try:
        order, seas, aic, bic = select_orders_with_auto_arima(
            y_train, X_train, seasonal_on=seasonal_on, m=m, trace=auto_arima_trace
        )

        selected_cols, _ = backward_elimination_exog(
            y_train, X_train, order, seas, p_thresh=p_thresh
        )

        # =========================
        # FIX 1: keep original column order (do NOT rely on elimination order)
        # =========================
        selected_cols = [c for c in X_train.columns if c in (selected_cols or [])]

        if selected_cols:
            X_tr_final = X_train[selected_cols]
            X_te_final = X_test[selected_cols]
        else:
            X_tr_final = None
            X_te_final = None
            selected_cols = []

        mod = sm.tsa.SARIMAX(
            y_train, exog=X_tr_final, order=order, seasonal_order=seas,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)

        pred_raw = res.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_te_final)
        pred = enforce_non_negative(
            pred_raw,
            X_test['Open'] if 'Open' in X_test.columns else None,
            index=pred_raw.index
        )

        rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
        mae = float(mean_absolute_error(y_test, pred))

        metrics, _ = calculate_all_metrics(y_test, pred, X_test)

        return {
            'split_id': split['split_id'],
            'train_start': split['train_start'],
            'train_end': split['train_end'],
            'test_start': split['test_start'],
            'test_end': split['test_end'],
            'rmse': rmse,
            'mae': mae,
            'mape': metrics['mape'],
            'smape': metrics['smape'],
            'order': order,
            'seasonal_order': seas,
            'selected_cols': selected_cols,
            'aic': aic,
            'bic': bic,
            'n_train': len(y_train),
            'n_test': len(y_test)
        }

    except Exception as e:
        print(f"    Error in split {split['split_id']}: {str(e)[:100]}...")
        return None


def run_exog_selection_sliding_window(
    df: pd.DataFrame,
    store_id: int,
    initial_train_start='2013-01-01',
    initial_train_end='2014-07-31',
    horizon_days=14,
    n_splits=26,
    seasonal_on=True,
    m=7,
    p_thresh=0.05,
    auto_arima_trace=False
):
    """
    Main function that implements CORRECT sliding window.
    """

    print(f"\n{'='*80}")
    print(f"EXOGENOUS VARIABLE SELECTION - STORE {store_id}")
    print(f"{'='*80}")
    print(f"Initial training window: {initial_train_start} to {initial_train_end} (1 year 7 months)")
    print(f"Forecast horizon: {horizon_days} days")
    print(f"Number of test blocks: {n_splits}")
    print(f"\n{'='*80}")
    print(f"SLIDING WINDOW MECHANISM:")
    print(f"{'='*80}")
    print(f"  * BOTH training AND test windows slide forward by {horizon_days} days each block")
    print(f"  * Training window always 1 year 7 months (577 days)")
    print(f"  * Test window always 14 days")
    print(f"  * 26 blocks covering from 2014-08-01 to 2015-07-31")

    y_full, X_full = prepare_store_data(df, store_id)
    print(f"\nData loaded: {len(y_full)} days from {y_full.index[0].date()} to {y_full.index[-1].date()}")
    print(f"Total exogenous variables available: {len(X_full.columns)}")
    print(f"Variables: {list(X_full.columns)}")

    splits = create_sliding_splits(
        initial_train_start=initial_train_start,
        initial_train_end=initial_train_end,
        horizon_days=horizon_days,
        n_splits=n_splits
    )

    print_splits_summary(splits)

    all_results = []

    print(f"\n{'='*80}")
    print(f"EVALUATING EACH SPLIT")
    print(f"{'='*80}")

    for split in splits:
        print(f"\nSplit {split['split_id']}:")
        print(f"  Train: {split['train_start'].date()} to {split['train_end'].date()}")
        print(f"  Test:  {split['test_start'].date()} to {split['test_end'].date()}")

        result = evaluate_single_split(
            y_full=y_full,
            X_full=X_full,
            split=split,
            seasonal_on=seasonal_on,
            m=m,
            p_thresh=p_thresh,
            auto_arima_trace=auto_arima_trace
        )

        if result:
            all_results.append(result)
            print(f"  * RMSE: {result['rmse']:.2f}")
            print(f"    Selected exog: {result['selected_cols'] if result['selected_cols'] else 'None'}")
            print(f"    Model order: {result['order']}{result['seasonal_order']}")

    if not all_results:
        print("\n*** No successful splits! ***")
        return None

    results_df = pd.DataFrame(all_results)

    return results_df, splits


def analyze_selection_frequency(results_df):
    """
    Analyze how often each exogenous variable is selected across splits.
    """
    var_selection_count = {}
    total_splits = len(results_df)

    all_vars = []
    for selected in results_df['selected_cols']:
        if selected:
            all_vars.extend(selected)
    all_vars = sorted(set(all_vars))

    for var in all_vars:
        count = 0
        for selected in results_df['selected_cols']:
            if selected and var in selected:
                count += 1
        var_selection_count[var] = count

    print(f"\n{'='*80}")
    print(f"VARIABLE SELECTION FREQUENCY ANALYSIS")
    print(f"{'='*80}")
    print(f"Across {total_splits} splits:")
    print()
    print(f"{'Variable':<20} {'Times Selected':<15} {'Frequency':<10}")
    print("-" * 45)

    for var, count in sorted(var_selection_count.items(), key=lambda x: x[1], reverse=True):
        freq = (count / total_splits) * 100
        print(f"  * {var:<18} {count:>5}/{total_splits:<9} {freq:>6.1f}%")

    consistent_vars = [var for var, count in var_selection_count.items()
                      if count / total_splits > 0.5]

    print(f"\nConsistently selected variables (>50% of splits):")
    if consistent_vars:
        for var in consistent_vars:
            freq = (var_selection_count[var] / total_splits) * 100
            print(f"  - {var} ({freq:.1f}%)")
    else:
        print(f"  No variables selected consistently")

    return var_selection_count, consistent_vars


def generate_stability_report(results_df):
    """
    Generate detailed stability report.
    """
    rmse_values = results_df['rmse'].values

    print(f"\n{'='*80}")
    print(f"STABILITY REPORT")
    print(f"{'='*80}")
    print(f"\nPerformance across {len(results_df)} splits:")
    print(f"  * Average RMSE: {np.mean(rmse_values):.2f}")
    print(f"  * Std Deviation: {np.std(rmse_values):.2f}")
    print(f"  * Coefficient of Variation: {(np.std(rmse_values)/np.mean(rmse_values))*100:.1f}%")
    print(f"  * Min RMSE: {np.min(rmse_values):.2f}")
    print(f"  * Max RMSE: {np.max(rmse_values):.2f}")
    print(f"  * Range: {np.max(rmse_values) - np.min(rmse_values):.2f}")

    print(f"\nSplit-by-split performance:")
    print("-" * 100)
    print(f"{'Split':<6} {'Test Period':<25} {'RMSE':<10} {'MAE':<10} {'MAPE':<8} {'Selected Exog'}")
    print("-" * 100)

    for _, row in results_df.sort_values('split_id').iterrows():
        test_period = f"{row['test_start'].date()} to {row['test_end'].date()}"
        selected = str(row['selected_cols'])[:30] if row['selected_cols'] else 'None'
        mape = f"{row['mape']:.1f}%" if not np.isnan(row['mape']) else "N/A"
        print(f"{row['split_id']:<6} {test_period:<25} {row['rmse']:<10.2f} {row['mae']:<10.2f} {mape:<8} {selected}")

    return results_df


# =============================================================================
# NEW FREQUENCY-BASED MODEL SELECTION
# =============================================================================

def select_best_model_based_on_frequency(results_df):
    """
    Select the best model based on:
    1. PRIMARY: Most frequent model (highest appearance count)
    2. SECONDARY: Among models with same frequency, pick lowest average RMSE
    Also shows the lowest RMSE model for comparison
    """

    print(f"\n{'='*80}")
    print(f"MODEL SELECTION BASED ON FREQUENCY")
    print(f"{'='*80}")

    # =========================
    # Create model key preserving exog ORDER
    # =========================
    results_df = results_df.copy()
    results_df['model_key'] = results_df.apply(
        lambda row: (
            tuple(row['selected_cols']) if row['selected_cols'] else tuple(),
            row['order'],
            row['seasonal_order']
        ), axis=1
    )

    model_performance = []

    for model_key in results_df['model_key'].unique():
        subset = results_df[results_df['model_key'] == model_key]

        exog_tuple, order, seasonal_order = model_key
        exog_list = list(exog_tuple) if exog_tuple else []

        avg_rmse = subset['rmse'].mean()
        std_rmse = subset['rmse'].std()
        frequency = len(subset)
        frequency_pct = (frequency / len(results_df)) * 100

        split_ids = sorted(subset['split_id'].tolist())
        rmse_values = subset['rmse'].tolist()

        model_performance.append({
            'exog_vars': exog_list,
            'order': order,
            'seasonal_order': seasonal_order,
            'model_str': f"{order}{seasonal_order}",
            'avg_rmse': avg_rmse,
            'std_rmse': std_rmse,
            'frequency': frequency,
            'frequency_pct': frequency_pct,
            'min_rmse': subset['rmse'].min(),
            'max_rmse': subset['rmse'].max(),
            'split_ids': split_ids,
            'rmse_values': rmse_values
        })

    model_df = pd.DataFrame(model_performance)
    
    # Sort by frequency (descending) first, then by avg_rmse (ascending)
    model_df = model_df.sort_values(['frequency', 'avg_rmse'], ascending=[False, True])

    # Get the most frequent model
    best_model = model_df.iloc[0]
    max_frequency = best_model['frequency']

    # Get the lowest RMSE model (regardless of frequency)
    lowest_rmse_model = model_df.loc[model_df['avg_rmse'].idxmin()]

    print(f"\n{'='*80}")
    print(f"MOST FREQUENT MODEL SELECTED")
    print(f"{'='*80}")
    print(f"\nSelection Criteria:")
    print(f"  1. PRIMARY: Highest frequency (most appearances)")
    print(f"  2. SECONDARY: Lowest average RMSE among models with same frequency")
    print(f"\nMost Frequent Model:")

    order_str = str(best_model['order'])
    seasonal_str = str(best_model['seasonal_order'])

    if best_model['exog_vars']:
        exog_str = '[' + ', '.join([f"'{v}'" for v in best_model['exog_vars']]) + ']'
        wrapped_exog = textwrap.wrap(exog_str, width=70, subsequent_indent=' ' * 19)
        print(f"  ARIMA Order: {order_str}{seasonal_str}")
        print(f"  Exogenous Variables: {wrapped_exog[0]}")
        for line in wrapped_exog[1:]:
            print(f"                     {line}")
    else:
        print(f"  ARIMA Order: {order_str}{seasonal_str}")
        print(f"  Exogenous Variables: None")

    print(f"  Frequency: {best_model['frequency']}/{len(results_df)} splits ({best_model['frequency_pct']:.1f}%) - MOST FREQUENT")
    print(f"  Average RMSE: {best_model['avg_rmse']:.2f}")
    print(f"  Std RMSE (stability): {best_model['std_rmse']:.2f}")
    print(f"  RMSE Range: {best_model['min_rmse']:.2f} - {best_model['max_rmse']:.2f}")
    print(f"  Appears in splits: {best_model['split_ids']}")

    if best_model['frequency'] > 1:
        print(f"  Individual RMSEs: {[round(r, 2) for r in best_model['rmse_values']]}")

    print(f"\n{'='*80}")
    print(f"LOWEST RMSE MODEL (for comparison)")
    print(f"{'='*80}")

    order_str_low = str(lowest_rmse_model['order'])
    seasonal_str_low = str(lowest_rmse_model['seasonal_order'])

    if lowest_rmse_model['exog_vars']:
        exog_str_low = '[' + ', '.join([f"'{v}'" for v in lowest_rmse_model['exog_vars']]) + ']'
        wrapped_exog_low = textwrap.wrap(exog_str_low, width=70, subsequent_indent=' ' * 19)
        print(f"  ARIMA Order: {order_str_low}{seasonal_str_low}")
        print(f"  Exogenous Variables: {wrapped_exog_low[0]}")
        for line in wrapped_exog_low[1:]:
            print(f"                     {line}")
    else:
        print(f"  ARIMA Order: {order_str_low}{seasonal_str_low}")
        print(f"  Exogenous Variables: None")

    print(f"  Frequency: {lowest_rmse_model['frequency']}/{len(results_df)} splits ({lowest_rmse_model['frequency_pct']:.1f}%)")
    print(f"  Average RMSE: {lowest_rmse_model['avg_rmse']:.2f} - LOWEST RMSE")
    print(f"  Std RMSE (stability): {lowest_rmse_model['std_rmse']:.2f}")
    print(f"  RMSE Range: {lowest_rmse_model['min_rmse']:.2f} - {lowest_rmse_model['max_rmse']:.2f}")
    print(f"  Appears in splits: {lowest_rmse_model['split_ids']}")

    if lowest_rmse_model['frequency'] > 1:
        print(f"  Individual RMSEs: {[round(r, 2) for r in lowest_rmse_model['rmse_values']]}")

    print(f"\n{'='*80}")
    print(f"TOP 5 MODELS BY FREQUENCY (then by lowest RMSE):")
    print("-" * 100)
    print(f"{'Rank':<6} {'Frequency':<15} {'Avg RMSE':<10} {'Std':<8} {'ARIMA Order':<20} {'Exogenous Variables'}")
    print("-" * 100)

    for rank in range(min(5, len(model_df))):
        row = model_df.iloc[rank]
        order_display = f"{row['order']}{row['seasonal_order']}"
        exog_display = str(row['exog_vars'])[:35] + "..." if len(str(row['exog_vars'])) > 35 else str(row['exog_vars'])
        print(f"{rank+1:<6} {row['frequency']}/{len(results_df)} ({row['frequency_pct']:3.0f}%)  {row['avg_rmse']:<10.2f} {row['std_rmse']:<8.2f} {order_display:<20} {exog_display}")

    return best_model, lowest_rmse_model, model_df


def print_block_summary(results_df):
    """
    Print a summary of each block showing SARIMA model and FULL exogenous variables.
    """
    print(f"\n{'='*80}")
    print(f"BLOCK-BY-BLOCK MODEL SUMMARY")
    print(f"{'='*80}")

    all_configs = []

    for _, row in results_df.sort_values('split_id').iterrows():
        block_num = row['split_id']
        model_order = f"{row['order']}{row['seasonal_order']}"

        # display None, but store empty list
        exog_vars = row['selected_cols'] if row['selected_cols'] else []
        config = {
            'block': block_num,
            'model': model_order,
            'exog': exog_vars,
            'exog_tuple': tuple(exog_vars)  # keep ORDER
        }
        all_configs.append(config)

    print(f"\n{'='*100}")
    print(f"{'Block':<8} {'SARIMA Model':<30} {'Exogenous Variables'}")
    print(f"{'='*100}")

    for config in all_configs:
        block_num = config['block']
        model_order = config['model']
        exog_vars = config['exog'] if config['exog'] else ['None']
        print(f"Block {block_num:<3}   {model_order:<30} {exog_vars}")

    print(f"{'='*100}")

    print(f"\nCONFIGURATION FREQUENCY:")
    print(f"{'-'*100}")

    config_summary = {}
    for config in all_configs:
        key = (config['model'], config['exog_tuple'])
        if key not in config_summary:
            config_summary[key] = {
                'model': config['model'],
                'exog': config['exog'],
                'blocks': [],
                'count': 0
            }
        config_summary[key]['blocks'].append(config['block'])
        config_summary[key]['count'] += 1

    for idx, (key, info) in enumerate(sorted(config_summary.items(),
                                              key=lambda x: x[1]['blocks'][0])):
        blocks_str = ', '.join([str(b) for b in info['blocks']])
        exog_str = str(info['exog'] if info['exog'] else ['None'])
        print(f"\nConfig {idx+1}: {info['model']}")
        print(f"  Exog: {exog_str}")
        print(f"  Appears in blocks: {blocks_str} ({info['count']}/{len(results_df)} blocks, {info['count']/len(results_df)*100:.1f}%)")

    print(f"{'-'*100}")

    most_common_key = max(config_summary.items(), key=lambda x: x[1]['count'])[0]
    most_common_info = config_summary[most_common_key]

    print(f"\nMOST COMMON MODEL CONFIGURATION:")
    print(f"  Appeared in {most_common_info['count']}/{len(results_df)} blocks ({most_common_info['count']/len(results_df)*100:.1f}%)")
    print(f"  SARIMA Model: {most_common_info['model']}")
    print(f"  Exogenous Variables: {most_common_info['exog'] if most_common_info['exog'] else ['None']}")
    print(f"  Blocks: {most_common_info['blocks']}")

    print(f"\nTOP 3 MOST COMMON CONFIGURATIONS:")
    print(f"{'-'*100}")
    print(f"{'Rank':<6} {'Frequency':<15} {'SARIMA Model':<30} {'Exogenous Variables'}")
    print(f"{'-'*100}")

    sorted_configs = sorted(config_summary.items(), key=lambda x: x[1]['count'], reverse=True)

    for rank, (key, info) in enumerate(sorted_configs[:3], 1):
        pct = (info['count'] / len(results_df)) * 100
        exog_str = str(info['exog'] if info['exog'] else ['None'])
        print(f"{rank:<6} {info['count']}/{len(results_df)} ({pct:5.1f}%)   {info['model']:<30} {exog_str}")

    print(f"{'='*100}")

    return all_configs


def save_exog_selection_results(results_df, store_id, var_selection_count, consistent_vars, best_model, lowest_rmse_model, block_configs):
    """
    Save all results to files.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    results_file = f"exog_selection_results_store_{store_id}_{timestamp}.csv"

    save_data = []
    for _, row in results_df.iterrows():
        save_data.append({
            'split_id': row['split_id'],
            'train_start': row['train_start'].date(),
            'train_end': row['train_end'].date(),
            'test_start': row['test_start'].date(),
            'test_end': row['test_end'].date(),
            'rmse': row['rmse'],
            'mae': row['mae'],
            'mape': row['mape'],
            'smape': row['smape'],
            'order': str(row['order']),
            'seasonal_order': str(row['seasonal_order']),
            'selected_cols': str(row['selected_cols']),
            'aic': row['aic'],
            'bic': row['bic'],
            'n_train': row['n_train'],
            'n_test': row['n_test']
        })

    save_df = pd.DataFrame(save_data)
    save_df.to_csv(results_file, index=False, encoding='utf-8')

    summary_file = f"exog_selection_summary_store_{store_id}_{timestamp}.txt"
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(f"EXOGENOUS VARIABLE SELECTION RESULTS - STORE {store_id}\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Selection method: Backward elimination (p-value threshold = 0.05)\n")
        f.write(f"Number of sliding windows: {len(results_df)}\n\n")

        f.write("SLIDING WINDOW CONFIGURATION:\n")
        f.write(f"  Initial training: 2013-01-01 to 2014-07-31\n")
        f.write(f"  Each window shifts forward by 14 days\n")
        f.write(f"  Training window length: 1 year 7 months (577 days)\n")
        f.write(f"  Test window length: 14 days\n\n")

        f.write("PERFORMANCE METRICS:\n")
        f.write(f"  Average RMSE: {results_df['rmse'].mean():.2f}\n")
        f.write(f"  Std RMSE: {results_df['rmse'].std():.2f}\n")
        f.write(f"  Min RMSE: {results_df['rmse'].min():.2f}\n")
        f.write(f"  Max RMSE: {results_df['rmse'].max():.2f}\n")
        f.write(f"  Average MAE: {results_df['mae'].mean():.2f}\n\n")

        f.write("=" * 60 + "\n")
        f.write("BLOCK-BY-BLOCK MODEL SUMMARY\n")
        f.write("=" * 60 + "\n")
        f.write(f"{'Block':<8} {'SARIMA Model':<30} {'Exogenous Variables'}\n")
        f.write("-" * 90 + "\n")

        for _, row in results_df.sort_values('split_id').iterrows():
            block_num = row['split_id']
            model_order = f"{row['order']}{row['seasonal_order']}"
            exog_vars = row['selected_cols'] if row['selected_cols'] else []
            f.write(f"Block {block_num:<3}   {model_order:<30} {exog_vars if exog_vars else ['None']}\n")

        f.write("-" * 90 + "\n\n")

        f.write("VARIABLE SELECTION FREQUENCY:\n")
        for var, count in sorted(var_selection_count.items(), key=lambda x: x[1], reverse=True):
            freq = (count / len(results_df)) * 100
            f.write(f"  * {var:20s}: {count:2d}/{len(results_df)} ({freq:5.1f}%)\n")

        f.write(f"\nCONSISTENTLY SELECTED VARIABLES (>50% of splits):\n")
        if consistent_vars:
            for var in consistent_vars:
                freq = (var_selection_count[var] / len(results_df)) * 100
                f.write(f"  - {var} ({freq:.1f}%)\n")
        else:
            f.write(f"  No variables selected consistently\n")

        f.write(f"\n{'='*60}\n")
        f.write(f"MOST FREQUENT MODEL (Primary Recommendation)\n")
        f.write(f"{'='*60}\n")
        f.write(f"\nSelection Criteria Applied:\n")
        f.write(f"  1. PRIMARY: Highest frequency (most appearances)\n")
        f.write(f"  2. SECONDARY: Lowest average RMSE among models with same frequency\n\n")

        order_str = str(best_model['order'])
        seasonal_str = str(best_model['seasonal_order'])

        f.write(f"MOST FREQUENT MODEL:\n")
        f.write(f"  ARIMA Order: {order_str}{seasonal_str}\n")
        f.write(f"  Exogenous Variables: {best_model['exog_vars'] if best_model['exog_vars'] else 'None'}\n")
        f.write(f"  Frequency: {best_model['frequency']}/{len(results_df)} splits ({best_model['frequency_pct']:.1f}%) - MOST FREQUENT\n")
        f.write(f"  Average RMSE: {best_model['avg_rmse']:.2f}\n")
        f.write(f"  Std RMSE (stability): {best_model['std_rmse']:.2f}\n")
        f.write(f"  RMSE Range: {best_model['min_rmse']:.2f} - {best_model['max_rmse']:.2f}\n")
        f.write(f"  Appears in splits: {best_model['split_ids']}\n\n")

        if best_model['frequency'] > 1:
            f.write(f"  Individual RMSEs: {[round(r, 2) for r in best_model['rmse_values']]}\n\n")

        f.write(f"\n{'='*60}\n")
        f.write(f"LOWEST RMSE MODEL (for comparison)\n")
        f.write(f"{'='*60}\n")

        order_str_low = str(lowest_rmse_model['order'])
        seasonal_str_low = str(lowest_rmse_model['seasonal_order'])

        f.write(f"LOWEST RMSE MODEL:\n")
        f.write(f"  ARIMA Order: {order_str_low}{seasonal_str_low}\n")
        f.write(f"  Exogenous Variables: {lowest_rmse_model['exog_vars'] if lowest_rmse_model['exog_vars'] else 'None'}\n")
        f.write(f"  Frequency: {lowest_rmse_model['frequency']}/{len(results_df)} splits ({lowest_rmse_model['frequency_pct']:.1f}%)\n")
        f.write(f"  Average RMSE: {lowest_rmse_model['avg_rmse']:.2f} - LOWEST RMSE\n")
        f.write(f"  Std RMSE (stability): {lowest_rmse_model['std_rmse']:.2f}\n")
        f.write(f"  RMSE Range: {lowest_rmse_model['min_rmse']:.2f} - {lowest_rmse_model['max_rmse']:.2f}\n")
        f.write(f"  Appears in splits: {lowest_rmse_model['split_ids']}\n\n")

        f.write(f"\n{'='*60}\n")
        f.write(f"END OF MODEL SELECTION REPORT\n")
        f.write(f"{'='*60}\n")

    print(f"\nResults saved:")
    print(f"  * Detailed results: {results_file}")
    print(f"  * Summary with model specification: {summary_file}")

    return results_file, summary_file


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def run_exog_selection_for_store(
    df: pd.DataFrame,
    store_id: int = 1,
    initial_train_start='2013-01-01',
    initial_train_end='2014-07-31',
    horizon_days=14,
    n_splits=26,
    seasonal_on=True,
    m=7,
    p_thresh=0.05,
    auto_arima_trace=False
):
    """
    Main function with CORRECT sliding window implementation.
    """

    print("\n" + "=" * 80)
    print("MASTER THESIS - TASK 1: EXOGENOUS VARIABLE SELECTION")
    print(f"Store ID: {store_id}")
    print("=" * 80)
    print("\nIMPLEMENTING CORRECT SLIDING WINDOW:")
    print("  * BOTH training AND test windows slide forward by 14 days each block")
    print("  * Training window always 1 year 7 months")
    print("  * 26 blocks from 2014-08-01 to 2015-07-31")
    print("\nYOUR EXACT EXOG LOGIC:")
    print("  * Using ALL available exogenous variables initially")
    print("  * Backward elimination based on p-values (statistical significance)")
    print("  * Selection done PER SPLIT")

    results_df, splits = run_exog_selection_sliding_window(
        df=df,
        store_id=store_id,
        initial_train_start=initial_train_start,
        initial_train_end=initial_train_end,
        horizon_days=horizon_days,
        n_splits=n_splits,
        seasonal_on=seasonal_on,
        m=m,
        p_thresh=p_thresh,
        auto_arima_trace=auto_arima_trace
    )

    if results_df is None or len(results_df) == 0:
        print("\n*** Exogenous selection failed. ***")
        return None

    var_selection_count, consistent_vars = analyze_selection_frequency(results_df)

    generate_stability_report(results_df)

    block_configs = print_block_summary(results_df)

    # Use the new frequency-based selection function
    best_model, lowest_rmse_model, model_rankings = select_best_model_based_on_frequency(results_df)

    save_exog_selection_results(results_df, store_id, var_selection_count, consistent_vars, best_model, lowest_rmse_model, block_configs)

    print("\n" + "=" * 80)
    print("TASK 1 COMPLETED SUCCESSFULLY!")
    print("=" * 80)
    print("\nFINAL OUTPUT:")
    print(f"  * Selection method: Backward elimination (p <= {p_thresh})")
    print(f"  * Number of successful splits: {len(results_df)}/{n_splits}")
    print(f"  * Average RMSE: {results_df['rmse'].mean():.2f}")
    print(f"  * Std RMSE: {results_df['rmse'].std():.2f}")

    print(f"\n  * MOST FREQUENT MODEL (Primary Recommendation):")
    if best_model['exog_vars']:
        exog_str = '[' + ', '.join([f"'{v}'" for v in best_model['exog_vars']]) + ']'
        if len(exog_str) > 60:
            print(f"    - Exogenous variables: {exog_str[:60]}...")
            print(f"      Full list: {exog_str}")
        else:
            print(f"    - Exogenous variables: {exog_str}")
    else:
        print(f"    - Exogenous variables: None")
    print(f"    - Frequency: {best_model['frequency']}/{len(results_df)} splits ({best_model['frequency_pct']:.1f}%) - MOST FREQUENT")
    print(f"    - Average RMSE: {best_model['avg_rmse']:.2f}")
    print(f"    - Std RMSE: {best_model['std_rmse']:.2f}")

    print(f"\n  * LOWEST RMSE MODEL (for comparison):")
    if lowest_rmse_model['exog_vars']:
        exog_str_low = '[' + ', '.join([f"'{v}'" for v in lowest_rmse_model['exog_vars']]) + ']'
        if len(exog_str_low) > 60:
            print(f"    - Exogenous variables: {exog_str_low[:60]}...")
            print(f"      Full list: {exog_str_low}")
        else:
            print(f"    - Exogenous variables: {exog_str_low}")
    else:
        print(f"    - Exogenous variables: None")
    print(f"    - Frequency: {lowest_rmse_model['frequency']}/{len(results_df)} splits ({lowest_rmse_model['frequency_pct']:.1f}%)")
    print(f"    - Average RMSE: {lowest_rmse_model['avg_rmse']:.2f} - LOWEST RMSE")
    print(f"    - Std RMSE: {lowest_rmse_model['std_rmse']:.2f}")

    if consistent_vars:
        print(f"\n  * CONSISTENTLY SELECTED VARIABLES (>50% of splits):")
        for var in consistent_vars:
            count = var_selection_count[var]
            freq = (count / len(results_df)) * 100
            print(f"    - {var}: {count}/{len(results_df)} times ({freq:.1f}%)")
        print(f"\n  -> RECOMMENDED MODEL: SARIMAX with {consistent_vars}")

    return {
        'store_id': store_id,
        'results_df': results_df,
        'splits': splits,
        'var_selection_count': var_selection_count,
        'consistent_vars': consistent_vars,
        'best_model': best_model,
        'lowest_rmse_model': lowest_rmse_model,
        'model_rankings': model_rankings,
        'block_configs': block_configs,
        'avg_rmse': results_df['rmse'].mean(),
        'std_rmse': results_df['rmse'].std()
    }


# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":

    print("\n" + "=" * 80)
    print("VERIFYING REAL DATA BEFORE ANALYSIS")
    print("=" * 80)

    print(f"Data shape: {df.shape}")
    print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"Number of unique stores: {df['Store'].nunique()}")
    print(f"\nColumns in dataframe:")
    print(df.columns.tolist())

    required_cols = ['Store', 'Date', 'Sales', 'Open', 'Promo', 'SchoolHoliday',
                     'StateHoliday', 'DayOfWeek', 'IsPromoMonth']

    print("\nChecking required columns:")
    all_good = True
    for col in required_cols:
        if col in df.columns:
            print(f"  * {col}")
        else:
            print(f"  X {col} - MISSING!")
            all_good = False

    if not all_good:
        print("\n*** Missing required columns. Please check your data loading code. ***")
        exit()

    p_thresh = 0.05
    STORE_ID = 1

    print(f"\n{'-'*50}")
    print(f"Analyzing Store {STORE_ID}")
    print('-'*50)

    if STORE_ID not in df['Store'].unique():
        print(f"*** Store {STORE_ID} not found in data! ***")
        print(f"Available stores: {sorted(df['Store'].unique())[:10]}...")
        exit()

    print("\n" + "=" * 80)
    print(f"STARTING TASK 1 FOR STORE {STORE_ID}")
    print("=" * 80)
    print("\nCORRECT SLIDING WINDOW CONFIGURATION:")
    print("  * Block 1: Train 2013-01-01 to 2014-07-31 | Test 2014-08-01 to 2014-08-14")
    print("  * Block 2: Train 2013-01-15 to 2014-08-14 | Test 2014-08-15 to 2014-08-28")
    print("  * Block 3: Train 2013-01-29 to 2014-08-28 | Test 2014-08-29 to 2014-09-11")
    print("  * ... and so on for 26 blocks")

    results = run_exog_selection_for_store(
        df=df,
        store_id=STORE_ID,
        initial_train_start='2013-01-01',
        initial_train_end='2014-07-31',
        horizon_days=14,
        n_splits=26,
        seasonal_on=True,
        m=7,
        p_thresh=p_thresh,
        auto_arima_trace=False  # Set to False as requested
    )

    if results:
        print(f"\n{'='*80}")
        print(f"TASK 1 COMPLETE - FINAL ANSWER FOR STORE {STORE_ID}")
        print(f"{'='*80}")

        best = results['best_model']
        lowest = results['lowest_rmse_model']
        
        print(f"\nMOST FREQUENT MODEL (Primary Recommendation):")
        order_str = str(best['order'])
        seasonal_str = str(best['seasonal_order'])

        if best['exog_vars']:
            exog_str = '[' + ', '.join([f"'{v}'" for v in best['exog_vars']]) + ']'
            wrapped = textwrap.wrap(exog_str, width=70, subsequent_indent=' ' * 19)
            print(f"  ARIMA Order: {order_str}{seasonal_str}")
            print(f"  Exogenous Variables: {wrapped[0]}")
            for line in wrapped[1:]:
                print(f"                     {line}")
        else:
            print(f"  ARIMA Order: {order_str}{seasonal_str}")
            print(f"  Exogenous Variables: None")

        print(f"  Frequency: {best['frequency']}/{len(results['results_df'])} splits ({best['frequency_pct']:.1f}%) - MOST FREQUENT")
        print(f"  Average RMSE: {best['avg_rmse']:.2f}")
        print(f"  Std RMSE (stability): {best['std_rmse']:.2f}")

        print(f"\nLOWEST RMSE MODEL (for comparison):")
        order_str_low = str(lowest['order'])
        seasonal_str_low = str(lowest['seasonal_order'])

        if lowest['exog_vars']:
            exog_str_low = '[' + ', '.join([f"'{v}'" for v in lowest['exog_vars']]) + ']'
            wrapped_low = textwrap.wrap(exog_str_low, width=70, subsequent_indent=' ' * 19)
            print(f"  ARIMA Order: {order_str_low}{seasonal_str_low}")
            print(f"  Exogenous Variables: {wrapped_low[0]}")
            for line in wrapped_low[1:]:
                print(f"                     {line}")
        else:
            print(f"  ARIMA Order: {order_str_low}{seasonal_str_low}")
            print(f"  Exogenous Variables: None")

        print(f"  Frequency: {lowest['frequency']}/{len(results['results_df'])} splits ({lowest['frequency_pct']:.1f}%)")
        print(f"  Average RMSE: {lowest['avg_rmse']:.2f} - LOWEST RMSE")
        print(f"  Std RMSE (stability): {lowest['std_rmse']:.2f}")

        print(f"\nRECOMMENDED MODEL SPECIFICATION for Store {STORE_ID}:")
        print(f"  SARIMAX({best['order']}{best['seasonal_order']}) with exogenous variables:")
        if best['exog_vars']:
            for var in best['exog_vars']:
                count = results['var_selection_count'].get(var, 0)
                freq = (count / len(results['results_df'])) * 100
                print(f"    * {var} (selected {count}/{len(results['results_df'])} times, {freq:.1f}%)")
        else:
            print(f"    * No exogenous variables")

        print(f"\n  This model was validated across {len(results['results_df'])} sliding windows")
        print(f"  It is the MOST FREQUENT model, appearing in {best['frequency']} out of {len(results['results_df'])} splits")
        print(f"  Average RMSE: {best['avg_rmse']:.2f}")
        print(f"  RMSE stability: +/-{best['std_rmse']:.2f}")

        print(f"\nThis recommendation is based on:")
        print(f"  * PRIMARY CRITERION: Highest frequency (most appearances = {best['frequency']} times)")
        print(f"  * Backward elimination with p-value threshold = {p_thresh}")
        print(f"  * {len(results['results_df'])} successful sliding windows out of 26")
        print(f"  * Among models with same frequency, selected lowest RMSE")
        print(f"\n  * For comparison, the lowest RMSE model has average RMSE = {lowest['avg_rmse']:.2f}")